In [ ]:
import os
import pandas as pd
from pathlib import Path

# ==========================================================
# 1. DEFINE MAIN FOLDERS
# ==========================================================
base_folders = {
    "Sediment": Path(r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison"),
    "WaterDepth": Path(r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison"),
    "EmbedmentDepth": Path(r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison"),
    "AllComparison": Path(r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/All_Comparison"),
}

# ==========================================================
# 2. CREATE ORGANIZED FOLDERS
# ==========================================================
for label, folder in base_folders.items():
    organized_folder = folder / "organized"
    organized_folder.mkdir(exist_ok=True)
    print(f"Created or confirmed: {organized_folder}")

# ==========================================================
# 3. FUNCTION TO SCAN FILES
# ==========================================================
def scan_folder(folder_path, source_label):
    records = []

    for root, dirs, files in os.walk(folder_path):
        # skip the organized folder itself so the inventory does not keep reading its own outputs
        dirs[:] = [d for d in dirs if d.lower() != "organized"]

        for file in files:
            full_path = Path(root) / file
            relative_path = full_path.relative_to(folder_path)

            records.append({
                "Source": source_label,
                "TopFolder": folder_path.name,
                "FileName": file,
                "Stem": full_path.stem,
                "Extension": full_path.suffix.lower(),
                "ParentFolder": full_path.parent.name,
                "RelativePath": str(relative_path),
                "FullPath": str(full_path),
                "FileSize_bytes": full_path.stat().st_size if full_path.exists() else None
            })

    return pd.DataFrame(records)

# ==========================================================
# 4. SCAN ALL FOLDERS
# ==========================================================
all_dfs = []

for label, folder in base_folders.items():
    df = scan_folder(folder, label)
    all_dfs.append(df)

    organized_folder = folder / "organized"

    # save full inventory for this folder
    folder_inventory_path = organized_folder / f"{label}_file_inventory.csv"
    df.to_csv(folder_inventory_path, index=False)

    print(f"\n--- {label} ---")
    print(f"Total files found: {len(df)}")
    if not df.empty:
        print("Extensions found:", sorted(df["Extension"].dropna().unique()))
        print("Example files:")
        print(df[["FileName", "Extension", "ParentFolder"]].head(20).to_string(index=False))
    else:
        print("No files found.")

# ==========================================================
# 5. COMBINE INTO MASTER INVENTORY
# ==========================================================
master_df = pd.concat(all_dfs, ignore_index=True)

master_output = Path(r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_file_inventory.csv")
master_df.to_csv(master_output, index=False)

print("\n==========================================================")
print("MASTER INVENTORY SAVED TO:")
print(master_output)
print("==========================================================")

# ==========================================================
# 6. SAVE EXTENSION SUMMARY
# ==========================================================
summary_rows = []

for label, df in zip(base_folders.keys(), all_dfs):
    if df.empty:
        continue

    ext_counts = df["Extension"].value_counts(dropna=False)
    for ext, count in ext_counts.items():
        summary_rows.append({
            "Source": label,
            "Extension": ext,
            "Count": count
        })

summary_df = pd.DataFrame(summary_rows)
summary_output = Path(r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_extension_summary.csv")
summary_df.to_csv(summary_output, index=False)

print("EXTENSION SUMMARY SAVED TO:")
print(summary_output)

# ==========================================================
# 7. OPTIONAL: SAVE CSV-ONLY INVENTORIES
# ==========================================================
for label, folder in base_folders.items():
    organized_folder = folder / "organized"
    df = master_df[master_df["Source"] == label].copy()

    csv_df = df[df["Extension"] == ".csv"].copy()
    csv_output = organized_folder / f"{label}_csv_only_inventory.csv"
    csv_df.to_csv(csv_output, index=False)

    print(f"CSV-only inventory saved: {csv_output}")

# ==========================================================
# 8. PRINT CLEAN PASTE-FRIENDLY SUMMARY
# ==========================================================
print("\n\n==================== PASTE THIS INTO CHAT ====================")

for label in base_folders.keys():
    sub = master_df[master_df["Source"] == label].copy()

    print(f"\n### {label}")
    print(f"Total files: {len(sub)}")

    if sub.empty:
        print("No files found.")
        continue

    print("File types:")
    print(sub["Extension"].value_counts().to_string())

    print("\nFiles:")
    for fp in sub["RelativePath"].tolist():
        print(fp)

print("\n=============================================================")

In [ ]:
import os
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def rename_duplicate_curve_columns(columns):
    """
    Example:
    10.0000.1 -> 10.0000_dup1
    """
    renamed = {}
    for c in columns:
        parts = str(c).split(".")
        if len(parts) > 2 and parts[-1].isdigit():
            base = ".".join(parts[:-1])
            suffix = parts[-1]
            renamed[c] = f"{base}_dup{suffix}"
        else:
            renamed[c] = c
    return renamed

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))
    s = str(val).strip()
    num = extract_numeric_value(s)
    if pd.notna(num):
        if float(num).is_integer():
            return str(int(num))
        return str(float(num))
    return s

def compute_tz_curve_stats(group, z_col="z (m)", t_col="t"):
    g = group.dropna(subset=[z_col, t_col]).sort_values(z_col).copy()

    if len(g) < 5:
        return pd.Series({
            "t_max": np.nan,
            "z_at_50_percent": np.nan,
            "initial_stiffness": np.nan,
            "area_under_curve": np.nan,
            "z_max": np.nan
        })

    z = g[z_col].to_numpy()
    t = g[t_col].to_numpy()

    t_max = np.max(t)

    t_50 = 0.5 * t_max
    idx_50 = np.argmin(np.abs(t - t_50))
    z_50 = z[idx_50]

    n_init = max(3, int(0.1 * len(g)))
    z_init = z[:n_init]
    t_init = t[:n_init]

    if len(np.unique(z_init)) > 1:
        initial_stiffness = np.polyfit(z_init, t_init, 1)[0]
    else:
        initial_stiffness = np.nan

    area = np.trapz(t, z)
    z_max = np.max(z)

    return pd.Series({
        "t_max": t_max,
        "z_at_50_percent": z_50,
        "initial_stiffness": initial_stiffness,
        "area_under_curve": area,
        "z_max": z_max
    })

def build_group_summary(stats_df, group_col):
    summary_df = (
        stats_df.groupby(group_col, observed=False)
        .agg(
            t_max_mean=("t_max", "mean"),
            t_max_std=("t_max", "std"),
            stiff_mean=("initial_stiffness", "mean"),
            stiff_std=("initial_stiffness", "std"),
            z50_mean=("z_at_50_percent", "mean"),
            area_mean=("area_under_curve", "mean"),
            count=("t_max", "count")
        )
        .reset_index()
    )

    summary_df["t_max_std"] = summary_df["t_max_std"].fillna(0)
    summary_df["stiff_std"] = summary_df["stiff_std"].fillna(0)
    summary_df["t_max_se"] = summary_df["t_max_std"] / np.sqrt(summary_df["count"].replace(0, np.nan))
    summary_df["stiff_se"] = summary_df["stiff_std"] / np.sqrt(summary_df["count"].replace(0, np.nan))

    return summary_df

def save_bar_plot(summary_df, group_col, mean_col, se_col, title, ylabel, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[mean_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(12, 6))
    plt.bar(
        plot_df[group_col],
        plot_df[mean_col],
        yerr=plot_df[se_col],
        capsize=5,
        color=colors
    )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def make_shared_legend_handles(color_map, ordered_labels):
    handles = []
    for label in ordered_labels:
        label_str = str(label)
        if label_str in color_map:
            handles.append(
                plt.Line2D([0], [0], color=color_map[label_str], lw=2, marker="o", markersize=4, label=label_str)
            )
    return handles

# ==========================================================
# 2. MASTER FUNCTION
# ==========================================================
def run_tz_analysis(config):
    print(f"\nRunning t-z analysis for: {config['name']}")

    input_csv = config["input_csv"]
    output_dir = make_output_dir(config["output_dir"])
    group_col = config["group_col_name"]
    z_col = config["z_col_name"]
    raw_layout_type = config["raw_layout_type"]

    df = pd.read_csv(input_csv)
    df = clean_columns(df)

    # ------------------------------------------------------
    # A. CLEAN RAW FILE
    # ------------------------------------------------------
    rename_dict = {}
    if len(df.columns) >= 1:
        rename_dict[df.columns[0]] = group_col
    if len(df.columns) >= 2:
        rename_dict[df.columns[1]] = z_col
    df = df.rename(columns=rename_dict)

    if raw_layout_type == "group_rows_curve_columns_with_spacer_cleanup":
        df = df.dropna(axis=1, how="all")

        cols_to_drop = []
        for col in df.columns:
            if col not in [group_col, z_col]:
                col_as_str = df[col].astype(str).str.strip()
                if ((df[col].isna()) | (col_as_str == "") | (col_as_str.str.lower() == "nan")).all():
                    cols_to_drop.append(col)
        if cols_to_drop:
            df = df.drop(columns=cols_to_drop)

    df[z_col] = pd.to_numeric(df[z_col], errors="coerce")
    df[f"{group_col}_num"] = df[group_col].apply(extract_numeric_value)

    # filter to requested groups when numeric
    selected_groups = config["selected_groups"]
    selected_numeric = []
    for x in selected_groups:
        try:
            selected_numeric.append(float(x))
        except:
            pass

    if len(selected_numeric) > 0 and df[f"{group_col}_num"].notna().any():
        df = df[df[f"{group_col}_num"].isin(selected_numeric)].copy()

    curve_cols = [c for c in df.columns if c not in [group_col, z_col, f"{group_col}_num"]]
    df = df.rename(columns=rename_duplicate_curve_columns(curve_cols))
    curve_cols = [c for c in df.columns if c not in [group_col, z_col, f"{group_col}_num"]]

    for c in curve_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # ------------------------------------------------------
    # B. RESHAPE TO LONG
    # ------------------------------------------------------
    long_df = df.melt(
        id_vars=[group_col, z_col, f"{group_col}_num"],
        value_vars=curve_cols,
        var_name="Curve",
        value_name="t"
    )

    long_df["Curve_num"] = long_df["Curve"].apply(
        lambda x: extract_numeric_value(str(x).replace("_dup1", "").replace("_dup2", "").replace("_dup3", ""))
    )
    long_df["t"] = pd.to_numeric(long_df["t"], errors="coerce")
    long_df = long_df.dropna(subset=[group_col, z_col, "t"]).copy()

    order_str = [normalize_group_label(x) for x in config["group_order"]]
    selected_str = [normalize_group_label(x) for x in config["selected_groups"]]

    long_df[group_col] = long_df[group_col].apply(normalize_group_label)
    long_df = long_df[long_df[group_col].isin(order_str)].copy()
    long_df[group_col] = pd.Categorical(long_df[group_col], categories=order_str, ordered=True)

    long_df = long_df.sort_values([group_col, z_col, "Curve"]).reset_index(drop=True)
    long_df.to_csv(os.path.join(output_dir, "t_z_long_format.csv"), index=False)

    # ------------------------------------------------------
    # C. STATS
    # ------------------------------------------------------
    stats_df = (
        long_df.groupby([group_col, "Curve", "Curve_num"], observed=False)
        .apply(lambda g: compute_tz_curve_stats(g, z_col=z_col, t_col="t"))
        .reset_index()
    )

    stats_df.to_csv(os.path.join(output_dir, "t_z_stats_by_group_and_curve.csv"), index=False)

    summary_df = build_group_summary(stats_df, group_col)
    summary_df.to_csv(os.path.join(output_dir, "t_z_summary_by_group.csv"), index=False)

    # ------------------------------------------------------
    # D. BAR PLOTS
    # ------------------------------------------------------
    save_bar_plot(
        summary_df=summary_df,
        group_col=group_col,
        mean_col="t_max_mean",
        se_col="t_max_se",
        title=f"Maximum Shaft Resistance by {config['name']}",
        ylabel="Mean t_max",
        colors_map=config["group_colors"],
        order=order_str,
        out_png=os.path.join(output_dir, "t_z_tmax.png"),
        out_pdf=os.path.join(output_dir, "t_z_tmax.pdf")
    )

    save_bar_plot(
        summary_df=summary_df,
        group_col=group_col,
        mean_col="stiff_mean",
        se_col="stiff_se",
        title=f"Initial t-z Stiffness by {config['name']}",
        ylabel="Mean Initial Stiffness",
        colors_map=config["group_colors"],
        order=order_str,
        out_png=os.path.join(output_dir, "t_z_stiffness.png"),
        out_pdf=os.path.join(output_dir, "t_z_stiffness.pdf")
    )

    # ------------------------------------------------------
    # E. FACET PLOTS: ONE PANEL PER CURVE
    # ------------------------------------------------------
    unique_curves = (
        long_df[["Curve", "Curve_num"]]
        .drop_duplicates()
        .sort_values("Curve_num")["Curve"]
        .tolist()
    )

    handles = make_shared_legend_handles(config["group_colors"], order_str)

    n_panels = len(unique_curves)
    ncols = 4
    nrows = math.ceil(n_panels / ncols)

    # full
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)
    axes = axes.flatten()

    for ax, curve in zip(axes, unique_curves):
        sub = long_df[long_df["Curve"] == curve].copy()

        for group in order_str:
            g = sub[sub[group_col].astype(str) == str(group)].sort_values(z_col)
            if g.empty:
                continue

            ax.plot(
                g[z_col],
                g["t"],
                marker="o",
                markersize=2.5,
                linewidth=1.2,
                color=config["group_colors"].get(str(group), "gray")
            )

        ax.set_title(f"Curve = {curve}")
        ax.set_xlabel(z_col)
        ax.set_ylabel("t")
        ax.grid(True)

    for ax in axes[len(unique_curves):]:
        ax.axis("off")

    fig.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", title=config["name"])
    plt.suptitle(f"t-z Curves by Curve and {config['name']}", y=0.995)
    plt.tight_layout(rect=[0, 0, 0.88, 0.97])
    plt.savefig(os.path.join(output_dir, "t_z_curves_all_full.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "t_z_curves_all_full.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # zoomed
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)
    axes = axes.flatten()

    for ax, curve in zip(axes, unique_curves):
        sub = long_df[long_df["Curve"] == curve].copy()

        for group in order_str:
            g = sub[sub[group_col].astype(str) == str(group)].sort_values(z_col)
            g = g[g[z_col] <= config["zoom_limit"]]
            if g.empty:
                continue

            ax.plot(
                g[z_col],
                g["t"],
                marker="o",
                markersize=2.5,
                linewidth=1.2,
                color=config["group_colors"].get(str(group), "gray")
            )

        ax.set_title(f"Curve = {curve}")
        ax.set_xlabel(z_col)
        ax.set_ylabel("t")
        ax.set_xlim(0, config["zoom_limit"])
        ax.grid(True)

    for ax in axes[len(unique_curves):]:
        ax.axis("off")

    fig.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", title=config["name"])
    plt.suptitle(f"t-z Curves by Curve and {config['name']} (z ≤ {config['zoom_limit']} m)", y=0.995)
    plt.tight_layout(rect=[0, 0, 0.88, 0.97])
    plt.savefig(os.path.join(output_dir, "t_z_curves_all_zoomed.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "t_z_curves_all_zoomed.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # F. SELECTED GROUPS ONLY
    # ------------------------------------------------------
    selected_present = [g for g in selected_str if g in long_df[group_col].astype(str).unique()]

    if len(selected_present) > 0:
        fig, axes = plt.subplots(1, len(selected_present), figsize=(4 * len(selected_present), 4), squeeze=False)
        axes = axes.flatten()

        for ax, group in zip(axes, selected_present):
            sub = long_df[long_df[group_col].astype(str) == str(group)].copy()

            for curve in unique_curves:
                g = sub[sub["Curve"] == curve].sort_values(z_col)
                if g.empty:
                    continue

                ax.plot(
                    g[z_col],
                    g["t"],
                    linewidth=1.5,
                    marker="o",
                    markersize=3,
                    label=str(curve)
                )

            ax.set_title(f"{config['name']} = {group}")
            ax.set_xlabel(z_col)
            ax.set_ylabel("t")
            ax.grid(True)

        plt.suptitle(f"Selected t-z Curves by {config['name']}", y=1.02)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(os.path.join(output_dir, "t_z_selected_groups_full.png"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(output_dir, "t_z_selected_groups_full.pdf"), bbox_inches="tight")
        plt.show()
        plt.close()

    if len(selected_present) > 0:
        fig, axes = plt.subplots(1, len(selected_present), figsize=(4 * len(selected_present), 4), squeeze=False)
        axes = axes.flatten()
    
        for ax, group in zip(axes, selected_present):
            sub = long_df[long_df[group_col].astype(str) == str(group)].copy()
    
            for curve in unique_curves:
                g = sub[sub["Curve"] == curve].sort_values(z_col)
    
                if g.empty:
                    continue
    
                # full curve in light gray dashed line
                ax.plot(
                    g[z_col],
                    g["t"],
                    color="lightgray",
                    linestyle="--",
                    linewidth=1.0,
                    alpha=0.9
                )
    
                # zoomed portion highlighted
                g_zoom = g[g[z_col] <= config["zoom_limit"]]
    
                if g_zoom.empty:
                    continue
    
                ax.plot(
                    g_zoom[z_col],
                    g_zoom["t"],
                    linewidth=1.5,
                    marker="o",
                    markersize=3,
                    label=str(curve)
                )
    
            ax.set_title(f"{config['name']} = {group}")
            ax.set_xlabel(z_col)
            ax.set_ylabel("t")
            ax.set_xlim(0, config["zoom_limit"])
            ax.grid(True)
    
        plt.suptitle(
            f"Selected t-z Curves by {config['name']} (z ≤ {config['zoom_limit']} m; Full Curve Shown in Gray)",
            y=1.02
        )
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(os.path.join(output_dir, "t_z_selected_groups_zoomed.png"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(output_dir, "t_z_selected_groups_zoomed.pdf"), bbox_inches="tight")
        plt.show()
        plt.close()

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 3. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/t_z.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/t_z",
    "group_col_name": "Sediment Type",
    "z_col_name": "z (m)",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    },
    "selected_groups": ["f_sand", "vc_sand", "sand_dgs_grvl", "sandy_silt"],
    "zoom_limit": 0.2,
    "raw_layout_type": "group_rows_curve_columns"
}

water_config = {
    "name": "Water Depth (m)",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/t-z.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/t_z",
    "group_col_name": "Water Depth",
    "z_col_name": "z (m)",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    },
    "selected_groups": [10, 30, 50, 70, 90],
    "zoom_limit": 0.2,
    "raw_layout_type": "group_rows_curve_columns_with_spacer_cleanup"
}

embedment_config = {
    "name": "Embedment Depth (m)",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/t-z.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/t_z",
    "group_col_name": "Embedment Depth",
    "z_col_name": "z (m)",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    },
    "selected_groups": [15, 25, 35, 45, 55],
    "zoom_limit": 0.2,
    "raw_layout_type": "group_rows_curve_columns_with_spacer_cleanup"
}

# ==========================================================
# 4. RUN ALL THREE
# ==========================================================
run_tz_analysis(sediment_config)
run_tz_analysis(water_config)
run_tz_analysis(embedment_config)

In [ ]:
import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan

    if isinstance(val, str):
        return val.strip()

    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))

    return str(val).strip()

def compute_qz_curve_stats(group, z_col="z (m)", q_col="Q (kN)"):
    g = group.dropna(subset=[z_col, q_col]).sort_values(z_col).copy()

    if len(g) < 4:
        return pd.Series({
            "Q_max": np.nan,
            "z_at_Qmax": np.nan,
            "z_at_50_percent": np.nan,
            "z_at_90_percent": np.nan,
            "initial_stiffness": np.nan,
            "area_under_curve": np.nan,
            "brittleness_index": np.nan,
            "mobilization_ratio": np.nan,
            "Q_end": np.nan,
            "z_max": np.nan
        })

    z = g[z_col].to_numpy()
    Q = g[q_col].to_numpy()

    idx_max = np.argmax(Q)
    Q_max = Q[idx_max]
    z_at_Qmax = z[idx_max]

    Q_50 = 0.5 * Q_max
    idx_50 = np.argmin(np.abs(Q - Q_50))
    z_at_50 = z[idx_50]

    Q_90 = 0.9 * Q_max
    idx_90 = np.argmin(np.abs(Q - Q_90))
    z_at_90 = z[idx_90]

    q10 = 0.10 * Q_max
    early_mask = Q <= q10

    z_init = z[early_mask]
    Q_init = Q[early_mask]

    if len(z_init) < 3:
        n_init = min(5, len(z))
        z_init = z[:n_init]
        Q_init = Q[:n_init]

    if len(z_init) >= 2 and np.ptp(z_init) > 0:
        initial_stiffness = np.polyfit(z_init, Q_init, 1)[0]
    else:
        initial_stiffness = np.nan

    area_under_curve = np.trapz(Q, z)

    Q_end = Q[-1]
    if Q_max > 0:
        brittleness_index = (Q_max - Q_end) / Q_max
    else:
        brittleness_index = np.nan

    if z_at_50 != 0:
        mobilization_ratio = z_at_90 / z_at_50
    else:
        mobilization_ratio = np.nan

    return pd.Series({
        "Q_max": Q_max,
        "z_at_Qmax": z_at_Qmax,
        "z_at_50_percent": z_at_50,
        "z_at_90_percent": z_at_90,
        "initial_stiffness": initial_stiffness,
        "area_under_curve": area_under_curve,
        "brittleness_index": brittleness_index,
        "mobilization_ratio": mobilization_ratio,
        "Q_end": Q_end,
        "z_max": np.max(z)
    })

def build_group_summary(stats_df, group_col):
    summary_df = (
        stats_df.groupby(group_col, observed=False)
        .agg(
            Q_max_mean=("Q_max", "mean"),
            Q_max_std=("Q_max", "std"),
            zQmax_mean=("z_at_Qmax", "mean"),
            z90_mean=("z_at_90_percent", "mean"),
            stiff_mean=("initial_stiffness", "mean"),
            stiff_std=("initial_stiffness", "std"),
            brittleness_mean=("brittleness_index", "mean"),
            area_mean=("area_under_curve", "mean"),
            count=("Q_max", "count")
        )
        .reset_index()
    )

    summary_df["Q_max_std"] = summary_df["Q_max_std"].fillna(0)
    summary_df["stiff_std"] = summary_df["stiff_std"].fillna(0)
    summary_df["Q_max_se"] = summary_df["Q_max_std"] / np.sqrt(summary_df["count"].replace(0, np.nan))
    summary_df["stiff_se"] = summary_df["stiff_std"] / np.sqrt(summary_df["count"].replace(0, np.nan))

    return summary_df

def save_bar_plot(summary_df, group_col, mean_col, se_col, title, ylabel, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[mean_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(12, 6))
    plt.bar(
        plot_df[group_col],
        plot_df[mean_col],
        yerr=plot_df[se_col] if se_col in plot_df.columns else None,
        capsize=5 if se_col in plot_df.columns else 0,
        color=colors
    )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def save_simple_bar_plot(summary_df, group_col, value_col, title, ylabel, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[value_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(12, 6))
    plt.bar(plot_df[group_col], plot_df[value_col], color=colors)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def make_shared_legend_handles(color_map, ordered_labels):
    handles = []
    for label in ordered_labels:
        label_str = str(label)
        if label_str in color_map:
            handles.append(
                plt.Line2D([0], [0], color=color_map[label_str], lw=2, marker="o", markersize=4, label=label_str)
            )
    return handles

# ==========================================================
# 2. MASTER FUNCTION
# ==========================================================
def run_qz_analysis(config):
    print(f"\nRunning q-z analysis for: {config['name']}")

    input_csv = config["input_csv"]
    output_dir = make_output_dir(config["output_dir"])
    group_col = config["group_col_name"]
    point_col = config["point_col_name"]
    q_col = config["q_col_name"]
    z_col = config["z_col_name"]

    df = pd.read_csv(input_csv)
    df = clean_columns(df)

    rename_dict = {}
    if len(df.columns) >= 1:
        rename_dict[df.columns[0]] = group_col
    if len(df.columns) >= 2:
        rename_dict[df.columns[1]] = point_col
    if len(df.columns) >= 3:
        rename_dict[df.columns[2]] = q_col
    if len(df.columns) >= 4:
        rename_dict[df.columns[3]] = z_col

    df = df.rename(columns=rename_dict)

    keep_cols = [group_col, point_col, q_col, z_col]
    df = df[keep_cols].copy()

    df[f"{group_col}_num"] = df[group_col].apply(extract_numeric_value)
    df[point_col] = pd.to_numeric(df[point_col], errors="coerce")
    df[q_col] = pd.to_numeric(df[q_col], errors="coerce")
    df[z_col] = pd.to_numeric(df[z_col], errors="coerce")
    
    # For text groups like sediment, do NOT require the numeric helper column.
    df = df.dropna(subset=[group_col, q_col, z_col]).copy()

    
# ------------------------------------------------------
# GROUP FILTERING
# ------------------------------------------------------
    order_str = [normalize_group_label(x) for x in config["group_order"]]
    selected_str = [normalize_group_label(x) for x in config["selected_groups"]]
    
    # detect whether this config is numeric or text-based
    all_selected_numeric = True
    selected_numeric = []
    
    for x in config["selected_groups"]:
        try:
            selected_numeric.append(float(x))
        except:
            all_selected_numeric = False
            break
    
    # For numeric groupings like water depth and embedment depth,
    # overwrite the group label using the cleaned numeric helper column.
    if all_selected_numeric:
        df[group_col] = df[f"{group_col}_num"].apply(normalize_group_label)
    else:
        df[group_col] = df[group_col].apply(normalize_group_label)
    
    all_selected_numeric = True
    selected_numeric = []
    
    for x in config["selected_groups"]:
        try:
            selected_numeric.append(float(x))
        except:
            all_selected_numeric = False
            break
    
    if all_selected_numeric:
        df = df[df[f"{group_col}_num"].isin(selected_numeric)].copy()
    else:
        df = df[df[group_col].isin(selected_str)].copy()
    
    print(f"\n{config['name']} groups found after filtering:")
    print(sorted(df[group_col].dropna().astype(str).unique()))
    print(f"Rows remaining: {len(df)}")
    

    
    df[group_col] = df[group_col].apply(normalize_group_label)
    
    # If selected groups are numeric-like, filter using numeric values.
    # Otherwise filter using normalized string labels.
    selected_numeric = []
    for x in config["selected_groups"]:
        try:
            selected_numeric.append(float(x))
        except:
            pass
    
    if len(selected_numeric) > 0 and df[f"{group_col}_num"].notna().any():
        df = df[df[f"{group_col}_num"].isin(selected_numeric)].copy()
    else:
        df = df[df[group_col].isin(selected_str)].copy()
    
        df = df.sort_values([group_col, z_col, point_col]).reset_index(drop=True)
        df.to_csv(os.path.join(output_dir, "q_z_cleaned_long_format.csv"), index=False)


    # ------------------------------------------------------
    # CURVE STATS
    # ------------------------------------------------------
    stats_list = []

    for group_value, g in df.groupby(group_col, observed=False):
        stats = compute_qz_curve_stats(g, z_col=z_col, q_col=q_col)
    
        if stats is None or stats.isna().all():
            continue
    
        stats_dict = stats.to_dict()
        stats_dict[group_col] = group_value
        stats_list.append(stats_dict)
    
    stats_df = pd.DataFrame(stats_list)
    
    if stats_df.empty:
        raise ValueError(f"No q-z statistics were computed for {config['name']}. Check group filtering and input columns.")
    
    cols = [group_col] + [c for c in stats_df.columns if c != group_col]
    stats_df = stats_df[cols]
    
    stats_df.to_csv(os.path.join(output_dir, "q_z_stats_by_group.csv"), index=False)
    
    print("\nstats_df preview:")
    print(stats_df.head())
    print("stats_df columns:", stats_df.columns.tolist())
    
    summary_df = build_group_summary(stats_df, group_col)
    
    print("\nsummary_df preview:")
    print(summary_df.head())
    print("summary_df columns:", summary_df.columns.tolist())

    
    print("\nsummary_df preview:")
    print(summary_df.head())
    print("summary_df columns:", summary_df.columns.tolist())
    # ------------------------------------------------------
    # CURVE PLOTS
    # ------------------------------------------------------
    handles = make_shared_legend_handles(config["group_colors"], order_str)

    plt.figure(figsize=(8, 6))
    for group in order_str:
        sub = df[df[group_col].astype(str) == str(group)].sort_values(z_col)
        if sub.empty:
            continue

        plt.plot(
            sub[z_col],
            sub[q_col],
            color=config["group_colors"].get(str(group), "gray"),
            linewidth=2,
            marker="o",
            label=str(group)
        )

    plt.xlabel(z_col)
    plt.ylabel(q_col)
    plt.title(f"q-z Curves by {config['name']}")
    plt.legend(title=config["name"])
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "q_z_curves_full.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "q_z_curves_full.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 6))

    for group in order_str:
        sub = df[df[group_col].astype(str) == str(group)].sort_values(z_col)
        if sub.empty:
            continue
    
        # full curve in faint gray to show continuation beyond zoom window
        plt.plot(
            sub[z_col],
            sub[q_col],
            color="lightgray",
            linewidth=1.2,
            linestyle="--",
            alpha=0.9
        )
    
        # highlighted portion inside zoom window
        sub_zoom = sub[sub[z_col] <= config["zoom_limit"]]
        if sub_zoom.empty:
            continue
    
        plt.plot(
            sub_zoom[z_col],
            sub_zoom[q_col],
            color=config["group_colors"].get(str(group), "gray"),
            linewidth=2.2,
            marker="o",
            label=str(group)
        )
    
    plt.xlim(0, config["zoom_limit"])
    plt.xlabel(z_col)
    plt.ylabel(q_col)
    plt.title(f"q-z Curves by {config['name']} (Zoomed, Full Curve Shown in Gray)")
    plt.legend(title=config["name"])
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "q_z_curves_zoomed.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "q_z_curves_zoomed.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # BAR PLOTS
    # ------------------------------------------------------
    save_bar_plot(
        summary_df, group_col, "Q_max_mean", "Q_max_se",
        f"Maximum Tip Resistance by {config['name']}",
        "Mean Q_max (kN)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "q_z_Qmax.png"),
        os.path.join(output_dir, "q_z_Qmax.pdf")
    )

    save_simple_bar_plot(
        summary_df, group_col, "zQmax_mean",
        f"Displacement at Peak Tip Resistance by {config['name']}",
        "Mean z at Q_max (m)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "q_z_z_at_Qmax.png"),
        os.path.join(output_dir, "q_z_z_at_Qmax.pdf")
    )

    save_simple_bar_plot(
        summary_df, group_col, "z90_mean",
        f"Displacement Required to Mobilize 90% of Tip Resistance by {config['name']}",
        "Mean z at 90% of Q_max (m)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "q_z_z90.png"),
        os.path.join(output_dir, "q_z_z90.pdf")
    )

    save_bar_plot(
        summary_df, group_col, "stiff_mean", "stiff_se",
        f"Initial q-z Stiffness by {config['name']}",
        "Mean Initial Stiffness",
        config["group_colors"], order_str,
        os.path.join(output_dir, "q_z_stiffness.png"),
        os.path.join(output_dir, "q_z_stiffness.pdf")
    )

    save_simple_bar_plot(
        summary_df, group_col, "z90_mean",
        f"Displacement Required to Mobilize 90% of Tip Resistance by {config['name']}",
        "Mean z at 90% of Q_max (m)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "q_z_z90.png"),
        os.path.join(output_dir, "q_z_z90.pdf")
    )

    save_simple_bar_plot(
        summary_df, group_col, "area_mean",
        f"Area Under q-z Curve by {config['name']}",
        "Mean Area Under Curve",
        config["group_colors"], order_str,
        os.path.join(output_dir, "q_z_area.png"),
        os.path.join(output_dir, "q_z_area.pdf")
    )

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 3. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/q_z.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/q_z",
    "group_col_name": "Sediment Type",
    "point_col_name": "No.",
    "q_col_name": "Q (kN)",
    "z_col_name": "z (m)",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    },
    "selected_groups": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "zoom_limit": 2.0
}

water_config = {
    "name": "Water Depth (m)",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/q-z.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/q_z",
    "group_col_name": "Water Depth",
    "point_col_name": "No.",
    "q_col_name": "Q (kN)",
    "z_col_name": "z (m)",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    },
    "selected_groups": [10, 30, 50, 70, 90],
    "zoom_limit": 2.0
}

embedment_config = {
    "name": "Embedment Depth (m)",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/q-z.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/q_z",
    "group_col_name": "Embedment Depth",
    "point_col_name": "No.",
    "q_col_name": "Q (kN)",
    "z_col_name": "z (m)",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    },
    "selected_groups": [15, 25, 30, 35, 40, 45, 55],
    "zoom_limit": 2.0
}

# ==========================================================
# 4. RUN ALL THREE
# ==========================================================
run_qz_analysis(sediment_config)
run_qz_analysis(water_config)
run_qz_analysis(embedment_config)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        return val.strip()
    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))
    return str(val).strip()

def safe_curve_value(col_name):
    s = str(col_name).strip()
    s = s.replace(".1", "")
    m = re.match(r"^(\d+\.\d+)", s)
    if m:
        return float(m.group(1))
    return np.nan

def compute_py_stats(group):
    group = group.sort_values("y (m)").copy()

    # collapse duplicate y by max p
    group = group.groupby("y (m)", as_index=False)["p (kN/m)"].max()

    y = group["y (m)"].to_numpy()
    p = group["p (kN/m)"].to_numpy()

    mask = np.isfinite(y) & np.isfinite(p)
    y = y[mask]
    p = p[mask]

    if len(y) < 3:
        return None

    idx_max = np.argmax(p)
    p_max = p[idx_max]
    y_at_pmax = y[idx_max]

    p10 = 0.10 * p_max
    early_mask = p <= p10

    y_init = y[early_mask]
    p_init = p[early_mask]

    if len(y_init) < 2:
        n_init = min(4, len(y))
        y_init = y[:n_init]
        p_init = p[:n_init]

    if len(np.unique(y_init)) >= 2:
        initial_stiffness = np.polyfit(y_init, p_init, 1)[0]
    else:
        initial_stiffness = np.nan

    if idx_max == len(p) - 1:
        curve_shape = "strain-hardening"
    elif p[-1] < 0.95 * p_max:
        curve_shape = "strain-softening"
    else:
        curve_shape = "approximately linear/plateau"

    return {
        "p_max": p_max,
        "y_at_pmax": y_at_pmax,
        "initial_stiffness": initial_stiffness,
        "curve_shape": curve_shape,
        "n_points": len(y)
    }

def save_bar_plot(summary_df, group_col, value_col, title, ylabel, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[value_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[value_col], color=colors)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.xlabel(group_col)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

# ==========================================================
# 2. PARSERS
# ==========================================================
def parse_py_sediment(file_path, sediment_order):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    df = df.rename(columns={
        df.columns[0]: "Sediment Type",
        df.columns[1]: "Point No."
    })

    df["Point No."] = pd.to_numeric(df["Point No."], errors="coerce")
    df["Sediment Type"] = df["Sediment Type"].astype(str).str.strip()

    df = df[df["Sediment Type"].isin(sediment_order)].copy()

    cols = list(df.columns)
    records = []

    for _, row in df.iterrows():
        sediment = row["Sediment Type"]
        point_no = row["Point No."]

        if pd.isna(sediment) or pd.isna(point_no):
            continue

        for i in range(2, len(cols) - 1, 2):
            p_col = cols[i]
            y_col = cols[i + 1]

            curve_value = safe_curve_value(p_col)
            if pd.isna(curve_value):
                continue

            p_val = pd.to_numeric(row[p_col], errors="coerce")
            y_val = pd.to_numeric(row[y_col], errors="coerce")

            if pd.notna(p_val) and pd.notna(y_val):
                records.append({
                    "Sediment Type": sediment,
                    "Curve": p_col,
                    "Curve Value": curve_value,
                    "Point No.": point_no,
                    "p (kN/m)": p_val,
                    "y (m)": y_val
                })

    return pd.DataFrame(records)

def parse_py_water(file_path, depth_order):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    df = df.rename(columns={
        df.columns[0]: "Water Depth",
        df.columns[1]: "Point No."
    })

    df["Point No."] = pd.to_numeric(df["Point No."], errors="coerce")
    df["Water Depth_num"] = df["Water Depth"].apply(extract_numeric_value)

    cols = list(df.columns)
    records = []

    for _, row in df.iterrows():
        water_depth_num = row["Water Depth_num"]
        point_no = row["Point No."]

        if pd.isna(water_depth_num) or pd.isna(point_no):
            continue
        if water_depth_num not in depth_order:
            continue

        for i in range(2, len(cols) - 1, 2):
            p_col = cols[i]
            y_col = cols[i + 1]

            curve_value = safe_curve_value(p_col)
            if pd.isna(curve_value):
                continue

            p_val = pd.to_numeric(row[p_col], errors="coerce")
            y_val = pd.to_numeric(row[y_col], errors="coerce")

            if pd.notna(p_val) and pd.notna(y_val):
                records.append({
                    "Water Depth": normalize_group_label(water_depth_num),
                    "Curve": p_col,
                    "Curve Value": curve_value,
                    "Point No.": point_no,
                    "p (kN/m)": p_val,
                    "y (m)": y_val
                })

    return pd.DataFrame(records)

def parse_py_embedment(file_path, embedment_order):
    df_raw = pd.read_csv(file_path)
    df_raw = clean_columns(df_raw)

    df = df_raw.rename(columns={
        df_raw.columns[0]: "Embedment Depth",
        df_raw.columns[1]: "Depth_or_No"
    }).copy()

    df["Embedment Depth"] = df["Embedment Depth"].astype(str).str.strip()
    df = df[df["Embedment Depth"].str.match(r"^\d+(\.\d+)?m$", na=False)].copy()
    df["Embedment Depth"] = df["Embedment Depth"].str.replace("m", "", regex=False).astype(float)

    df = df[df["Embedment Depth"].isin(embedment_order)].copy()

    all_cols = df.columns.tolist()
    data_cols = [c for c in all_cols if c not in ["Embedment Depth", "Depth_or_No"]]

    if len(data_cols) % 2 != 0:
        raise ValueError("Expected an even number of p-y data columns after the first two columns.")

    curve_pairs = []
    for i in range(0, len(data_cols), 2):
        p_col = data_cols[i]
        y_col = data_cols[i + 1]
        curve_pairs.append((p_col, y_col))

    long_records = []

    for embedment in sorted(df["Embedment Depth"].unique()):
        sub = df[df["Embedment Depth"] == embedment].copy()
        sub["Point No."] = pd.to_numeric(sub["Depth_or_No"], errors="coerce")

        for p_col, y_col in curve_pairs:
            curve_value = safe_curve_value(p_col)

            temp = pd.DataFrame({
                "Embedment Depth": normalize_group_label(embedment),
                "Curve": p_col,
                "Curve Value": curve_value,
                "Point No.": sub["Point No."],
                "p (kN/m)": pd.to_numeric(sub[p_col], errors="coerce"),
                "y (m)": pd.to_numeric(sub[y_col], errors="coerce")
            })

            temp = temp.dropna(subset=["Point No.", "p (kN/m)", "y (m)"]).copy()
            long_records.append(temp)

    return pd.concat(long_records, ignore_index=True)

# ==========================================================
# 3. MASTER FUNCTION
# ==========================================================
def run_py_analysis(config):
    print(f"\nRunning p-y analysis for: {config['name']}")
    output_dir = make_output_dir(config["output_dir"])

    if config["kind"] == "sediment":
        long_df = parse_py_sediment(config["input_csv"], config["group_order"])
    elif config["kind"] == "water":
        long_df = parse_py_water(config["input_csv"], config["group_order"])
    elif config["kind"] == "embedment":
        long_df = parse_py_embedment(config["input_csv"], config["group_order"])
    else:
        raise ValueError("Unknown config kind.")

    group_col = config["group_col_name"]
    order_str = [normalize_group_label(x) for x in config["group_order"]]
    selected_str = [normalize_group_label(x) for x in config["selected_groups"]]

    long_df[group_col] = long_df[group_col].astype(str).str.strip()
    long_df = long_df[long_df[group_col].isin(order_str)].copy()

    long_df = long_df.drop_duplicates()
    long_df = long_df.sort_values([group_col, "Curve Value", "Curve", "Point No."]).reset_index(drop=True)

    long_df.to_csv(os.path.join(output_dir, "p_y_long_cleaned.csv"), index=False)

    # ------------------------------------------------------
    # PER-CURVE STATS
    # ------------------------------------------------------
    stats_records = []

    for (group_value, curve, curve_value), sub in long_df.groupby([group_col, "Curve", "Curve Value"]):
        stats = compute_py_stats(sub)
        if stats is None:
            continue

        stats["Curve"] = curve
        stats["Curve Value"] = curve_value
        stats[group_col] = group_value
        stats_records.append(stats)

    stats_df = pd.DataFrame(stats_records)

    if stats_df.empty:
        raise ValueError(f"No valid p-y stats were computed for {config['name']}.")

    stats_df = stats_df[[group_col, "Curve", "Curve Value", "p_max", "y_at_pmax", "initial_stiffness", "curve_shape", "n_points"]]
    stats_df.to_csv(os.path.join(output_dir, "p_y_stats_by_group_and_curve.csv"), index=False)

    # ------------------------------------------------------
    # SUMMARY
    # ------------------------------------------------------
    summary_df = (
        stats_df.groupby(group_col, as_index=False)
        .agg(
            mean_p_max=("p_max", "mean"),
            mean_initial_stiffness=("initial_stiffness", "mean"),
            mean_y_at_pmax=("y_at_pmax", "mean")
        )
        .sort_values(group_col)
    )

    summary_df.to_csv(os.path.join(output_dir, "p_y_summary_by_group.csv"), index=False)

    shape_counts = (
        stats_df.groupby([group_col, "curve_shape"])
        .size()
        .reset_index(name="count")
        .sort_values([group_col, "curve_shape"])
    )
    shape_counts.to_csv(os.path.join(output_dir, "p_y_curve_shape_counts_by_group.csv"), index=False)

    # ------------------------------------------------------
    # ONE FIGURE PER GROUP: FULL AND ZOOMED
    # ------------------------------------------------------
    for group_value, sub in long_df.groupby(group_col):
        safe_name = str(group_value).replace(" ", "_")

        curve_pairs_sorted = (
            sub[["Curve", "Curve Value"]]
            .drop_duplicates()
            .sort_values(["Curve Value", "Curve"])
            .values.tolist()
        )

        # full
        plt.figure(figsize=(8, 6))
        for curve_name, curve_val in curve_pairs_sorted:
            curve_sub = sub[sub["Curve"] == curve_name].sort_values("Point No.")

            plt.plot(
                curve_sub["y (m)"],
                curve_sub["p (kN/m)"],
                linewidth=2,
                color=config["group_colors"].get(str(group_value), "gray"),
                label=str(curve_name)
            )

        plt.xlabel("y (m)")
        plt.ylabel("p (kN/m)")
        plt.title(f"p-y Curves for {config['name']} = {group_value}")
        plt.legend(title="Curve", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"p_y_{safe_name}_full.png"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(output_dir, f"p_y_{safe_name}_full.pdf"), bbox_inches="tight")
        plt.show()
        plt.close()

        # zoomed with full curve shown in gray
        plt.figure(figsize=(8, 6))
        for curve_name, curve_val in curve_pairs_sorted:
            curve_sub = sub[sub["Curve"] == curve_name].sort_values("Point No.")

            plt.plot(
                curve_sub["y (m)"],
                curve_sub["p (kN/m)"],
                color="lightgray",
                linestyle="--",
                linewidth=1.0,
                alpha=0.9
            )

            curve_zoom = curve_sub[curve_sub["y (m)"] <= config["zoom_limit"]]
            if curve_zoom.empty:
                continue

            plt.plot(
                curve_zoom["y (m)"],
                curve_zoom["p (kN/m)"],
                linewidth=2,
                color=config["group_colors"].get(str(group_value), "gray"),
                label=str(curve_name)
            )

        plt.xlabel("y (m)")
        plt.ylabel("p (kN/m)")
        plt.title(f"p-y Curves for {config['name']} = {group_value} (y ≤ {config['zoom_limit']} m; Full Curve Shown in Gray)")
        plt.xlim(0, config["zoom_limit"])
        plt.legend(title="Curve", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"p_y_{safe_name}_zoom.png"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(output_dir, f"p_y_{safe_name}_zoom.pdf"), bbox_inches="tight")
        plt.show()
        plt.close()

    # ------------------------------------------------------
    # SUMMARY BAR PLOTS
    # ------------------------------------------------------
    save_bar_plot(
        summary_df, group_col, "mean_p_max",
        f"Mean Ultimate Lateral Resistance by {config['name']}",
        "Mean p_max (kN/m)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "p_y_mean_pmax.png"),
        os.path.join(output_dir, "p_y_mean_pmax.pdf")
    )

    save_bar_plot(
        summary_df, group_col, "mean_initial_stiffness",
        f"Mean Initial p-y Stiffness by {config['name']}",
        "Mean Initial Stiffness",
        config["group_colors"], order_str,
        os.path.join(output_dir, "p_y_mean_stiffness.png"),
        os.path.join(output_dir, "p_y_mean_stiffness.pdf")
    )

    save_bar_plot(
        summary_df, group_col, "mean_y_at_pmax",
        f"Mean Displacement at Peak Lateral Resistance by {config['name']}",
        "Mean y at p_max (m)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "p_y_mean_y_at_pmax.png"),
        os.path.join(output_dir, "p_y_mean_y_at_pmax.pdf")
    )

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 4. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "kind": "sediment",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/p-y.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y",
    "group_col_name": "Sediment Type",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    },
    "selected_groups": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "zoom_limit": 2.0
}

water_config = {
    "name": "Water Depth",
    "kind": "water",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/p-y.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y",
    "group_col_name": "Water Depth",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    },
    "selected_groups": [10, 30, 50, 70, 90],
    "zoom_limit": 2.0
}

embedment_config = {
    "name": "Embedment Depth",
    "kind": "embedment",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/p-y.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y",
    "group_col_name": "Embedment Depth",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    },
    "selected_groups": [15, 25, 30, 35, 40, 45, 55],
    "zoom_limit": 2.0
}

# ==========================================================
# 5. RUN ALL THREE
# ==========================================================
run_py_analysis(sediment_config)
run_py_analysis(water_config)
run_py_analysis(embedment_config)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        return val.strip()
    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))
    return str(val).strip()

def rename_first_seven_columns(df, group_col):
    if df.shape[1] < 7:
        raise ValueError("Expected at least 7 columns in pile capacity file.")

    df = df.rename(columns={
        df.columns[0]: group_col,
        df.columns[1]: "No.",
        df.columns[2]: "Soil layer",
        df.columns[3]: "Depth (m)",
        df.columns[4]: "Shaft friction (kN)",
        df.columns[5]: "Base capacity (kN)",
        df.columns[6]: "Total capacity (kN)"
    })

    return df[[group_col, "No.", "Soil layer", "Depth (m)",
               "Shaft friction (kN)", "Base capacity (kN)", "Total capacity (kN)"]].copy()

def save_bar_plot(summary_df, group_col, value_col, title, ylabel, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[value_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[value_col], color=colors)
    plt.ylabel(ylabel)
    plt.xlabel(group_col)
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def save_stacked_percent_plot(summary_df, group_col, shaft_col, base_col, title, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[shaft_col, base_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[shaft_col], label="Shaft %", color=colors)
    plt.bar(plot_df[group_col], plot_df[base_col], bottom=plot_df[shaft_col], label="Base %", color="lightgray")
    plt.ylabel("Contribution (%)")
    plt.xlabel(group_col)
    plt.title(title)
    plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def save_stacked_absolute_plot(summary_df, group_col, shaft_col, base_col, title, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[shaft_col, base_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[shaft_col], label="Shaft friction", color=colors)
    plt.bar(plot_df[group_col], plot_df[base_col], bottom=plot_df[shaft_col], label="Base capacity", color="lightgray")
    plt.ylabel("Capacity (kN)")
    plt.xlabel(group_col)
    plt.title(title)
    plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

# ==========================================================
# 2. MASTER FUNCTION
# ==========================================================
def run_pile_capacity_analysis(config):
    print(f"\nRunning pile capacity analysis for: {config['name']}")

    group_col = config["group_col_name"]
    order_str = [normalize_group_label(x) for x in config["group_order"]]
    output_dir = make_output_dir(config["output_dir"])

    df = pd.read_csv(config["input_csv"])
    df = clean_columns(df)
    df = rename_first_seven_columns(df, group_col)

    # clean group
    if config["kind"] == "sediment":
        df[group_col] = df[group_col].astype(str).str.strip()
    else:
        df[f"{group_col}_num"] = df[group_col].apply(extract_numeric_value)
        df[group_col] = df[f"{group_col}_num"].apply(normalize_group_label)

    # numeric columns
    numeric_cols = ["No.", "Soil layer", "Depth (m)", "Shaft friction (kN)", "Base capacity (kN)", "Total capacity (kN)"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[group_col, "Depth (m)", "Shaft friction (kN)", "Base capacity (kN)", "Total capacity (kN)"]).copy()
    df = df[df[group_col].isin(order_str)].copy()

    if df.empty:
        raise ValueError(f"No valid rows remained for {config['name']} after cleaning/filtering.")

    # contributions
    df["shaft_pct"] = np.where(
        df["Total capacity (kN)"] > 0,
        df["Shaft friction (kN)"] / df["Total capacity (kN)"] * 100,
        np.nan
    )
    df["base_pct"] = np.where(
        df["Total capacity (kN)"] > 0,
        df["Base capacity (kN)"] / df["Total capacity (kN)"] * 100,
        np.nan
    )
    df["capacity_check"] = df["Shaft friction (kN)"] + df["Base capacity (kN)"] - df["Total capacity (kN)"]

    df = df.sort_values([group_col, "Depth (m)"]).reset_index(drop=True)
    df.to_csv(os.path.join(output_dir, "pile_capacity_cleaned.csv"), index=False)

    # ------------------------------------------------------
    # ULTIMATE SUMMARY: row of max total capacity per group
    # ------------------------------------------------------
    ultimate_rows = []

    for group_value in order_str:
        sub = df[df[group_col].astype(str) == str(group_value)].copy()
        if sub.empty:
            continue

        idx = sub["Total capacity (kN)"].idxmax()
        row = sub.loc[idx].copy()

        ultimate_rows.append({
            group_col: group_value,
            "Depth at ultimate (m)": row["Depth (m)"],
            "Soil layer at ultimate": row["Soil layer"],
            "Ultimate shaft friction (kN)": row["Shaft friction (kN)"],
            "Ultimate base capacity (kN)": row["Base capacity (kN)"],
            "Ultimate total capacity (kN)": row["Total capacity (kN)"],
            "Shaft contribution at ultimate (%)": row["shaft_pct"],
            "Base contribution at ultimate (%)": row["base_pct"]
        })

    ultimate_df = pd.DataFrame(ultimate_rows)
    if ultimate_df.empty:
        raise ValueError(f"No ultimate summary rows were created for {config['name']}.")

    ultimate_df.to_csv(os.path.join(output_dir, "pile_capacity_ultimate_summary_by_group.csv"), index=False)

    # ------------------------------------------------------
    # PROFILE SUMMARY: mean values across rows per group
    # ------------------------------------------------------
    profile_summary_df = (
        df.groupby(group_col, as_index=False)
        .agg(
            mean_shaft_friction_kN=("Shaft friction (kN)", "mean"),
            mean_base_capacity_kN=("Base capacity (kN)", "mean"),
            mean_total_capacity_kN=("Total capacity (kN)", "mean"),
            mean_shaft_pct=("shaft_pct", "mean"),
            mean_base_pct=("base_pct", "mean"),
            max_total_capacity_kN=("Total capacity (kN)", "max"),
            std_total_capacity_kN=("Total capacity (kN)", "std")
        )
    )

    profile_summary_df.to_csv(os.path.join(output_dir, "pile_capacity_profile_summary_by_group.csv"), index=False)

    # ------------------------------------------------------
    # PROFILE PLOTS
    # ------------------------------------------------------
    # total capacity vs depth
    plt.figure(figsize=(8, 6))
    for group_value in order_str:
        sub = df[df[group_col].astype(str) == str(group_value)].copy()
        if sub.empty:
            continue
        sub = sub.sort_values("Depth (m)")

        plt.plot(
            sub["Total capacity (kN)"],
            sub["Depth (m)"],
            color=config["group_colors"].get(str(group_value), "gray"),
            linewidth=2,
            marker="o",
            label=str(group_value)
        )

    plt.gca().invert_yaxis()
    plt.ylabel("Depth (m)")
    plt.xlabel("Total capacity (kN)")
    plt.title(f"Pile Capacity Profiles by {config['name']}")
    plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "pile_capacity_profiles.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "pile_capacity_profiles.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # shaft friction vs depth
    plt.figure(figsize=(8, 6))
    for group_value in order_str:
        sub = df[df[group_col].astype(str) == str(group_value)].copy()
        if sub.empty:
            continue
        sub = sub.sort_values("Depth (m)")

        plt.plot(
            sub["Shaft friction (kN)"],
            sub["Depth (m)"],
            color=config["group_colors"].get(str(group_value), "gray"),
            linewidth=2,
            marker="o",
            label=str(group_value)
        )

    plt.gca().invert_yaxis()
    plt.ylabel("Depth (m)")
    plt.xlabel("Shaft friction (kN)")
    plt.title(f"Shaft Friction Profiles by {config['name']}")
    plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "pile_shaft_friction_profiles.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "pile_shaft_friction_profiles.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # base capacity vs depth
    plt.figure(figsize=(8, 6))
    for group_value in order_str:
        sub = df[df[group_col].astype(str) == str(group_value)].copy()
        if sub.empty:
            continue
        sub = sub.sort_values("Depth (m)")

        plt.plot(
            sub["Base capacity (kN)"],
            sub["Depth (m)"],
            color=config["group_colors"].get(str(group_value), "gray"),
            linewidth=2,
            marker="o",
            label=str(group_value)
        )

    plt.gca().invert_yaxis()
    plt.ylabel("Depth (m)")
    plt.xlabel("Base capacity (kN)")
    plt.title(f"Base Capacity Profiles by {config['name']}")
    plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "pile_base_capacity_profiles.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "pile_base_capacity_profiles.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # SUMMARY BAR PLOTS
    # ------------------------------------------------------
    save_bar_plot(
        ultimate_df, group_col, "Ultimate total capacity (kN)",
        f"Ultimate Pile Capacity by {config['name']}",
        "Ultimate Total Capacity (kN)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "pile_capacity_ultimate_total.png"),
        os.path.join(output_dir, "pile_capacity_ultimate_total.pdf")
    )

    save_bar_plot(
        ultimate_df, group_col, "Ultimate shaft friction (kN)",
        f"Ultimate Shaft Friction by {config['name']}",
        "Ultimate Shaft Friction (kN)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "pile_capacity_ultimate_shaft.png"),
        os.path.join(output_dir, "pile_capacity_ultimate_shaft.pdf")
    )

    save_bar_plot(
        ultimate_df, group_col, "Ultimate base capacity (kN)",
        f"Ultimate Base Capacity by {config['name']}",
        "Ultimate Base Capacity (kN)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "pile_capacity_ultimate_base.png"),
        os.path.join(output_dir, "pile_capacity_ultimate_base.pdf")
    )

    save_stacked_absolute_plot(
        ultimate_df, group_col,
        "Ultimate shaft friction (kN)", "Ultimate base capacity (kN)",
        f"Ultimate Shaft vs Base Capacity by {config['name']}",
        config["group_colors"], order_str,
        os.path.join(output_dir, "pile_capacity_ultimate_absolute_components.png"),
        os.path.join(output_dir, "pile_capacity_ultimate_absolute_components.pdf")
    )

    save_stacked_percent_plot(
        ultimate_df, group_col,
        "Shaft contribution at ultimate (%)", "Base contribution at ultimate (%)",
        f"Ultimate Shaft vs Base Contribution by {config['name']}",
        config["group_colors"], order_str,
        os.path.join(output_dir, "pile_capacity_ultimate_percent_components.png"),
        os.path.join(output_dir, "pile_capacity_ultimate_percent_components.pdf")
    )

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 3. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "kind": "sediment",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/Pile_Capacity.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/pile_capacity",
    "group_col_name": "Sediment Type",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    }
}

water_config = {
    "name": "Water Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/pile_capacity.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/pile_capacity",
    "group_col_name": "Water Depth",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    }
}

embedment_config = {
    "name": "Embedment Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/pile_capacity.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/pile_capacity",
    "group_col_name": "Embedment Depth",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    }
}

# ==========================================================
# 4. RUN ALL THREE
# ==========================================================
run_pile_capacity_analysis(sediment_config)
run_pile_capacity_analysis(water_config)
run_pile_capacity_analysis(embedment_config)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        return val.strip()
    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))
    return str(val).strip()

def rename_first_six_columns(df, group_col):
    if df.shape[1] < 6:
        raise ValueError("Expected at least 6 columns in unit resistance file.")

    df = df.rename(columns={
        df.columns[0]: group_col,
        df.columns[1]: "No.",
        df.columns[2]: "Soil layer",
        df.columns[3]: "Depth (m)",
        df.columns[4]: "Unit shaft friction (kN/m²)",
        df.columns[5]: "Unit base resistance (kN/m²)"
    })

    return df[[group_col, "No.", "Soil layer", "Depth (m)",
               "Unit shaft friction (kN/m²)", "Unit base resistance (kN/m²)"]].copy()

def save_bar_plot(summary_df, group_col, value_col, title, ylabel, colors_map, order, out_png, out_pdf, invert_y=False):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[value_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[value_col], color=colors)
    if invert_y:
        plt.gca().invert_yaxis()
    plt.ylabel(ylabel)
    plt.xlabel(group_col)
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

# ==========================================================
# 2. MASTER FUNCTION
# ==========================================================
def run_unit_resistance_analysis(config):
    print(f"\nRunning unit resistance analysis for: {config['name']}")

    group_col = config["group_col_name"]
    order_str = [normalize_group_label(x) for x in config["group_order"]]
    output_dir = make_output_dir(config["output_dir"])

    df = pd.read_csv(config["input_csv"])
    df = clean_columns(df)
    df = rename_first_six_columns(df, group_col)

    # clean group column
    if config["kind"] == "sediment":
        df[group_col] = df[group_col].astype(str).str.strip()
    else:
        df[f"{group_col}_num"] = df[group_col].apply(extract_numeric_value)
        df[group_col] = df[f"{group_col}_num"].apply(normalize_group_label)

    # numeric columns
    numeric_cols = [
        "No.",
        "Soil layer",
        "Depth (m)",
        "Unit shaft friction (kN/m²)",
        "Unit base resistance (kN/m²)"
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[group_col, "Depth (m)", "Unit shaft friction (kN/m²)", "Unit base resistance (kN/m²)"]).copy()
    df = df[df[group_col].isin(order_str)].copy()

    if df.empty:
        raise ValueError(f"No valid rows remained for {config['name']} after cleaning/filtering.")

    df = df.sort_values([group_col, "Depth (m)", "Soil layer"]).reset_index(drop=True)
    df.to_csv(os.path.join(output_dir, "unit_resistance_cleaned.csv"), index=False)

    # ------------------------------------------------------
    # OVERALL SUMMARY BY GROUP
    # ------------------------------------------------------
    summary_df = (
        df.groupby(group_col, as_index=False)
        .agg(
            mean_unit_shaft=("Unit shaft friction (kN/m²)", "mean"),
            std_unit_shaft=("Unit shaft friction (kN/m²)", "std"),
            var_unit_shaft=("Unit shaft friction (kN/m²)", "var"),
            min_unit_shaft=("Unit shaft friction (kN/m²)", "min"),
            max_unit_shaft=("Unit shaft friction (kN/m²)", "max"),

            mean_unit_base=("Unit base resistance (kN/m²)", "mean"),
            std_unit_base=("Unit base resistance (kN/m²)", "std"),
            var_unit_base=("Unit base resistance (kN/m²)", "var"),
            min_unit_base=("Unit base resistance (kN/m²)", "min"),
            max_unit_base=("Unit base resistance (kN/m²)", "max")
        )
    )

    summary_df["cv_unit_shaft"] = summary_df["std_unit_shaft"] / summary_df["mean_unit_shaft"]
    summary_df["cv_unit_base"] = summary_df["std_unit_base"] / summary_df["mean_unit_base"]

    summary_df.to_csv(os.path.join(output_dir, "unit_resistance_summary_by_group.csv"), index=False)

    # ------------------------------------------------------
    # LAYER SUMMARY
    # ------------------------------------------------------
    layer_summary_df = (
        df.groupby([group_col, "Soil layer"], as_index=False)
        .agg(
            mean_unit_shaft=("Unit shaft friction (kN/m²)", "mean"),
            std_unit_shaft=("Unit shaft friction (kN/m²)", "std"),
            var_unit_shaft=("Unit shaft friction (kN/m²)", "var"),

            mean_unit_base=("Unit base resistance (kN/m²)", "mean"),
            std_unit_base=("Unit base resistance (kN/m²)", "std"),
            var_unit_base=("Unit base resistance (kN/m²)", "var"),

            min_depth=("Depth (m)", "min"),
            max_depth=("Depth (m)", "max")
        )
        .sort_values([group_col, "Soil layer"])
    )

    layer_summary_df.to_csv(os.path.join(output_dir, "unit_resistance_layer_summary_by_group.csv"), index=False)

    # ------------------------------------------------------
    # DEPTH TREND TABLE
    # ------------------------------------------------------
    depth_trend_df = (
        df.groupby([group_col, "Depth (m)"], as_index=False)
        .agg(
            mean_unit_shaft=("Unit shaft friction (kN/m²)", "mean"),
            mean_unit_base=("Unit base resistance (kN/m²)", "mean"),
            n_layers=("Soil layer", "nunique")
        )
        .sort_values([group_col, "Depth (m)"])
    )

    depth_trend_df.to_csv(os.path.join(output_dir, "unit_resistance_depth_trends_by_group.csv"), index=False)

    # ------------------------------------------------------
    # STRONGEST DEPTH TABLES
    # ------------------------------------------------------
    shaft_idx = depth_trend_df.groupby(group_col)["mean_unit_shaft"].idxmax()
    base_idx = depth_trend_df.groupby(group_col)["mean_unit_base"].idxmax()

    max_shaft_df = depth_trend_df.loc[shaft_idx].copy().sort_values(group_col).reset_index(drop=True)
    max_base_df = depth_trend_df.loc[base_idx].copy().sort_values(group_col).reset_index(drop=True)

    max_shaft_df = max_shaft_df.rename(columns={
        "Depth (m)": "Depth of max mean unit shaft friction (m)",
        "mean_unit_shaft": "Maximum mean unit shaft friction"
    })[[group_col, "Depth of max mean unit shaft friction (m)", "Maximum mean unit shaft friction"]]

    max_base_df = max_base_df.rename(columns={
        "Depth (m)": "Depth of max mean unit base resistance (m)",
        "mean_unit_base": "Maximum mean unit base resistance"
    })[[group_col, "Depth of max mean unit base resistance (m)", "Maximum mean unit base resistance"]]

    max_shaft_df.to_csv(os.path.join(output_dir, "max_unit_shaft_depth_by_group.csv"), index=False)
    max_base_df.to_csv(os.path.join(output_dir, "max_unit_base_depth_by_group.csv"), index=False)

    # ------------------------------------------------------
    # PROFILE PLOTS
    # ------------------------------------------------------
    # shaft
    plt.figure(figsize=(8, 7))
    for group_value in order_str:
        sub = df[df[group_col].astype(str) == str(group_value)].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("Depth (m)")
        plt.plot(
            sub["Unit shaft friction (kN/m²)"],
            sub["Depth (m)"],
            marker="o",
            linewidth=2,
            color=config["group_colors"].get(str(group_value), "gray"),
            label=str(group_value)
        )

    plt.gca().invert_yaxis()
    plt.xlabel("Unit Shaft Friction (kN/m²)")
    plt.ylabel("Depth (m)")
    plt.title(f"Unit Shaft Friction vs Depth by {config['name']}")
    plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "unit_shaft_friction_profiles.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "unit_shaft_friction_profiles.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # base
    plt.figure(figsize=(8, 7))
    for group_value in order_str:
        sub = df[df[group_col].astype(str) == str(group_value)].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("Depth (m)")
        plt.plot(
            sub["Unit base resistance (kN/m²)"],
            sub["Depth (m)"],
            marker="o",
            linewidth=2,
            color=config["group_colors"].get(str(group_value), "gray"),
            label=str(group_value)
        )

    plt.gca().invert_yaxis()
    plt.xlabel("Unit Base Resistance (kN/m²)")
    plt.ylabel("Depth (m)")
    plt.title(f"Unit Base Resistance vs Depth by {config['name']}")
    plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "unit_base_resistance_profiles.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "unit_base_resistance_profiles.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # BAR PLOTS
    # ------------------------------------------------------
    save_bar_plot(
        summary_df, group_col, "cv_unit_shaft",
        f"Heterogeneity in Unit Shaft Friction by {config['name']}",
        "Coefficient of Variation",
        config["group_colors"], order_str,
        os.path.join(output_dir, "cv_unit_shaft.png"),
        os.path.join(output_dir, "cv_unit_shaft.pdf")
    )

    save_bar_plot(
        summary_df, group_col, "cv_unit_base",
        f"Heterogeneity in Unit Base Resistance by {config['name']}",
        "Coefficient of Variation",
        config["group_colors"], order_str,
        os.path.join(output_dir, "cv_unit_base.png"),
        os.path.join(output_dir, "cv_unit_base.pdf")
    )

    save_bar_plot(
        summary_df, group_col, "mean_unit_shaft",
        f"Mean Unit Shaft Friction by {config['name']}",
        "Mean Unit Shaft Friction (kN/m²)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "mean_unit_shaft.png"),
        os.path.join(output_dir, "mean_unit_shaft.pdf")
    )

    save_bar_plot(
        summary_df, group_col, "mean_unit_base",
        f"Mean Unit Base Resistance by {config['name']}",
        "Mean Unit Base Resistance (kN/m²)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "mean_unit_base.png"),
        os.path.join(output_dir, "mean_unit_base.pdf")
    )

    save_bar_plot(
        max_shaft_df, group_col, "Depth of max mean unit shaft friction (m)",
        f"Depth of Maximum Mean Unit Shaft Friction by {config['name']}",
        "Depth (m)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "depth_of_max_unit_shaft.png"),
        os.path.join(output_dir, "depth_of_max_unit_shaft.pdf"),
        invert_y=True
    )

    save_bar_plot(
        max_base_df, group_col, "Depth of max mean unit base resistance (m)",
        f"Depth of Maximum Mean Unit Base Resistance by {config['name']}",
        "Depth (m)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "depth_of_max_unit_base.png"),
        os.path.join(output_dir, "depth_of_max_unit_base.pdf"),
        invert_y=True
    )

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 3. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "kind": "sediment",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/Unit_Resistance.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/unit_resistance",
    "group_col_name": "Sediment Type",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    }
}

water_config = {
    "name": "Water Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/unit_resistance.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/unit_resistance",
    "group_col_name": "Water Depth",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    }
}

embedment_config = {
    "name": "Embedment Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/unit_resistance.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/unit_resistance",
    "group_col_name": "Embedment Depth",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    }
}

# ==========================================================
# 4. RUN ALL THREE
# ==========================================================
run_unit_resistance_analysis(sediment_config)
run_unit_resistance_analysis(water_config)
run_unit_resistance_analysis(embedment_config)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        return val.strip()
    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))
    return str(val).strip()

def rename_first_seven_columns(df, group_col):
    if df.shape[1] < 7:
        raise ValueError("Expected at least 7 columns in pile settlement file.")

    df = df.rename(columns={
        df.columns[0]: group_col,
        df.columns[1]: "Load-case",
        df.columns[2]: "Axial load (kN)",
        df.columns[3]: "Pile head settlement (mm)",
        df.columns[4]: "Pile base settlement (mm)",
        df.columns[5]: "Pile shaft settlement (mm)",
        df.columns[6]: "Status"
    })

    return df[
        [
            group_col,
            "Load-case",
            "Axial load (kN)",
            "Pile head settlement (mm)",
            "Pile base settlement (mm)",
            "Pile shaft settlement (mm)",
            "Status"
        ]
    ].copy()

def classify_curve(load, settlement):
    load = np.array(load, dtype=float)
    s = np.array(settlement, dtype=float)

    mask = np.isfinite(load) & np.isfinite(s)
    load = load[mask]
    s = s[mask]

    if len(load) < 3:
        return "unknown"

    order = np.argsort(load)
    load = load[order]
    s = s[order]

    dload = np.diff(load)
    dsettle = np.diff(s)

    valid = dload != 0
    dload = dload[valid]
    dsettle = dsettle[valid]

    if len(dload) < 2:
        return "unknown"

    slopes = dsettle / dload
    slope_start = slopes[0]
    slope_end = slopes[-1]

    if slope_end > slope_start * 1.2:
        return "nonlinear-hardening"
    elif slope_end < slope_start * 0.8:
        return "nonlinear-softening"
    else:
        return "approximately linear"

def save_bar_plot(summary_df, group_col, value_col, title, ylabel, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[value_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[value_col], color=colors)
    plt.ylabel(ylabel)
    plt.xlabel(group_col)
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def plot_head_base_boxplots_side_by_side(
    df,
    group_col,
    group_order,
    group_colors,
    title,
    out_png,
    out_pdf
):
    df = df.copy()

    axial_order = sorted(df["Axial load (kN)"].dropna().unique())
    if len(axial_order) == 0:
        raise ValueError("No axial loads found for boxplot figure.")

    fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharex=True)

    plot_specs = [
        ("Pile head settlement (mm)", "Pile Head Settlement (mm)", "Pile Head Settlement"),
        ("Pile base settlement (mm)", "Pile Base Settlement (mm)", "Pile Base Settlement")
    ]

    offsets = np.linspace(-0.20, 0.20, max(len(group_order), 2))
    offset_map = {str(g): offsets[i] for i, g in enumerate(group_order)}

    for ax, (value_col, ylab, panel_title) in zip(axes, plot_specs):
        # boxplot data by axial load
        box_data = []
        for load in axial_order:
            vals = df.loc[df["Axial load (kN)"] == load, value_col].dropna().values
            box_data.append(vals)

        positions = np.arange(1, len(axial_order) + 1)

        bp = ax.boxplot(
            box_data,
            positions=positions,
            widths=0.55,
            patch_artist=True,
            showfliers=False
        )

        for patch in bp["boxes"]:
            patch.set_facecolor("whitesmoke")
            patch.set_edgecolor("black")

        for median in bp["medians"]:
            median.set_color("black")
            median.set_linewidth(1.5)

        # overlay points colored by group
        for load_i, load in enumerate(axial_order, start=1):
            sub_load = df[df["Axial load (kN)"] == load].copy()

            for g in group_order:
                g_str = str(g)
                sub = sub_load[sub_load[group_col].astype(str) == g_str]

                if sub.empty:
                    continue

                yvals = sub[value_col].dropna().values
                if len(yvals) == 0:
                    continue

                xvals = np.full(len(yvals), load_i + offset_map[g_str])

                ax.scatter(
                    xvals,
                    yvals,
                    color=group_colors.get(g_str, "gray"),
                    s=65,
                    zorder=3
                )

        ax.set_xticks(positions)
        ax.set_xticklabels(
            [f"{int(x):,}" if float(x).is_integer() else f"{x:g}" for x in axial_order],
            rotation=0
        )
        ax.set_xlabel("Axial load (kN)")
        ax.set_ylabel(ylab)
        ax.set_title(panel_title)
        ax.grid(True, axis="y", alpha=0.3)

    handles = [
        axes[1].scatter([], [], color=group_colors.get(str(g), "gray"), s=65, label=str(g))
        for g in group_order
    ]

    fig.legend(
        handles=handles,
        title=group_col,
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

# ==========================================================
# 2. MASTER FUNCTION
# ==========================================================
def run_pile_settlement_analysis(config):
    print(f"\nRunning pile settlement analysis for: {config['name']}")

    group_col = config["group_col_name"]
    order_str = [normalize_group_label(x) for x in config["group_order"]]
    output_dir = make_output_dir(config["output_dir"])

    df = pd.read_csv(config["input_csv"])
    df = clean_columns(df)
    df = rename_first_seven_columns(df, group_col)

    # clean grouping variable
    if config["kind"] == "sediment":
        df[group_col] = df[group_col].astype(str).str.strip()
    else:
        df[f"{group_col}_num"] = df[group_col].apply(extract_numeric_value)
        df[group_col] = df[f"{group_col}_num"].apply(normalize_group_label)

    # numeric cleanup
    num_cols = [
        "Load-case",
        "Axial load (kN)",
        "Pile head settlement (mm)",
        "Pile base settlement (mm)",
        "Pile shaft settlement (mm)"
    ]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Status"] = df["Status"].astype(str).str.strip().str.upper()

    df = df[
        (df["Status"] == "OK") &
        (df[group_col].isin(order_str))
    ].copy()

    df = df.dropna(
        subset=[
            group_col,
            "Load-case",
            "Axial load (kN)",
            "Pile head settlement (mm)",
            "Pile base settlement (mm)",
            "Pile shaft settlement (mm)"
        ]
    ).copy()

    if df.empty:
        raise ValueError(f"No valid OK rows remained for {config['name']}.")

    # settlement partitioning
    df["shaft_from_difference_mm"] = (
        df["Pile head settlement (mm)"] - df["Pile base settlement (mm)"]
    )

    df["base_pct"] = np.where(
        df["Pile head settlement (mm)"] != 0,
        df["Pile base settlement (mm)"] / df["Pile head settlement (mm)"] * 100,
        np.nan
    )

    df["shaft_pct"] = np.where(
        df["Pile head settlement (mm)"] != 0,
        df["Pile shaft settlement (mm)"] / df["Pile head settlement (mm)"] * 100,
        np.nan
    )

    df["closure_check_mm"] = (
        df["Pile base settlement (mm)"] +
        df["Pile shaft settlement (mm)"] -
        df["Pile head settlement (mm)"]
    )

    df = df.sort_values([group_col, "Load-case", "Axial load (kN)"]).reset_index(drop=True)
    df.to_csv(os.path.join(output_dir, "pile_settlement_cleaned.csv"), index=False)

    # ------------------------------------------------------
    # STATS BY GROUP
    # ------------------------------------------------------
    stats = []

    for group_value, group in df.groupby(group_col):
        group = group.sort_values("Axial load (kN)")

        load = group["Axial load (kN)"]
        head = group["Pile head settlement (mm)"]
        base = group["Pile base settlement (mm)"]
        shaft = group["Pile shaft settlement (mm)"]

        max_head = head.max()
        max_base = base.max()
        max_shaft = shaft.max()

        design = group[group["Load-case"] == 1]
        if not design.empty:
            head_design = design["Pile head settlement (mm)"].iloc[0]
            base_design = design["Pile base settlement (mm)"].iloc[0]
            shaft_design = design["Pile shaft settlement (mm)"].iloc[0]
            design_load = design["Axial load (kN)"].iloc[0]
            design_base_pct = design["base_pct"].iloc[0]
            design_shaft_pct = design["shaft_pct"].iloc[0]
        else:
            head_design = np.nan
            base_design = np.nan
            shaft_design = np.nan
            design_load = np.nan
            design_base_pct = np.nan
            design_shaft_pct = np.nan

        lc3 = group[group["Load-case"] == 3]
        if not lc3.empty:
            head_lc3 = lc3["Pile head settlement (mm)"].iloc[0]
            base_lc3 = lc3["Pile base settlement (mm)"].iloc[0]
            shaft_lc3 = lc3["Pile shaft settlement (mm)"].iloc[0]
            lc3_load = lc3["Axial load (kN)"].iloc[0]
            lc3_base_pct = lc3["base_pct"].iloc[0]
            lc3_shaft_pct = lc3["shaft_pct"].iloc[0]
        else:
            head_lc3 = np.nan
            base_lc3 = np.nan
            shaft_lc3 = np.nan
            lc3_load = np.nan
            lc3_base_pct = np.nan
            lc3_shaft_pct = np.nan

        shape = classify_curve(load, head)

        stats.append({
            group_col: group_value,
            "Max Head (mm)": max_head,
            "Max Base (mm)": max_base,
            "Max Shaft (mm)": max_shaft,
            "Design Load (kN)": design_load,
            "Design Head (mm)": head_design,
            "Design Base (mm)": base_design,
            "Design Shaft (mm)": shaft_design,
            "Design Base (%)": design_base_pct,
            "Design Shaft (%)": design_shaft_pct,
            "LC3 Load (kN)": lc3_load,
            "LC3 Head (mm)": head_lc3,
            "LC3 Base (mm)": base_lc3,
            "LC3 Shaft (mm)": shaft_lc3,
            "LC3 Base (%)": lc3_base_pct,
            "LC3 Shaft (%)": lc3_shaft_pct,
            "Curve Shape": shape,
            "Head-Base Difference (mm)": max_head - max_base,
            "Head Range (mm)": head.max() - head.min(),
            "Base Range (mm)": base.max() - base.min(),
            "Shaft Range (mm)": shaft.max() - shaft.min(),
            "Mean Closure Check (mm)": group["closure_check_mm"].mean()
        })

    stats_df = pd.DataFrame(stats).sort_values(group_col).reset_index(drop=True)
    stats_df.to_csv(os.path.join(output_dir, "pile_settlement_stats_by_group.csv"), index=False)

    # ------------------------------------------------------
    # SUMMARY ACROSS ALL LOAD CASES
    # ------------------------------------------------------
    summary_df = (
        df.groupby(group_col, as_index=False)
        .agg(
            mean_head_settlement_mm=("Pile head settlement (mm)", "mean"),
            std_head_settlement_mm=("Pile head settlement (mm)", "std"),
            mean_base_settlement_mm=("Pile base settlement (mm)", "mean"),
            std_base_settlement_mm=("Pile base settlement (mm)", "std"),
            mean_shaft_settlement_mm=("Pile shaft settlement (mm)", "mean"),
            std_shaft_settlement_mm=("Pile shaft settlement (mm)", "std"),
            mean_base_pct=("base_pct", "mean"),
            mean_shaft_pct=("shaft_pct", "mean"),
            max_head_settlement_mm=("Pile head settlement (mm)", "max"),
            min_head_settlement_mm=("Pile head settlement (mm)", "min")
        )
        .sort_values(group_col)
    )

    summary_df.to_csv(os.path.join(output_dir, "pile_settlement_summary_by_group.csv"), index=False)

    # ------------------------------------------------------
    # DESIGN HEAD WITH BASE MARKER
    # ------------------------------------------------------
    plot_df = stats_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex(order_str).reset_index()
    plot_df = plot_df.dropna(subset=["Design Head (mm)"])

    x_labels = plot_df[group_col]
    bar_colors = [config["group_colors"].get(str(x), "gray") for x in x_labels]

    plt.figure(figsize=(10, 6))
    plt.bar(x_labels, plot_df["Design Head (mm)"], color=bar_colors)

    plt.scatter(
        x_labels,
        plot_df["Design Base (mm)"],
        color="black",
        marker="o",
        s=50,
        zorder=3,
        label="Base settlement"
    )

    plt.ylabel("Settlement at Design Load (mm)")
    plt.xlabel(group_col)
    plt.title(f"Pile Head Settlement at Design Load with Base Settlement Marker by {config['name']}")
    plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "design_head_with_base_marker.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "design_head_with_base_marker.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # STACKED PERCENT: DESIGN
    # ------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.bar(
        x_labels,
        plot_df["Design Base (%)"],
        color=bar_colors,
        label="Base contribution (%)"
    )

    plt.bar(
        x_labels,
        plot_df["Design Shaft (%)"],
        bottom=plot_df["Design Base (%)"],
        color="lightgray",
        label="Shaft contribution (%)"
    )

    plt.ylabel("Percent of Total Head Settlement (%)")
    plt.xlabel(group_col)
    plt.title(f"Base Contribution vs Shaft Settlement at Design Load by {config['name']}")
    plt.ylim(0, 100)
    plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "base_vs_shaft_percent_design.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "base_vs_shaft_percent_design.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # STACKED PERCENT: MEAN
    # ------------------------------------------------------
    mean_plot_df = summary_df.copy()
    mean_plot_df[group_col] = mean_plot_df[group_col].astype(str)
    mean_plot_df = mean_plot_df.set_index(group_col).reindex(order_str).reset_index()
    mean_plot_df = mean_plot_df.dropna(subset=["mean_base_pct", "mean_shaft_pct"])

    mean_labels = mean_plot_df[group_col]
    mean_colors = [config["group_colors"].get(str(x), "gray") for x in mean_labels]

    plt.figure(figsize=(10, 6))
    plt.bar(
        mean_labels,
        mean_plot_df["mean_base_pct"],
        color=mean_colors,
        label="Mean base contribution (%)"
    )

    plt.bar(
        mean_labels,
        mean_plot_df["mean_shaft_pct"],
        bottom=mean_plot_df["mean_base_pct"],
        color="lightgray",
        label="Mean shaft contribution (%)"
    )

    plt.ylabel("Percent of Total Head Settlement (%)")
    plt.xlabel(group_col)
    plt.title(f"Mean Base vs Shaft Settlement Contribution Across Load Cases by {config['name']}")
    plt.ylim(0, 100)
    plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "base_vs_shaft_percent_mean.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "base_vs_shaft_percent_mean.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()


   # ------------------------------------------------------
    # SIDE-BY-SIDE BOXPLOTS: HEAD AND BASE VS AXIAL LOAD
    # ------------------------------------------------------
    plot_head_base_boxplots_side_by_side(
        df=df,
        group_col=group_col,
        group_order=order_str,
        group_colors=config["group_colors"],
        title=f"Pile Head and Base Settlement vs Axial Load by {config['name']}",
        out_png=os.path.join(output_dir, "head_base_settlement_boxplots_vs_axial_load.png"),
        out_pdf=os.path.join(output_dir, "head_base_settlement_boxplots_vs_axial_load.pdf")
    )

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 3. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "kind": "sediment",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/Pile_Settlement.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/pile_settlement",
    "group_col_name": "Sediment Type",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    }
}

water_config = {
    "name": "Water Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/pile_settlement.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/pile_settlement",
    "group_col_name": "Water Depth",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    }
}

embedment_config = {
    "name": "Embedment Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/pile_settlement.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/pile_settlement",
    "group_col_name": "Embedment Depth",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    }
}

# ==========================================================
# 4. RUN ALL THREE
# ==========================================================
run_pile_settlement_analysis(sediment_config)
run_pile_settlement_analysis(water_config)
run_pile_settlement_analysis(embedment_config)


# ==========================================================
# 5. EXTRA SEDIMENT BOXPLOT EXCLUDING sandy_silt and slc_sand_slc
# ==========================================================
excluded_sediments = ["sandy_silt", "slc_sand_slc"]

sediment_df_full = pd.read_csv(sediment_config["input_csv"])
sediment_df_full = clean_columns(sediment_df_full)
sediment_df_full = rename_first_seven_columns(sediment_df_full, "Sediment Type")

sediment_df_full["Sediment Type"] = sediment_df_full["Sediment Type"].astype(str).str.strip()

num_cols = [
    "Load-case",
    "Axial load (kN)",
    "Pile head settlement (mm)",
    "Pile base settlement (mm)",
    "Pile shaft settlement (mm)"
]
for col in num_cols:
    sediment_df_full[col] = pd.to_numeric(sediment_df_full[col], errors="coerce")

sediment_df_full["Status"] = sediment_df_full["Status"].astype(str).str.strip().str.upper()

sediment_df_full = sediment_df_full[
    (sediment_df_full["Status"] == "OK") &
    (~sediment_df_full["Sediment Type"].isin(excluded_sediments))
].copy()

sediment_zoom_order = [s for s in sediment_config["group_order"] if s not in excluded_sediments]
sediment_zoom_colors = {k: v for k, v in sediment_config["group_colors"].items() if k in sediment_zoom_order}

plot_head_base_boxplots_side_by_side(
    df=sediment_df_full,
    group_col="Sediment Type",
    group_order=sediment_zoom_order,
    group_colors=sediment_zoom_colors,
    title="Pile Head and Base Settlement vs Axial Load by Sediment Type (Excluding sandy_silt and slc_sand_slc)",
    out_png=os.path.join(sediment_config["output_dir"], "head_base_settlement_boxplots_vs_axial_load_excluding_outliers.png"),
    out_pdf=os.path.join(sediment_config["output_dir"], "head_base_settlement_boxplots_vs_axial_load_excluding_outliers.pdf")
)


# ==========================================================
# 6 EXTRA EMBEDMENT BOXPLOT EXCLUDING 15 m
# ==========================================================
excluded_embedment = ["15"]

embed_df_full = pd.read_csv(embedment_config["input_csv"])
embed_df_full = clean_columns(embed_df_full)
embed_df_full = rename_first_seven_columns(embed_df_full, "Embedment Depth")

# Convert embedment to numeric + normalized string
embed_df_full["Embedment Depth_num"] = embed_df_full["Embedment Depth"].apply(extract_numeric_value)
embed_df_full["Embedment Depth"] = embed_df_full["Embedment Depth_num"].apply(normalize_group_label)

# Convert numeric columns
num_cols = [
    "Load-case",
    "Axial load (kN)",
    "Pile head settlement (mm)",
    "Pile base settlement (mm)",
    "Pile shaft settlement (mm)"
]

for col in num_cols:
    embed_df_full[col] = pd.to_numeric(embed_df_full[col], errors="coerce")

# Clean status
embed_df_full["Status"] = embed_df_full["Status"].astype(str).str.strip().str.upper()

# Filter valid + remove 15 m
embed_df_full = embed_df_full[
    (embed_df_full["Status"] == "OK") &
    (~embed_df_full["Embedment Depth"].isin(excluded_embedment))
].copy()

embed_df_full = embed_df_full.dropna(
    subset=[
        "Embedment Depth",
        "Axial load (kN)",
        "Pile head settlement (mm)",
        "Pile base settlement (mm)"
    ]
)

# Updated order + colors
embedment_zoom_order = [str(x) for x in embedment_config["group_order"] if str(x) != "15"]
embedment_zoom_colors = {
    k: v for k, v in embedment_config["group_colors"].items()
    if k in embedment_zoom_order
}

# ----------------------------------------------------------
# PLOT
# ----------------------------------------------------------
plot_head_base_boxplots_side_by_side(
    df=embed_df_full,
    group_col="Embedment Depth",
    group_order=embedment_zoom_order,
    group_colors=embedment_zoom_colors,
    title="Pile Head and Base Settlement vs Axial Load by Embedment Depth (Excluding 15 m)",
    out_png=os.path.join(embedment_config["output_dir"], "head_base_settlement_boxplots_vs_axial_load_excluding_15m.png"),
    out_pdf=os.path.join(embedment_config["output_dir"], "head_base_settlement_boxplots_vs_axial_load_excluding_15m.pdf")
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        return val.strip()
    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))
    return str(val).strip()

def classify_curve(depth, settlement):
    depth = np.array(depth, dtype=float)
    s = np.array(settlement, dtype=float)

    mask = np.isfinite(depth) & np.isfinite(s)
    depth = depth[mask]
    s = s[mask]

    if len(depth) < 5:
        return "unknown"

    order = np.argsort(depth)
    depth = depth[order]
    s = s[order]

    ddepth = np.diff(depth)
    ds = np.diff(s)

    valid = ddepth != 0
    ddepth = ddepth[valid]
    ds = ds[valid]

    if len(ddepth) < 2:
        return "unknown"

    slopes = ds / ddepth
    slope_start = slopes[0]
    slope_end = slopes[-1]

    if slope_end > slope_start * 1.2:
        return "nonlinear-hardening"
    elif slope_end < slope_start * 0.8:
        return "nonlinear-softening"
    else:
        return "approximately linear"

def save_bar_plot(summary_df, group_col, value_col, title, ylabel, colors_map, order, out_png, out_pdf):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[value_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[value_col], color=colors)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.xlabel(group_col)
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def filter_out_sediment_outliers(df, group_col, exclude_list):
    return df[~df[group_col].astype(str).isin(exclude_list)].copy()

# ==========================================================
# 2. MASTER FUNCTION
# ==========================================================
def run_axial_settlement_analysis(config):
    print(f"\nRunning axial settlement analysis for: {config['name']}")

    group_col = config["group_col_name"]
    order_str = [normalize_group_label(x) for x in config["group_order"]]
    output_dir = make_output_dir(config["output_dir"])

    df = pd.read_csv(config["input_csv"])
    df = clean_columns(df)

    if df.shape[1] < 6:
        raise ValueError("Axial settlement file does not have enough columns.")

    # rename first columns by position
    rename_map = {
        df.columns[0]: group_col,
        df.columns[1]: "Node",
        df.columns[2]: "Depth (m)"
    }

    # rename remaining columns as LC1, LC2, LC3...
    remaining_cols = list(df.columns[3:])
    lc_names = [f"LC{i+1}" for i in range(len(remaining_cols))]
    for old, new in zip(remaining_cols, lc_names):
        rename_map[old] = new

    df = df.rename(columns=rename_map)

    # group cleaning
    if config["kind"] == "sediment":
        df[group_col] = df[group_col].astype(str).str.strip()
    else:
        df[f"{group_col}_num"] = df[group_col].apply(extract_numeric_value)
        df[group_col] = df[f"{group_col}_num"].apply(normalize_group_label)

    # numeric conversion
    numeric_cols = ["Node", "Depth (m)"]
    lc_cols = [c for c in df.columns if c.startswith("LC")]
    numeric_cols.extend(lc_cols)

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # require LC1 and LC3
    for lc in ["LC1", "LC3"]:
        if lc not in df.columns:
            raise ValueError(f"{lc} not found in file for {config['name']}")

    df = df.dropna(subset=[group_col, "Depth (m)", "LC1", "LC3"]).copy()
    df = df[df[group_col].isin(order_str)].copy()

    if df.empty:
        raise ValueError(f"No valid rows remained for {config['name']}.")

    df = df.sort_values([group_col, "Depth (m)", "Node"]).reset_index(drop=True)
    df.to_csv(os.path.join(output_dir, "axial_settlement_cleaned.csv"), index=False)

    # ------------------------------------------------------
    # STATS
    # ------------------------------------------------------
    stats = []

    for group_value, group in df.groupby(group_col):
        group = group.sort_values("Depth (m)")

        depth = group["Depth (m)"]
        lc1 = group["LC1"]
        lc3 = group["LC3"]

        max_lc1 = lc1.max()
        max_lc3 = lc3.max()

        top = group[group["Depth (m)"] == group["Depth (m)"].min()]
        if not top.empty:
            lc1_design = top["LC1"].iloc[0]
            lc3_design = top["LC3"].iloc[0]
        else:
            lc1_design = np.nan
            lc3_design = np.nan

        shape_lc1 = classify_curve(depth, lc1)
        shape_lc3 = classify_curve(depth, lc3)

        depth_contrib_lc1 = lc1.iloc[0] - lc1.iloc[-1]
        depth_contrib_lc3 = lc3.iloc[0] - lc3.iloc[-1]

        stats.append({
            group_col: group_value,
            "Max LC1 (mm)": max_lc1,
            "Max LC3 (mm)": max_lc3,
            "Design LC1 (mm)": lc1_design,
            "Design LC3 (mm)": lc3_design,
            "Delta Max (LC3-LC1)": max_lc3 - max_lc1,
            "Delta Design (LC3-LC1)": lc3_design - lc1_design,
            "LC1 Shape": shape_lc1,
            "LC3 Shape": shape_lc3,
            "Depth Contribution LC1": depth_contrib_lc1,
            "Depth Contribution LC3": depth_contrib_lc3
        })

    stats_df = pd.DataFrame(stats)
    stats_df[group_col] = pd.Categorical(stats_df[group_col], categories=order_str, ordered=True)
    stats_df = stats_df.sort_values(group_col).reset_index(drop=True)
    stats_df.to_csv(os.path.join(output_dir, "axial_settlement_stats_by_group.csv"), index=False)

    # ------------------------------------------------------
    # PROFILE PLOTS
    # ------------------------------------------------------
    # LC1
    plt.figure(figsize=(8, 8))
    for g in order_str:
        sub = df[df[group_col].astype(str) == str(g)].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("Depth (m)")
        plt.plot(
            sub["LC1"],
            sub["Depth (m)"],
            color=config["group_colors"].get(str(g), "gray"),
            linewidth=2,
            marker="o",
            label=str(g)
        )

    plt.gca().invert_yaxis()
    plt.xlabel("Axial Settlement (mm)")
    plt.ylabel("Depth (m)")
    plt.title(f"Axial Settlement Profile (LC1) by {config['name']}")
    plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "axial_settlement_LC1.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "axial_settlement_LC1.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # LC3
    plt.figure(figsize=(8, 8))
    for g in order_str:
        sub = df[df[group_col].astype(str) == str(g)].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("Depth (m)")
        plt.plot(
            sub["LC3"],
            sub["Depth (m)"],
            color=config["group_colors"].get(str(g), "gray"),
            linewidth=2,
            marker="o",
            label=str(g)
        )

    plt.gca().invert_yaxis()
    plt.xlabel("Axial Settlement (mm)")
    plt.ylabel("Depth (m)")
    plt.title(f"Axial Settlement Profile (LC3) by {config['name']}")
    plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "axial_settlement_LC3.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "axial_settlement_LC3.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # BAR PLOTS
    # ------------------------------------------------------
    save_bar_plot(
        stats_df, group_col, "Delta Design (LC3-LC1)",
        f"Increase in Axial Settlement from LC1 to LC3 at Design Level by {config['name']}",
        "Increase in Settlement (mm)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "axial_settlement_delta_design.png"),
        os.path.join(output_dir, "axial_settlement_delta_design.pdf")
    )

    save_bar_plot(
        stats_df, group_col, "Delta Max (LC3-LC1)",
        f"Increase in Maximum Axial Settlement from LC1 to LC3 by {config['name']}",
        "Increase in Max Settlement (mm)",
        config["group_colors"], order_str,
        os.path.join(output_dir, "axial_settlement_delta_max.png"),
        os.path.join(output_dir, "axial_settlement_delta_max.pdf")
    )

    # LC1 vs LC3 design-level comparison
    plot_df = stats_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex(order_str).reset_index()
    plot_df = plot_df.dropna(subset=["Design LC1 (mm)", "Design LC3 (mm)"])

    x = np.arange(len(plot_df))
    width = 0.38
    colors = [config["group_colors"].get(str(g), "gray") for g in plot_df[group_col]]

    plt.figure(figsize=(11, 6))
    plt.bar(
        x - width/2,
        plot_df["Design LC1 (mm)"],
        width,
        color=colors,
        label="LC1"
    )
    plt.bar(
        x + width/2,
        plot_df["Design LC3 (mm)"],
        width,
        color="lightgray",
        label="LC3"
    )

    plt.xticks(x, plot_df[group_col], rotation=45, ha="right")
    plt.ylabel("Design Settlement (mm)")
    plt.xlabel(group_col)
    plt.title(f"Design-Level Axial Settlement: LC1 vs LC3 by {config['name']}")
    plt.legend()
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "axial_settlement_design_LC1_vs_LC3.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, "axial_settlement_design_LC1_vs_LC3.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

    # ------------------------------------------------------
    # EXTRA FILTERED SEDIMENT VERSION
    # ------------------------------------------------------
    if config["kind"] == "sediment":
        exclude_list = ["sandy_silt", "slc_sand_slc"]
        df_filtered = filter_out_sediment_outliers(df, group_col, exclude_list)

        if df_filtered.empty:
            print("Filtered dataset is empty — skipping filtered plots.")
        else:
            filtered_order = [g for g in order_str if g not in exclude_list]
            filtered_colors = {
                k: v for k, v in config["group_colors"].items()
                if k in filtered_order
            }

            filtered_output_dir = make_output_dir(
                os.path.join(config["output_dir"], "no_outliers")
            )

            print(f"Creating filtered plots in: {filtered_output_dir}")

            # LC1 filtered
            plt.figure(figsize=(8, 8))
            for g in filtered_order:
                sub = df_filtered[df_filtered[group_col].astype(str) == str(g)]
                if sub.empty:
                    continue

                sub = sub.sort_values("Depth (m)")
                plt.plot(
                    sub["LC1"],
                    sub["Depth (m)"],
                    color=filtered_colors.get(str(g), "gray"),
                    linewidth=2,
                    marker="o",
                    label=str(g)
                )

            plt.gca().invert_yaxis()
            plt.xlabel("Axial Settlement (mm)")
            plt.ylabel("Depth (m)")
            plt.title("Axial Settlement Profile (LC1) — Filtered")
            plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(filtered_output_dir, "axial_settlement_LC1_filtered.png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(filtered_output_dir, "axial_settlement_LC1_filtered.pdf"), bbox_inches="tight")
            plt.show()
            plt.close()

            # LC3 filtered
            plt.figure(figsize=(8, 8))
            for g in filtered_order:
                sub = df_filtered[df_filtered[group_col].astype(str) == str(g)]
                if sub.empty:
                    continue

                sub = sub.sort_values("Depth (m)")
                plt.plot(
                    sub["LC3"],
                    sub["Depth (m)"],
                    color=filtered_colors.get(str(g), "gray"),
                    linewidth=2,
                    marker="o",
                    label=str(g)
                )

            plt.gca().invert_yaxis()
            plt.xlabel("Axial Settlement (mm)")
            plt.ylabel("Depth (m)")
            plt.title("Axial Settlement Profile (LC3) — Filtered")
            plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(filtered_output_dir, "axial_settlement_LC3_filtered.png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(filtered_output_dir, "axial_settlement_LC3_filtered.pdf"), bbox_inches="tight")
            plt.show()
            plt.close()

            # delta design filtered
            stats_filtered = stats_df[~stats_df[group_col].astype(str).isin(exclude_list)].copy()

            save_bar_plot(
                stats_filtered,
                group_col,
                "Delta Design (LC3-LC1)",
                "Increase in Axial Settlement (Filtered)",
                "Increase in Settlement (mm)",
                filtered_colors,
                filtered_order,
                os.path.join(filtered_output_dir, "delta_design_filtered.png"),
                os.path.join(filtered_output_dir, "delta_design_filtered.pdf")
            )

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 3. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "kind": "sediment",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/Axial_Settlement.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/axial_settlement",
    "group_col_name": "Sediment Type",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    }
}

water_config = {
    "name": "Water Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/axial_settlement.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/axial_settlement",
    "group_col_name": "Water Depth",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    }
}

embedment_config = {
    "name": "Embedment Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/axial_settlement.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/axial_settlement",
    "group_col_name": "Embedment Depth",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    }
}

# ==========================================================
# 4. RUN ALL THREE
# ==========================================================
run_axial_settlement_analysis(sediment_config)
run_axial_settlement_analysis(water_config)
run_axial_settlement_analysis(embedment_config)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. SHARED HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(val))
    if match:
        return float(match.group())
    return np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        return val.strip()
    if isinstance(val, (int, float, np.integer, np.floating)):
        if float(val).is_integer():
            return str(int(val))
        return str(float(val))
    return str(val).strip()

def save_bar_plot(summary_df, group_col, value_col, title, ylabel, colors_map, order, out_png, out_pdf, invert_y=False):
    plot_df = summary_df.copy()
    plot_df[group_col] = plot_df[group_col].astype(str)
    plot_df = plot_df.set_index(group_col).reindex([str(x) for x in order]).reset_index()
    plot_df = plot_df.dropna(subset=[value_col])

    colors = [colors_map.get(str(x), "gray") for x in plot_df[group_col]]

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df[group_col], plot_df[value_col], color=colors)
    if invert_y:
        plt.gca().invert_yaxis()
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.xlabel(group_col)
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

def filter_out_sediment_outliers(df, group_col, exclude_list):
    return df[~df[group_col].astype(str).isin(exclude_list)].copy()

# ==========================================================
# 2. MASTER FUNCTION
# ==========================================================
def run_axial_force_analysis(config):
    print(f"\nRunning axial force analysis for: {config['name']}")

    group_col = config["group_col_name"]
    order_str = [normalize_group_label(x) for x in config["group_order"]]
    output_dir = make_output_dir(config["output_dir"])

    df = pd.read_csv(config["input_csv"])
    df = clean_columns(df)

    if df.shape[1] < 6:
        raise ValueError("Axial force file does not have enough columns.")

    # rename first columns by position
    rename_map = {
        df.columns[0]: group_col,
        df.columns[1]: "Node",
        df.columns[2]: "Depth (m)"
    }

    remaining_cols = list(df.columns[3:])
    lc_names = [f"LC{i+1}" for i in range(len(remaining_cols))]
    for old, new in zip(remaining_cols, lc_names):
        rename_map[old] = new

    df = df.rename(columns=rename_map)

    # Fill down grouping labels in case only the first row of each block is labeled
    df[group_col] = df[group_col].ffill()
    
    # clean group labels
    if config["kind"] == "sediment":
        df[group_col] = df[group_col].astype(str).str.strip()
    else:
        df[f"{group_col}_num"] = df[group_col].apply(extract_numeric_value)
        df[group_col] = df[f"{group_col}_num"].apply(normalize_group_label)


    # numeric conversion
    numeric_cols = ["Node", "Depth (m)"]
    lc_cols = [c for c in df.columns if c.startswith("LC")]
    numeric_cols.extend(lc_cols)

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[group_col, "Depth (m)"]).copy()
    df = df[df[group_col].isin(order_str)].copy()
    
    print(f"\nGroups found for {config['name']}:")
    print(sorted(df[group_col].dropna().astype(str).unique()))

    if df.empty:
        raise ValueError(f"No valid rows remained for {config['name']}.")

    df = df.sort_values([group_col, "Depth (m)", "Node"]).reset_index(drop=True)
    df.to_csv(os.path.join(output_dir, "axial_force_cleaned.csv"), index=False)

    # ------------------------------------------------------
    # LONG FORMAT FOR ALL LOAD CASES
    # ------------------------------------------------------
    long_df = df.melt(
        id_vars=[group_col, "Node", "Depth (m)"],
        value_vars=lc_cols,
        var_name="Load Case",
        value_name="Axial Force (kN)"
    )

    long_df["Axial Force (kN)"] = pd.to_numeric(long_df["Axial Force (kN)"], errors="coerce")
    long_df = long_df.dropna(subset=["Axial Force (kN)"]).copy()
    long_df = long_df.sort_values([group_col, "Load Case", "Depth (m)", "Node"]).reset_index(drop=True)

    long_df.to_csv(os.path.join(output_dir, "axial_force_long_cleaned.csv"), index=False)

    # ------------------------------------------------------
    # STATS BY GROUP AND LOAD CASE
    # ------------------------------------------------------
    stats_rows = []

    for (group_value, lc), group in long_df.groupby([group_col, "Load Case"]):
        group = group.sort_values("Depth (m)").copy()

        vals = group["Axial Force (kN)"].to_numpy(dtype=float)
        depth = group["Depth (m)"].to_numpy(dtype=float)

        mask = np.isfinite(vals) & np.isfinite(depth)
        vals = vals[mask]
        depth = depth[mask]

        if len(vals) < 3:
            continue

        idx_max = np.argmax(vals)
        max_force = vals[idx_max]
        depth_max = depth[idx_max]

        top_idx = np.argmin(depth)
        design_force = vals[top_idx]

        gradient = np.gradient(vals, depth)
        mean_gradient = np.nanmean(gradient)
        max_gradient = np.nanmax(np.abs(gradient))
        force_range = np.nanmax(vals) - np.nanmin(vals)

        stats_rows.append({
            group_col: group_value,
            "Load Case": lc,
            "Max Force (kN)": max_force,
            "Depth of Max (m)": depth_max,
            "Design / Top Force (kN)": design_force,
            "Mean Gradient": mean_gradient,
            "Max Gradient": max_gradient,
            "Force Range (kN)": force_range
        })

    stats_df = pd.DataFrame(stats_rows)

    if stats_df.empty:
        raise ValueError(f"No axial-force stats were computed for {config['name']}.")

    stats_df[group_col] = pd.Categorical(stats_df[group_col], categories=order_str, ordered=True)
    stats_df = stats_df.sort_values([group_col, "Load Case"]).reset_index(drop=True)
    stats_df.to_csv(os.path.join(output_dir, "axial_force_stats_by_group_and_loadcase.csv"), index=False)

    # ------------------------------------------------------
    # SUMMARY BY GROUP
    # ------------------------------------------------------
    summary_df = (
        stats_df.groupby(group_col, as_index=False)
        .agg(
            mean_max_force_kN=("Max Force (kN)", "mean"),
            mean_depth_of_max_m=("Depth of Max (m)", "mean"),
            mean_top_force_kN=("Design / Top Force (kN)", "mean"),
            mean_gradient=("Mean Gradient", "mean"),
            max_gradient=("Max Gradient", "mean"),
            mean_force_range_kN=("Force Range (kN)", "mean")
        )
    )

    summary_df.to_csv(os.path.join(output_dir, "axial_force_summary_by_group.csv"), index=False)

    # ------------------------------------------------------
    # LC1 / LC3 COMPARISON TABLE
    # ------------------------------------------------------
    if "LC1" in stats_df["Load Case"].values and "LC3" in stats_df["Load Case"].values:
        compare_df = (
            stats_df[stats_df["Load Case"].isin(["LC1", "LC3"])]
            .pivot(index=group_col, columns="Load Case", values=["Max Force (kN)", "Design / Top Force (kN)", "Depth of Max (m)"])
        )

        compare_df.columns = [f"{metric}_{lc}" for metric, lc in compare_df.columns]
        compare_df = compare_df.reset_index()

        compare_df["Delta Max Force (LC3-LC1)"] = compare_df["Max Force (kN)_LC3"] - compare_df["Max Force (kN)_LC1"]
        compare_df["Delta Design Force (LC3-LC1)"] = compare_df["Design / Top Force (kN)_LC3"] - compare_df["Design / Top Force (kN)_LC1"]
        compare_df["Delta Depth of Max (LC3-LC1)"] = compare_df["Depth of Max (m)_LC3"] - compare_df["Depth of Max (m)_LC1"]

        compare_df.to_csv(os.path.join(output_dir, "axial_force_LC1_vs_LC3_comparison.csv"), index=False)
    else:
        compare_df = pd.DataFrame()

    # ------------------------------------------------------
    # PROFILE PLOTS: EACH LOAD CASE
    # ------------------------------------------------------
    for lc in lc_cols:
        plt.figure(figsize=(8, 10))

        for g in order_str:
            group = df[df[group_col].astype(str) == str(g)].sort_values(["Depth (m)", "Node"])
            if group.empty or lc not in group.columns:
                continue

            group = group.dropna(subset=[lc])
            if group.empty:
                continue

            plt.plot(
                group[lc],
                group["Depth (m)"],
                marker="o",
                markersize=3,
                linewidth=1.5,
                label=str(g),
                color=config["group_colors"].get(str(g), "gray")
            )

        plt.gca().invert_yaxis()
        plt.xlabel("Axial Force (kN)")
        plt.ylabel("Depth (m)")
        plt.title(f"{lc} Axial Force vs Depth by {config['name']}")
        plt.legend(title=config["name"], bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{lc}_axial_force_profile.png"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(output_dir, f"{lc}_axial_force_profile.pdf"), bbox_inches="tight")
        plt.show()
        plt.close()

    # ------------------------------------------------------
    # BAR PLOTS
    # ------------------------------------------------------
    # LC1 depth of max
    lc1_stats = stats_df[stats_df["Load Case"] == "LC1"].copy()
    if not lc1_stats.empty:
        save_bar_plot(
            lc1_stats,
            group_col,
            "Depth of Max (m)",
            f"Depth of Maximum Axial Force (LC1) by {config['name']}",
            "Depth of Max Axial Force (m)",
            config["group_colors"],
            order_str,
            os.path.join(output_dir, "depth_of_max_force_LC1.png"),
            os.path.join(output_dir, "depth_of_max_force_LC1.pdf"),
            invert_y=True
        )

    # LC3 depth of max
    lc3_stats = stats_df[stats_df["Load Case"] == "LC3"].copy()
    if not lc3_stats.empty:
        save_bar_plot(
            lc3_stats,
            group_col,
            "Depth of Max (m)",
            f"Depth of Maximum Axial Force (LC3) by {config['name']}",
            "Depth of Max Axial Force (m)",
            config["group_colors"],
            order_str,
            os.path.join(output_dir, "depth_of_max_force_LC3.png"),
            os.path.join(output_dir, "depth_of_max_force_LC3.pdf"),
            invert_y=True
        )

    # delta max force
    if not compare_df.empty:
        save_bar_plot(
            compare_df,
            group_col,
            "Delta Max Force (LC3-LC1)",
            f"Increase in Maximum Axial Force from LC1 to LC3 by {config['name']}",
            "Difference in Max Axial Force (kN)",
            config["group_colors"],
            order_str,
            os.path.join(output_dir, "delta_max_force_LC3_minus_LC1.png"),
            os.path.join(output_dir, "delta_max_force_LC3_minus_LC1.pdf")
        )

        save_bar_plot(
            compare_df,
            group_col,
            "Delta Design Force (LC3-LC1)",
            f"Increase in Top Axial Force from LC1 to LC3 by {config['name']}",
            "Difference in Top Axial Force (kN)",
            config["group_colors"],
            order_str,
            os.path.join(output_dir, "delta_top_force_LC3_minus_LC1.png"),
            os.path.join(output_dir, "delta_top_force_LC3_minus_LC1.pdf")
        )

    # ------------------------------------------------------
    # EXTRA FILTERED SEDIMENT VERSION
    # ------------------------------------------------------
    if config["kind"] == "sediment":
        exclude_list = ["sandy_silt", "slc_sand_slc"]
        df_filtered = filter_out_sediment_outliers(df, group_col, exclude_list)

        if not df_filtered.empty:
            filtered_order = [g for g in order_str if g not in exclude_list]
            filtered_colors = {k: v for k, v in config["group_colors"].items() if k in filtered_order}
            filtered_output_dir = make_output_dir(os.path.join(config["output_dir"], "no_outliers"))

            for lc in lc_cols:
                plt.figure(figsize=(8, 10))

                for g in filtered_order:
                    group = df_filtered[df_filtered[group_col].astype(str) == str(g)].sort_values(["Depth (m)", "Node"])
                    if group.empty or lc not in group.columns:
                        continue

                    group = group.dropna(subset=[lc])
                    if group.empty:
                        continue

                    plt.plot(
                        group[lc],
                        group["Depth (m)"],
                        marker="o",
                        markersize=3,
                        linewidth=1.5,
                        label=str(g),
                        color=filtered_colors.get(str(g), "gray")
                    )

                plt.gca().invert_yaxis()
                plt.xlabel("Axial Force (kN)")
                plt.ylabel("Depth (m)")
                plt.title(f"{lc} Axial Force vs Depth — Filtered")
                plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(os.path.join(filtered_output_dir, f"{lc}_axial_force_profile_filtered.png"), dpi=300, bbox_inches="tight")
                plt.savefig(os.path.join(filtered_output_dir, f"{lc}_axial_force_profile_filtered.pdf"), bbox_inches="tight")
                plt.show()
                plt.close()

    print(f"Finished: {config['name']}")
    print(f"Outputs saved to: {output_dir}")

# ==========================================================
# 3. CONFIGS
# ==========================================================
sediment_config = {
    "name": "Sediment Type",
    "kind": "sediment",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/Axial_Force.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/axial_force",
    "group_col_name": "Sediment Type",
    "group_order": [
        "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
        "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
        "sand_silt_sand", "f_silty_sand", "sandy_silt"
    ],
    "group_colors": {
        "f_sand": "#1f77b4",
        "vc_sand": "#ff7f0e",
        "sand_slc_sand": "#2ca02c",
        "sand_psc_sand": "#d62728",
        "slc_sand_slc": "#9467bd",
        "sand_dgs_grvl": "#8c564b",
        "sand_ngs_grvl": "#e377c2",
        "sand_silt_sand": "#7f7f7f",
        "f_silty_sand": "#bcbd22",
        "sandy_silt": "#17becf"
    }
}

water_config = {
    "name": "Water Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/axial_force.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/axial_force",
    "group_col_name": "Water Depth",
    "group_order": [10, 30, 50, 70, 90],
    "group_colors": {
        "10": "#ADD8E6",
        "30": "#87CEFA",
        "50": "#6495ED",
        "70": "#6A5ACD",
        "90": "#9370DB"
    }
}

embedment_config = {
    "name": "Embedment Depth",
    "kind": "numeric",
    "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/axial_force_plots/axial_force_long_cleaned.csv",
    "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/axial_force",
    "group_col_name": "Embedment Depth",
    "group_order": [15, 25, 30, 35, 40, 45, 55],
    "group_colors": {
        "15": "#FFD700",
        "25": "#FFA500",
        "30": "#FF8C00",
        "35": "#FF6347",
        "40": "#FF4500",
        "45": "#DC143C",
        "55": "#8B0000"
    }
}

# ==========================================================
# 4. RUN ALL THREE
# ==========================================================
run_axial_force_analysis(sediment_config)
run_axial_force_analysis(water_config)
run_axial_force_analysis(embedment_config)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def save_show(basepath):
    plt.savefig(basepath + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(basepath + ".pdf", bbox_inches="tight")
    plt.show()
    plt.close()

def normalize_lc_name(col):
    s = str(col).strip()
    m = re.search(r"LC[- ]?(\d+)", s, re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    return s

# =========================================================
# COLOR MAPS
# =========================================================
sediment_order = [
    "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
    "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
    "sand_silt_sand", "f_silty_sand", "sandy_silt"
]
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
sediment_colors = {
    sed: default_colors[i % len(default_colors)]
    for i, sed in enumerate(sediment_order)
}

water_order = [10, 30, 50, 70, 90]
water_cmap = LinearSegmentedColormap.from_list(
    "waterdepth_gradient", ["#ADD8E6", "#9370DB"]
)
water_colors = {
    d: to_hex(water_cmap(i / (len(water_order) - 1)))
    for i, d in enumerate(water_order)
}

embedment_order = [15, 25, 30, 35, 40, 45, 55]
embedment_cmap = LinearSegmentedColormap.from_list(
    "embedment_gradient", ["#FFD700", "#FF8C00", "#CC0000"]
)
embedment_colors = {
    d: to_hex(embedment_cmap(i / (len(embedment_order) - 1)))
    for i, d in enumerate(embedment_order)
}

# =========================================================
# CORE PREP FUNCTIONS
# =========================================================
def prep_sediment_axial_stress(file_path):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    # Rename LC columns
    rename_map = {}
    for col in df.columns:
        if col == "Node":
            rename_map[col] = "Node"
        elif col == "Depth (m)":
            rename_map[col] = "Depth (m)"
        elif str(col).startswith("LC-1:"):
            rename_map[col] = "LC1"
        elif str(col).startswith("LC-2:"):
            rename_map[col] = "LC2"
        elif str(col).startswith("LC-3:"):
            rename_map[col] = "LC3"
        elif str(col).startswith("LC-4:"):
            rename_map[col] = "LC4"
        elif str(col).startswith("LC-5:"):
            rename_map[col] = "LC5"
        elif str(col).startswith("LC-6:"):
            rename_map[col] = "LC6"

    df = df.rename(columns=rename_map)

    for col in ["Node", "Depth (m)", "LC1", "LC2", "LC3", "LC4", "LC5", "LC6"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Sediment Type" not in df.columns:
        raise ValueError("Sediment Type column not found.")

    lc_cols = [c for c in ["LC1", "LC2", "LC3", "LC4", "LC5", "LC6"] if c in df.columns]

    long_df = df.melt(
        id_vars=["Sediment Type", "Node", "Depth (m)"],
        value_vars=lc_cols,
        var_name="Load Case",
        value_name="Axial Stress"
    ).dropna(subset=["Axial Stress"]).copy()

    return df, long_df

def prep_water_axial_stress(file_path):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    if df.shape[1] < 8:
        raise ValueError("Water depth axial_stress.csv does not have the expected columns.")

    df = df.rename(columns={
        df.columns[0]: "Water Depth",
        df.columns[1]: "Node",
        df.columns[2]: "Depth (m)",
        df.columns[3]: "LC1",
        df.columns[4]: "LC2",
        df.columns[5]: "LC3",
        df.columns[6]: "LC4",
        df.columns[7]: "LC5"
    })

    keep_cols = ["Water Depth", "Node", "Depth (m)", "LC1", "LC2", "LC3", "LC4", "LC5"]
    df = df[keep_cols].copy()

    df["Water Depth Numeric"] = df["Water Depth"].apply(extract_number)

    for col in ["Node", "Depth (m)", "LC1", "LC2", "LC3", "LC4", "LC5"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Water Depth Numeric", "Depth (m)"]).copy()

    lc_cols = ["LC1", "LC2", "LC3", "LC4", "LC5"]
    long_df = df.melt(
        id_vars=["Water Depth", "Water Depth Numeric", "Node", "Depth (m)"],
        value_vars=lc_cols,
        var_name="Load Case",
        value_name="Axial Stress"
    ).dropna(subset=["Axial Stress"]).copy()

    return df, long_df

def prep_embedment_axial_stress(file_path):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    if df.shape[1] < 4:
        raise ValueError("Embedment depth axial_stress.csv does not have the expected columns.")

    df = df.rename(columns={
        df.columns[0]: "Embedment Depth",
        df.columns[1]: "Node",
        df.columns[2]: "Depth (m)"
    })

    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False, na=False)].copy()

    df["Embedment Depth Numeric"] = df["Embedment Depth"].apply(extract_number)
    df["Node"] = pd.to_numeric(df["Node"], errors="coerce")
    df["Depth (m)"] = pd.to_numeric(df["Depth (m)"], errors="coerce")

    df = df.dropna(subset=["Embedment Depth Numeric", "Node", "Depth (m)"]).copy()

    id_cols = ["Embedment Depth", "Embedment Depth Numeric", "Node", "Depth (m)"]
    lc_cols = [c for c in df.columns if c not in id_cols]

    if not lc_cols:
        raise ValueError("No load case columns found for embedment file.")

    long_df = df.melt(
        id_vars=id_cols,
        value_vars=lc_cols,
        var_name="LoadCase_raw",
        value_name="Axial Stress"
    ).dropna(subset=["Axial Stress"]).copy()

    long_df["Load Case"] = long_df["LoadCase_raw"].apply(normalize_lc_name)

    return df, long_df

# =========================================================
# STATS
# =========================================================
def compute_axial_stress_stats(long_df, group_col):
    rows = []

    for (group, lc), sub in long_df.groupby([group_col, "Load Case"]):
        sub = sub.sort_values("Depth (m)").copy()

        stress = pd.to_numeric(sub["Axial Stress"], errors="coerce").to_numpy(dtype=float)
        depth = pd.to_numeric(sub["Depth (m)"], errors="coerce").to_numpy(dtype=float)

        mask = np.isfinite(stress) & np.isfinite(depth)
        stress = stress[mask]
        depth = depth[mask]

        if len(stress) < 3:
            continue

        idx_max = np.argmax(stress)
        max_stress = stress[idx_max]
        depth_of_max = depth[idx_max]

        top_idx = np.argmin(depth)
        top_stress = stress[top_idx]

        gradient = np.gradient(stress, depth)
        mean_gradient = np.nanmean(gradient)
        max_abs_gradient = np.nanmax(np.abs(gradient))
        stress_range = np.nanmax(stress) - np.nanmin(stress)

        rows.append({
            group_col: group,
            "Load Case": lc,
            "Max Stress": max_stress,
            "Depth of Max": depth_of_max,
            "Top / Design Stress": top_stress,
            "Mean Gradient": mean_gradient,
            "Max Abs Gradient": max_abs_gradient,
            "Stress Range": stress_range
        })

    return pd.DataFrame(rows)

# =========================================================
# PLOTTING
# =========================================================
def plot_profiles(long_df, group_col, group_order, color_map, output_dir, prefix):
    for lc in sorted(long_df["Load Case"].dropna().unique()):
        sub_lc = long_df[long_df["Load Case"] == lc].copy()
        if sub_lc.empty:
            continue

        plt.figure(figsize=(8, 10))

        for grp in group_order:
            sub = sub_lc[sub_lc[group_col] == grp].sort_values("Depth (m)")
            if sub.empty:
                continue

            label = f"{int(grp)} m" if isinstance(grp, (int, float, np.integer, np.floating)) else str(grp)

            plt.plot(
                sub["Axial Stress"],
                sub["Depth (m)"],
                marker="o",
                linewidth=2,
                markersize=4,
                label=label,
                color=color_map.get(grp, None)
            )

        plt.gca().invert_yaxis()
        plt.xlabel("Axial Stress")
        plt.ylabel("Depth (m)")
        plt.title(f"Axial Stress vs Depth ({lc})")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_profiles_{lc}"))

def plot_stat_bars(stats_df, group_col, group_order, color_map, stat_col, output_dir, prefix, ylabel, title_base):
    for lc in sorted(stats_df["Load Case"].dropna().unique()):
        sub = stats_df[stats_df["Load Case"] == lc].copy()
        if sub.empty:
            continue

        ordered = [g for g in group_order if g in sub[group_col].values]
        sub = sub.set_index(group_col).loc[ordered].reset_index()

        labels = [
            f"{int(v)} m" if isinstance(v, (int, float, np.integer, np.floating)) else str(v)
            for v in sub[group_col]
        ]
        colors = [color_map.get(v, None) for v in sub[group_col]]

        plt.figure(figsize=(10, 6))
        plt.bar(labels, sub[stat_col], color=colors)
        plt.ylabel(ylabel)
        plt.xlabel(group_col)
        plt.title(f"{title_base} ({lc})")
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_{stat_col.lower().replace('/', '').replace(' ', '_')}_{lc}"))

# =========================================================
# RUNNERS
# =========================================================
def run_sediment_axial_stress(file_path, output_dir):
    output_dir = make_output_dir(output_dir)
    wide_df, long_df = prep_sediment_axial_stress(file_path)

    wide_df.to_csv(os.path.join(output_dir, "axial_stress_processed.csv"), index=False)
    long_df.to_csv(os.path.join(output_dir, "axial_stress_long.csv"), index=False)

    stats_df = compute_axial_stress_stats(long_df, "Sediment Type")
    stats_df.to_csv(os.path.join(output_dir, "axial_stress_stats.csv"), index=False)

    plot_profiles(long_df, "Sediment Type", sediment_order, sediment_colors, output_dir, "axial_stress")
    plot_stat_bars(stats_df, "Sediment Type", sediment_order, sediment_colors,
                   "Depth of Max", output_dir, "axial_stress",
                   "Depth of Max Axial Stress (m)", "Depth of Maximum Axial Stress")
    plot_stat_bars(stats_df, "Sediment Type", sediment_order, sediment_colors,
                   "Max Abs Gradient", output_dir, "axial_stress",
                   "Maximum Absolute Stress Gradient", "Maximum Axial Stress Gradient")

    return wide_df, long_df, stats_df

def run_water_axial_stress(file_path, output_dir):
    output_dir = make_output_dir(output_dir)
    wide_df, long_df = prep_water_axial_stress(file_path)

    wide_df.to_csv(os.path.join(output_dir, "axial_stress_cleaned_by_water_depth.csv"), index=False)
    long_df.to_csv(os.path.join(output_dir, "axial_stress_long_by_water_depth.csv"), index=False)

    stats_df = compute_axial_stress_stats(long_df, "Water Depth Numeric")
    stats_df = stats_df.sort_values(["Water Depth Numeric", "Load Case"]).reset_index(drop=True)
    stats_df.to_csv(os.path.join(output_dir, "axial_stress_stats_by_water_depth.csv"), index=False)

    plot_profiles(long_df, "Water Depth Numeric", water_order, water_colors, output_dir, "axial_stress_by_water_depth")
    plot_stat_bars(stats_df, "Water Depth Numeric", water_order, water_colors,
                   "Depth of Max", output_dir, "axial_stress_by_water_depth",
                   "Depth of Max Axial Stress (m)", "Depth of Maximum Axial Stress")
    plot_stat_bars(stats_df, "Water Depth Numeric", water_order, water_colors,
                   "Max Abs Gradient", output_dir, "axial_stress_by_water_depth",
                   "Maximum Absolute Stress Gradient", "Maximum Axial Stress Gradient")

    return wide_df, long_df, stats_df

def run_embedment_axial_stress(file_path, output_dir):
    output_dir = make_output_dir(output_dir)
    wide_df, long_df = prep_embedment_axial_stress(file_path)

    wide_df.to_csv(os.path.join(output_dir, "axial_stress_cleaned_by_embedment_depth.csv"), index=False)
    long_df.to_csv(os.path.join(output_dir, "axial_stress_long_cleaned.csv"), index=False)

    stats_df = compute_axial_stress_stats(long_df, "Embedment Depth Numeric")
    stats_df = stats_df.sort_values(["Embedment Depth Numeric", "Load Case"]).reset_index(drop=True)
    stats_df.to_csv(os.path.join(output_dir, "axial_stress_stats_by_embedment_depth.csv"), index=False)

    plot_profiles(long_df, "Embedment Depth Numeric", embedment_order, embedment_colors, output_dir, "axial_stress_by_embedment_depth")
    plot_stat_bars(stats_df, "Embedment Depth Numeric", embedment_order, embedment_colors,
                   "Depth of Max", output_dir, "axial_stress_by_embedment_depth",
                   "Depth of Max Axial Stress (m)", "Depth of Maximum Axial Stress")
    plot_stat_bars(stats_df, "Embedment Depth Numeric", embedment_order, embedment_colors,
                   "Max Abs Gradient", output_dir, "axial_stress_by_embedment_depth",
                   "Maximum Absolute Stress Gradient", "Maximum Axial Stress Gradient")

    return wide_df, long_df, stats_df


# =========================================================
# PATHS
# =========================================================
sediment_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/Axial_Stress.csv"
sediment_out  = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/axial_stress_plots"

water_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/axial_stress.csv"
water_out  = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/axial_stress_plots"

embedment_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/axial_stress.csv"
embedment_out  = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/axial_stress_plots"
# =========================================================
# RUN
# =========================================================
sediment_wide, sediment_long, sediment_stats = run_sediment_axial_stress(sediment_file, sediment_out)
water_wide, water_long, water_stats = run_water_axial_stress(water_file, water_out)
embedment_wide, embedment_long, embedment_stats = run_embedment_axial_stress(embedment_file, embedment_out)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.colors import LinearSegmentedColormap, to_hex

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def normalize_load_case(val):
    s = str(val).strip()
    m = re.search(r"(\d+)", s)
    return f"LC{m.group(1)}" if m else s

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def save_show(basepath):
    plt.savefig(basepath + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(basepath + ".pdf", bbox_inches="tight")
    plt.show()
    plt.close()

def get_inflection_depths(z, y, window=5, polyorder=2):
    z = np.asarray(z, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(z) & np.isfinite(y)
    z = z[mask]
    y = y[mask]

    if len(z) < max(window, 5):
        return np.array([], dtype=float)

    w = min(window, len(y) if len(y) % 2 == 1 else len(y) - 1)
    if w < 5 or w <= polyorder:
        return np.array([], dtype=float)

    y_smooth = savgol_filter(y, window_length=w, polyorder=polyorder)
    dy_dz = np.gradient(y_smooth, z)
    d2y_dz2 = np.gradient(dy_dz, z)

    eps = max(np.nanmax(np.abs(d2y_dz2)) * 1e-3, 1e-9)
    d2_clean = d2y_dz2.copy()
    d2_clean[np.abs(d2_clean) < eps] = 0.0

    sign_change_idx = np.where(np.diff(np.sign(d2_clean)) != 0)[0]
    if len(sign_change_idx) == 0:
        return np.array([], dtype=float)

    return z[sign_change_idx]

# =========================================================
# COLOR MAPS
# =========================================================
sediment_order = [
    "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
    "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
    "sand_silt_sand", "f_silty_sand", "sandy_silt"
]
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
sediment_colors = {
    sed: default_colors[i % len(default_colors)]
    for i, sed in enumerate(sediment_order)
}

water_order = [10, 30, 50, 70, 90]
water_cmap = LinearSegmentedColormap.from_list(
    "waterdepth_gradient", ["#ADD8E6", "#9370DB"]
)
water_colors = {
    d: to_hex(water_cmap(i / (len(water_order) - 1)))
    for i, d in enumerate(water_order)
}

embedment_order = [15, 25, 30, 35, 40, 45, 55]
embedment_cmap = LinearSegmentedColormap.from_list(
    "embedment_gradient", ["#FFD700", "#FF8C00", "#CC0000"]
)
embedment_colors = {
    d: to_hex(embedment_cmap(i / (len(embedment_order) - 1)))
    for i, d in enumerate(embedment_order)
}

# =========================================================
# CORE PREP
# =========================================================
def prep_sediment_deflection(file_path):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    combo_col = "LoadCase- SedimentType"
    if combo_col not in df.columns:
        raise ValueError(f"Could not find '{combo_col}' column.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)
    df["Load Case"] = split_cols[0].str.strip().apply(normalize_load_case)
    df["Sediment Type"] = split_cols[1].str.strip()

    numeric_cols = [
        "No.",
        "Depth (m)",
        "Lateral deflection (mm)",
        "Slope (rad)",
        "Soil pressure (kN/m)"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Depth (m)", "Lateral deflection (mm)", "Sediment Type", "Load Case"]).copy()
    return df

def prep_water_deflection(file_path):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    combo_col = None
    for col in df.columns:
        if "load" in col.lower() and "-" in col:
            combo_col = col
            break
    if combo_col is None:
        raise ValueError("Could not find combined load case / water depth column.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)
    df["Load Case"] = split_cols[0].str.strip().apply(normalize_load_case)
    df["Water Depth"] = split_cols[1].str.strip()
    df["Water Depth Numeric"] = df["Water Depth"].apply(extract_number)

    numeric_cols = [
        "No.",
        "Depth (m)",
        "Lateral deflection (mm)",
        "Slope (rad)",
        "Soil pressure (kN/m)"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Depth (m)", "Lateral deflection (mm)", "Water Depth Numeric", "Load Case"]).copy()
    df = df[df["Water Depth Numeric"].isin(water_order)].copy()
    return df

def prep_embedment_deflection(file_path):
    df = pd.read_csv(file_path)
    df = clean_columns(df)

    combo_col = None
    for col in df.columns:
        low = col.lower()
        if "load" in low and "-" in col:
            combo_col = col
            break
    if combo_col is None:
        raise ValueError("Could not find combined load case / embedment depth column.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)
    df["Load Case"] = split_cols[0].str.strip().apply(normalize_load_case)
    df["Embedment Depth"] = split_cols[1].str.strip()
    df["Embedment Depth Numeric"] = df["Embedment Depth"].apply(extract_number)

    numeric_cols = [
        "No.",
        "Depth (m)",
        "Lateral deflection (mm)",
        "Slope (rad)",
        "Soil pressure (kN/m)"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Depth (m)", "Lateral deflection (mm)", "Embedment Depth Numeric", "Load Case"]).copy()
    df = df[df["Embedment Depth Numeric"].isin(embedment_order)].copy()
    return df

# =========================================================
# STATS
# =========================================================
def compute_deflection_stats(df, group_col):
    stats_rows = []
    inflection_rows = []

    for (group, lc), sub in df.groupby([group_col, "Load Case"]):
        sub = sub.sort_values("Depth (m)").copy()

        z = sub["Depth (m)"].to_numpy(dtype=float)
        y = sub["Lateral deflection (mm)"].to_numpy(dtype=float)

        mask = np.isfinite(z) & np.isfinite(y)
        z = z[mask]
        y = y[mask]

        if len(z) < 5:
            continue

        idx_max = np.argmax(np.abs(y))
        max_defl = y[idx_max]
        depth_max = z[idx_max]

        inflections = get_inflection_depths(z, y, window=5, polyorder=2)
        first_inflection = inflections[0] if len(inflections) > 0 else np.nan

        stats_rows.append({
            group_col: group,
            "Load Case": lc,
            "Max Deflection (mm)": max_defl,
            "Max Deflection Magnitude (mm)": abs(max_defl),
            "Depth of Max (m)": depth_max,
            "Num Inflection Points": len(inflections),
            "First Inflection Depth (m)": first_inflection,
            "Inflection Depths (m)": ", ".join(f"{d:.2f}" for d in inflections)
        })

        for d in inflections:
            inflection_rows.append({
                group_col: group,
                "Load Case": lc,
                "Inflection Depth (m)": d
            })

    stats_df = pd.DataFrame(stats_rows)
    inflection_df = pd.DataFrame(inflection_rows)
    return stats_df, inflection_df

# =========================================================
# PLOTTING
# =========================================================
def plot_profiles(df, group_col, group_order, color_map, output_dir, prefix):
    for lc in sorted(df["Load Case"].dropna().unique()):
        plt.figure(figsize=(8, 10))
        plotted_any = False

        for grp in group_order:
            sub = df[(df[group_col] == grp) & (df["Load Case"] == lc)].copy()
            if sub.empty:
                continue

            plotted_any = True
            sub = sub.sort_values("Depth (m)")

            label = f"{int(grp)} m" if isinstance(grp, (int, float, np.integer, np.floating)) else str(grp)

            plt.plot(
                sub["Lateral deflection (mm)"],
                sub["Depth (m)"],
                marker="o",
                linewidth=2,
                markersize=4,
                label=label,
                color=color_map.get(grp, None)
            )

        if not plotted_any:
            plt.close()
            continue

        plt.gca().invert_yaxis()
        plt.xlabel("Lateral Deflection (mm)")
        plt.ylabel("Depth (m)")
        plt.title(f"Deflection vs Depth ({lc})")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_deflection_{lc}"))

def plot_slope_profiles(df, group_col, group_order, color_map, output_dir, prefix):
    for lc in sorted(df["Load Case"].dropna().unique()):
        plt.figure(figsize=(8, 10))
        plotted_any = False

        for grp in group_order:
            sub = df[(df[group_col] == grp) & (df["Load Case"] == lc)].copy()
            if sub.empty:
                continue

            plotted_any = True
            sub = sub.sort_values("Depth (m)")
            label = f"{int(grp)} m" if isinstance(grp, (int, float, np.integer, np.floating)) else str(grp)

            plt.plot(
                sub["Slope (rad)"],
                sub["Depth (m)"],
                marker="o",
                linewidth=2,
                markersize=4,
                label=label,
                color=color_map.get(grp, None)
            )

        if not plotted_any:
            plt.close()
            continue

        plt.gca().invert_yaxis()
        plt.xlabel("Slope (rad)")
        plt.ylabel("Depth (m)")
        plt.title(f"Slope vs Depth ({lc})")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_slope_{lc}"))

def plot_pressure_profiles(df, group_col, group_order, color_map, output_dir, prefix, min_depth=25):
    for lc in sorted(df["Load Case"].dropna().unique()):
        plt.figure(figsize=(8, 10))
        plotted_any = False

        for grp in group_order:
            sub = df[(df[group_col] == grp) & (df["Load Case"] == lc)].copy()
            sub = sub[sub["Depth (m)"] >= min_depth]
            if sub.empty:
                continue

            plotted_any = True
            sub = sub.sort_values("Depth (m)")
            label = f"{int(grp)} m" if isinstance(grp, (int, float, np.integer, np.floating)) else str(grp)

            plt.plot(
                sub["Soil pressure (kN/m)"],
                sub["Depth (m)"],
                marker="o",
                linewidth=2,
                markersize=4,
                label=label,
                color=color_map.get(grp, None)
            )

        if not plotted_any:
            plt.close()
            continue

        plt.gca().invert_yaxis()
        plt.xlabel("Soil Pressure (kN/m)")
        plt.ylabel("Depth (m)")
        plt.title(f"Soil Pressure vs Depth ({lc})")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_soil_pressure_{lc}"))

# =========================================================
# RUNNERS
# =========================================================
def run_sediment_deflection(file_path, output_dir):
    output_dir = make_output_dir(output_dir)
    df = prep_sediment_deflection(file_path)

    df.to_csv(os.path.join(output_dir, "deflection_processed.csv"), index=False)

    stats_df, inflection_df = compute_deflection_stats(df, "Sediment Type")
    stats_df.to_csv(os.path.join(output_dir, "deflection_stats.csv"), index=False)
    inflection_df.to_csv(os.path.join(output_dir, "deflection_inflection_points_long.csv"), index=False)

    plot_profiles(df, "Sediment Type", sediment_order, sediment_colors, output_dir, "sediment")
    plot_slope_profiles(df, "Sediment Type", sediment_order, sediment_colors, output_dir, "sediment")
    plot_pressure_profiles(df, "Sediment Type", sediment_order, sediment_colors, output_dir, "sediment")

    return df, stats_df, inflection_df

def run_water_deflection(file_path, output_dir):
    output_dir = make_output_dir(output_dir)
    df = prep_water_deflection(file_path)

    df.to_csv(os.path.join(output_dir, "deflection_cleaned_by_water_depth.csv"), index=False)

    stats_df, inflection_df = compute_deflection_stats(df, "Water Depth Numeric")
    stats_df = stats_df.sort_values(["Water Depth Numeric", "Load Case"]).reset_index(drop=True)
    inflection_df = inflection_df.sort_values(["Water Depth Numeric", "Load Case"]).reset_index(drop=True)

    stats_df.to_csv(os.path.join(output_dir, "deflection_stats_by_water_depth.csv"), index=False)
    inflection_df.to_csv(os.path.join(output_dir, "deflection_inflection_points_by_water_depth.csv"), index=False)

    plot_profiles(df, "Water Depth Numeric", water_order, water_colors, output_dir, "water")
    plot_slope_profiles(df, "Water Depth Numeric", water_order, water_colors, output_dir, "water")
    plot_pressure_profiles(df, "Water Depth Numeric", water_order, water_colors, output_dir, "water")

    return df, stats_df, inflection_df

def run_embedment_deflection(file_path, output_dir):
    output_dir = make_output_dir(output_dir)
    df = prep_embedment_deflection(file_path)

    df.to_csv(os.path.join(output_dir, "deflection_cleaned_by_embedment_depth.csv"), index=False)

    stats_df, inflection_df = compute_deflection_stats(df, "Embedment Depth Numeric")
    stats_df = stats_df.sort_values(["Embedment Depth Numeric", "Load Case"]).reset_index(drop=True)
    inflection_df = inflection_df.sort_values(["Embedment Depth Numeric", "Load Case"]).reset_index(drop=True)

    stats_df.to_csv(os.path.join(output_dir, "deflection_stats_by_embedment_depth.csv"), index=False)
    inflection_df.to_csv(os.path.join(output_dir, "deflection_inflection_points_by_embedment_depth.csv"), index=False)

    plot_profiles(df, "Embedment Depth Numeric", embedment_order, embedment_colors, output_dir, "embedment")
    plot_slope_profiles(df, "Embedment Depth Numeric", embedment_order, embedment_colors, output_dir, "embedment")
    plot_pressure_profiles(df, "Embedment Depth Numeric", embedment_order, embedment_colors, output_dir, "embedment")

    return df, stats_df, inflection_df

# =========================================================
# PATHS
# =========================================================
sediment_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/LC_Defelction_Slope_Pressure.csv"
sediment_out  = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/deflection_plots"

water_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/deflection_slope_pressure.csv"
water_out  = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/deflection_plots"

embedment_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/deflection_slope_pressure.csv"
embedment_out  = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/deflection_slope_pressure_plots"

# =========================================================
# RUN
# =========================================================
sediment_defl_df, deflection_stats_df_sediment, deflection_inflection_df_sediment = run_sediment_deflection(sediment_file, sediment_out)
water_defl_df, deflection_stats_df_water, deflection_inflection_df_water = run_water_deflection(water_file, water_out)
embedment_defl_df, deflection_stats_df_embedment, deflection_inflection_df_embedment = run_embedment_deflection(embedment_file, embedment_out)

In [ ]:
sediment_defl_df, deflection_stats_df_sediment, deflection_inflection_df_sediment = run_sediment_deflection(sediment_file, sediment_out)
water_defl_df, deflection_stats_df_water, deflection_inflection_df_water = run_water_deflection(water_file, water_out)
embedment_defl_df, deflection_stats_df_embedment, deflection_inflection_df_embedment = run_embedment_deflection(embedment_file, embedment_out)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.colors import LinearSegmentedColormap, to_hex

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def extract_lc_number(val):
    m = re.search(r"LC\s*-?\s*(\d+)", str(val), flags=re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

def normalize_load_case(val):
    n = extract_lc_number(val)
    return f"LC{int(n)}" if pd.notna(n) else str(val).strip()

def make_output_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def save_show(basepath):
    plt.savefig(basepath + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(basepath + ".pdf", bbox_inches="tight")
    plt.show()
    plt.close()

def get_inflection_depths(z, y, window=5, polyorder=2):
    z = np.asarray(z, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(z) & np.isfinite(y)
    z = z[mask]
    y = y[mask]

    if len(z) < max(window, 5):
        return np.array([], dtype=float)

    w = min(window, len(y) if len(y) % 2 == 1 else len(y) - 1)
    if w < 5 or w <= polyorder:
        return np.array([], dtype=float)

    try:
        y_smooth = savgol_filter(y, window_length=w, polyorder=polyorder)
    except Exception:
        return np.array([], dtype=float)

    dy_dz = np.gradient(y_smooth, z)
    d2y_dz2 = np.gradient(dy_dz, z)

    eps = max(np.nanmax(np.abs(d2y_dz2)) * 1e-3, 1e-9)
    d2_clean = d2y_dz2.copy()
    d2_clean[np.abs(d2_clean) < eps] = 0.0

    sign_change_idx = np.where(np.diff(np.sign(d2_clean)) != 0)[0]
    if len(sign_change_idx) == 0:
        return np.array([], dtype=float)

    return z[sign_change_idx]

# =========================================================
# COLOR MAPS
# =========================================================
sediment_order = [
    "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
    "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
    "sand_silt_sand", "f_silty_sand", "sandy_silt"
]
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
sediment_colors = {
    sed: default_colors[i % len(default_colors)]
    for i, sed in enumerate(sediment_order)
}

water_order = [10, 30, 50, 70, 90]
water_cmap = LinearSegmentedColormap.from_list(
    "waterdepth_gradient", ["#ADD8E6", "#9370DB"]
)
water_colors = {
    d: to_hex(water_cmap(i / (len(water_order) - 1)))
    for i, d in enumerate(water_order)
}

embedment_order = [15, 25, 30, 35, 40, 45, 55]
embedment_cmap = LinearSegmentedColormap.from_list(
    "embedment_gradient", ["#FFD700", "#FF8C00", "#CC0000"]
)
embedment_colors = {
    d: to_hex(embedment_cmap(i / (len(embedment_order) - 1)))
    for i, d in enumerate(embedment_order)
}

# =========================================================
# PREP FUNCTIONS
# =========================================================
def prep_sediment_bm(bm_file):
    df = pd.read_csv(bm_file)
    df = clean_columns(df)

    combo_col = "LoadCase- SedimentType"
    if combo_col not in df.columns:
        raise ValueError(f"Could not find '{combo_col}' in sediment BM file.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)
    df["Load Case"] = split_cols[0].str.strip().apply(normalize_load_case)
    df["Sediment Type"] = split_cols[1].str.strip()

    for col in ["Node", "Depth (m)", "Shear force (kN)", "Bending Moment (kNm)"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Depth (m)", "Bending Moment (kNm)", "Shear force (kN)", "Sediment Type", "Load Case"]).copy()
    return df

def prep_sediment_defl(defl_file):
    df = pd.read_csv(defl_file)
    df = clean_columns(df)

    combo_col = "LoadCase- SedimentType"
    if combo_col not in df.columns:
        raise ValueError(f"Could not find '{combo_col}' in sediment deflection file.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)
    df["Load Case"] = split_cols[0].str.strip().apply(normalize_load_case)
    df["Sediment Type"] = split_cols[1].str.strip()

    for col in ["No.", "Depth (m)", "Lateral deflection (mm)", "Slope (rad)", "Soil pressure (kN/m)"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Depth (m)", "Lateral deflection (mm)", "Sediment Type", "Load Case"]).copy()
    return df

def prep_water_or_embedment_bm(bm_file, second_name):
    df = pd.read_csv(bm_file)
    df = clean_columns(df)

    combo_col = None
    for col in df.columns:
        low = col.lower()
        if ("loadcase" in low or "lc" in low) and "-" in col:
            combo_col = col
            break
    if combo_col is None:
        raise ValueError(f"Could not find combined LoadCase/{second_name} column in BM file.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)
    df["Load Case"] = split_cols[0].str.strip().apply(normalize_load_case)
    df[second_name] = split_cols[1].str.strip()
    df[f"{second_name} Numeric"] = df[second_name].apply(extract_number)

    for col in ["Node", "Depth (m)", "Shear force (kN)", "Bending Moment (kNm)"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Depth (m)", "Bending Moment (kNm)", "Shear force (kN)", f"{second_name} Numeric", "Load Case"]).copy()
    return df

def prep_water_or_embedment_defl(defl_file, second_name):
    df = pd.read_csv(defl_file)
    df = clean_columns(df)

    combo_col = None
    for col in df.columns:
        low = col.lower()
        if ("loadcase" in low or "lc" in low) and "-" in col:
            combo_col = col
            break
    if combo_col is None:
        raise ValueError(f"Could not find combined LoadCase/{second_name} column in deflection file.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)
    df["Load Case"] = split_cols[0].str.strip().apply(normalize_load_case)
    df[second_name] = split_cols[1].str.strip()
    df[f"{second_name} Numeric"] = df[second_name].apply(extract_number)

    for col in ["No.", "Depth (m)", "Lateral deflection (mm)", "Slope (rad)", "Soil pressure (kN/m)"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Depth (m)", "Lateral deflection (mm)", f"{second_name} Numeric", "Load Case"]).copy()
    return df

# =========================================================
# STATS
# =========================================================
def compute_bm_stats(df, group_col):
    rows = []

    for (group, lc), sub in df.groupby([group_col, "Load Case"]):
        sub = sub.sort_values("Depth (m)").copy()

        z = sub["Depth (m)"].to_numpy(dtype=float)
        M = sub["Bending Moment (kNm)"].to_numpy(dtype=float)
        V = sub["Shear force (kN)"].to_numpy(dtype=float)

        mask = np.isfinite(z) & np.isfinite(M) & np.isfinite(V)
        z = z[mask]
        M = M[mask]
        V = V[mask]

        if len(z) < 3:
            continue

        idx_M = np.argmax(np.abs(M))
        idx_V = np.argmax(np.abs(V))

        rows.append({
            group_col: group,
            "Load Case": lc,
            "Mmax (kNm)": M[idx_M],
            "Mmax Magnitude (kNm)": abs(M[idx_M]),
            "Depth Mmax (m)": z[idx_M],
            "Vmax (kN)": V[idx_V],
            "Vmax Magnitude (kN)": abs(V[idx_V]),
            "Depth Vmax (m)": z[idx_V]
        })

    return pd.DataFrame(rows)

def compute_first_inflection_df(defl_df, group_col):
    rows = []

    for (group, lc), sub in defl_df.groupby([group_col, "Load Case"]):
        sub = sub.sort_values("Depth (m)").copy()

        z = sub["Depth (m)"].to_numpy(dtype=float)
        y = sub["Lateral deflection (mm)"].to_numpy(dtype=float)

        inflections = get_inflection_depths(z, y, window=5, polyorder=2)
        first_inflection = inflections[0] if len(inflections) > 0 else np.nan

        rows.append({
            group_col: group,
            "Load Case": lc,
            "First Inflection Depth (m)": first_inflection
        })

    return pd.DataFrame(rows)

# =========================================================
# PLOTTING
# =========================================================
def plot_bm_profiles(df, group_col, group_order, color_map, output_dir, prefix):
    for lc in sorted(df["Load Case"].dropna().unique(), key=lambda x: extract_lc_number(x)):
        plt.figure(figsize=(8, 10))
        plotted = False

        for grp in group_order:
            sub = df[(df[group_col] == grp) & (df["Load Case"] == lc)].copy()
            if sub.empty:
                continue

            plotted = True
            sub = sub.sort_values("Depth (m)")
            z = sub["Depth (m)"].to_numpy(dtype=float)
            M = sub["Bending Moment (kNm)"].to_numpy(dtype=float)

            label = f"{int(grp)} m" if isinstance(grp, (int, float, np.integer, np.floating)) else str(grp)

            plt.plot(
                M, z,
                marker="o",
                linewidth=2,
                markersize=4,
                label=label,
                color=color_map.get(grp, None)
            )

            idx_max = np.argmax(np.abs(M))
            plt.scatter(M[idx_max], z[idx_max], color="red", s=30, zorder=5)

        if not plotted:
            plt.close()
            continue

        plt.gca().invert_yaxis()
        plt.xlabel("Bending Moment (kNm)")
        plt.ylabel("Depth (m)")
        plt.title(f"Bending Moment vs Depth ({lc})")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_bending_moment_{lc}"))

def plot_shear_profiles(df, group_col, group_order, color_map, output_dir, prefix):
    for lc in sorted(df["Load Case"].dropna().unique(), key=lambda x: extract_lc_number(x)):
        plt.figure(figsize=(8, 10))
        plotted = False

        for grp in group_order:
            sub = df[(df[group_col] == grp) & (df["Load Case"] == lc)].copy()
            if sub.empty:
                continue

            plotted = True
            sub = sub.sort_values("Depth (m)")
            z = sub["Depth (m)"].to_numpy(dtype=float)
            V = sub["Shear force (kN)"].to_numpy(dtype=float)

            label = f"{int(grp)} m" if isinstance(grp, (int, float, np.integer, np.floating)) else str(grp)

            plt.plot(
                V, z,
                marker="o",
                linewidth=2,
                markersize=4,
                label=label,
                color=color_map.get(grp, None)
            )

            idx_max = np.argmax(np.abs(V))
            plt.scatter(V[idx_max], z[idx_max], color="red", s=30, zorder=5)

        if not plotted:
            plt.close()
            continue

        plt.gca().invert_yaxis()
        plt.xlabel("Shear Force (kN)")
        plt.ylabel("Depth (m)")
        plt.title(f"Shear Force vs Depth ({lc})")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_shear_force_{lc}"))

def plot_moment_vs_inflection(compare_df, group_col, group_order, color_map, output_dir, prefix):
    for lc in sorted(compare_df["Load Case"].dropna().unique(), key=lambda x: extract_lc_number(x)):
        sub = compare_df[compare_df["Load Case"] == lc].copy()
        if sub.empty:
            continue

        ordered = [g for g in group_order if g in sub[group_col].values]
        sub = sub.set_index(group_col).loc[ordered].reset_index()

        x = np.arange(len(sub))
        labels = [
            f"{int(v)} m" if isinstance(v, (int, float, np.integer, np.floating)) else str(v)
            for v in sub[group_col]
        ]
        colors = [color_map.get(v, None) for v in sub[group_col]]

        plt.figure(figsize=(10, 6))
        plt.bar(
            x - 0.15,
            sub["Depth Mmax (m)"],
            width=0.3,
            color=colors,
            label="Depth of Max Bending Moment"
        )
        plt.scatter(
            x + 0.15,
            sub["First Inflection Depth (m)"],
            color="black",
            marker="D",
            s=50,
            label="First Inflection Depth",
            zorder=5
        )

        plt.gca().invert_yaxis()
        plt.xticks(x, labels, rotation=45, ha="right")
        plt.ylabel("Depth (m)")
        plt.xlabel(group_col)
        plt.title(f"Depth of Max Bending Moment vs First Inflection Depth ({lc})")
        plt.legend()
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        save_show(os.path.join(output_dir, f"{prefix}_moment_vs_inflection_{lc}"))

# =========================================================
# RUNNERS
# =========================================================
def run_sediment_shear_bending(bm_file, defl_file, output_dir):
    output_dir = make_output_dir(output_dir)

    bm_df = prep_sediment_bm(bm_file)
    defl_df = prep_sediment_defl(defl_file)

    bm_df.to_csv(os.path.join(output_dir, "shear_force_bending_moment_cleaned.csv"), index=False)
    defl_df.to_csv(os.path.join(output_dir, "deflection_cleaned_for_inflection.csv"), index=False)

    bm_stats = compute_bm_stats(bm_df, "Sediment Type")
    first_inflection_df = compute_first_inflection_df(defl_df, "Sediment Type")
    compare_df = bm_stats.merge(first_inflection_df, on=["Sediment Type", "Load Case"], how="left")
    compare_df["Depth Difference (m)"] = compare_df["Depth Mmax (m)"] - compare_df["First Inflection Depth (m)"]
    compare_df["Abs Depth Difference (m)"] = compare_df["Depth Difference (m)"].abs()

    bm_stats.to_csv(os.path.join(output_dir, "bending_moment_shear_stats.csv"), index=False)
    first_inflection_df.to_csv(os.path.join(output_dir, "first_inflection_depths.csv"), index=False)
    compare_df.to_csv(os.path.join(output_dir, "moment_vs_inflection_comparison.csv"), index=False)

    plot_bm_profiles(bm_df, "Sediment Type", sediment_order, sediment_colors, output_dir, "sediment")
    plot_shear_profiles(bm_df, "Sediment Type", sediment_order, sediment_colors, output_dir, "sediment")
    plot_moment_vs_inflection(compare_df, "Sediment Type", sediment_order, sediment_colors, output_dir, "sediment")

    return bm_df, bm_stats, first_inflection_df, compare_df

def run_water_shear_bending(bm_file, defl_file, output_dir):
    output_dir = make_output_dir(output_dir)

    bm_df = prep_water_or_embedment_bm(bm_file, "Water Depth")
    defl_df = prep_water_or_embedment_defl(defl_file, "Water Depth")

    bm_df = bm_df[bm_df["Water Depth Numeric"].isin(water_order)].copy()
    defl_df = defl_df[defl_df["Water Depth Numeric"].isin(water_order)].copy()

    bm_df.to_csv(os.path.join(output_dir, "shear_force_bending_moment_cleaned_by_water_depth.csv"), index=False)
    defl_df.to_csv(os.path.join(output_dir, "deflection_cleaned_for_inflection_by_water_depth.csv"), index=False)

    bm_stats = compute_bm_stats(bm_df, "Water Depth Numeric")
    first_inflection_df = compute_first_inflection_df(defl_df, "Water Depth Numeric")
    compare_df = bm_stats.merge(first_inflection_df, on=["Water Depth Numeric", "Load Case"], how="left")
    compare_df["Depth Difference (m)"] = compare_df["Depth Mmax (m)"] - compare_df["First Inflection Depth (m)"]
    compare_df["Abs Depth Difference (m)"] = compare_df["Depth Difference (m)"].abs()

    bm_stats.to_csv(os.path.join(output_dir, "bending_moment_shear_stats_by_water_depth.csv"), index=False)
    first_inflection_df.to_csv(os.path.join(output_dir, "first_inflection_depths_by_water_depth.csv"), index=False)
    compare_df.to_csv(os.path.join(output_dir, "moment_vs_inflection_comparison_by_water_depth.csv"), index=False)

    plot_bm_profiles(bm_df, "Water Depth Numeric", water_order, water_colors, output_dir, "water")
    plot_shear_profiles(bm_df, "Water Depth Numeric", water_order, water_colors, output_dir, "water")
    plot_moment_vs_inflection(compare_df, "Water Depth Numeric", water_order, water_colors, output_dir, "water")

    return bm_df, bm_stats, first_inflection_df, compare_df

def run_embedment_shear_bending(bm_file, defl_file, output_dir):
    output_dir = make_output_dir(output_dir)

    bm_df = prep_water_or_embedment_bm(bm_file, "Embedment Depth")
    defl_df = prep_water_or_embedment_defl(defl_file, "Embedment Depth")

    bm_df = bm_df[bm_df["Embedment Depth Numeric"].isin(embedment_order)].copy()
    defl_df = defl_df[defl_df["Embedment Depth Numeric"].isin(embedment_order)].copy()

    bm_df.to_csv(os.path.join(output_dir, "shear_force_bending_moment_cleaned_by_embedment_depth.csv"), index=False)
    defl_df.to_csv(os.path.join(output_dir, "deflection_cleaned_for_inflection_by_embedment_depth.csv"), index=False)

    bm_stats = compute_bm_stats(bm_df, "Embedment Depth Numeric")
    first_inflection_df = compute_first_inflection_df(defl_df, "Embedment Depth Numeric")
    compare_df = bm_stats.merge(first_inflection_df, on=["Embedment Depth Numeric", "Load Case"], how="left")
    compare_df["Depth Difference (m)"] = compare_df["Depth Mmax (m)"] - compare_df["First Inflection Depth (m)"]
    compare_df["Abs Depth Difference (m)"] = compare_df["Depth Difference (m)"].abs()

    bm_stats.to_csv(os.path.join(output_dir, "bending_moment_shear_stats_by_embedment_depth.csv"), index=False)
    first_inflection_df.to_csv(os.path.join(output_dir, "first_inflection_depths_by_embedment_depth.csv"), index=False)
    compare_df.to_csv(os.path.join(output_dir, "moment_vs_inflection_comparison_by_embedment_depth.csv"), index=False)

    plot_bm_profiles(bm_df, "Embedment Depth Numeric", embedment_order, embedment_colors, output_dir, "embedment")
    plot_shear_profiles(bm_df, "Embedment Depth Numeric", embedment_order, embedment_colors, output_dir, "embedment")
    plot_moment_vs_inflection(compare_df, "Embedment Depth Numeric", embedment_order, embedment_colors, output_dir, "embedment")

    return bm_df, bm_stats, first_inflection_df, compare_df

# =========================================================
# PATHS
# =========================================================
sediment_bm_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/LC_Shear_Force_Bending_Moment.csv"
sediment_defl_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/LC_Defelction_Slope_Pressure.csv"
sediment_out = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/shear_bending_plots"

water_bm_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/shear_force_bending_moment.csv"
water_defl_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/deflection_slope_pressure.csv"
water_out = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/shear_bending_plots"

embedment_bm_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/shear_force_bending_moment.csv"
embedment_defl_file = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/deflection_slope_pressure.csv"
embedment_out = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/shear_bending_plots"

# =========================================================
# RUN
# =========================================================
bm_df_sediment, bm_stats_sediment, first_inflection_df_sediment, compare_df_sediment = run_sediment_shear_bending(
    sediment_bm_file, sediment_defl_file, sediment_out
)

bm_df_water, bm_stats_water, first_inflection_df_water, compare_df_water = run_water_shear_bending(
    water_bm_file, water_defl_file, water_out
)

bm_df_embedment, bm_stats_embedment, first_inflection_df_embedment, compare_df_embedment = run_embedment_shear_bending(
    embedment_bm_file, embedment_defl_file, embedment_out
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# PLOT DEFLECTION WITH INFLECTION POINTS
# =========================================================
def plot_deflection_with_inflections(df, group_col, group_order, color_map, output_dir, prefix):
    load_cases = sorted(df["Load Case"].dropna().unique(), key=lambda x: extract_lc_number(x))

    for lc in load_cases:
        plt.figure(figsize=(8, 10))
        plotted = False

        for grp in group_order:
            sub = df[(df[group_col] == grp) & (df["Load Case"] == lc)].copy()
            if sub.empty:
                continue

            plotted = True
            sub = sub.sort_values("Depth (m)")

            z = sub["Depth (m)"].to_numpy(dtype=float)
            y = sub["Lateral deflection (mm)"].to_numpy(dtype=float)

            label = f"{int(grp)} m" if isinstance(grp, (int, float, np.integer, np.floating)) else str(grp)

            plt.plot(
                y,
                z,
                marker="o",
                linewidth=2,
                markersize=4,
                label=label,
                color=color_map.get(grp, None)
            )

            inflections = get_inflection_depths(z, y, window=5, polyorder=2)

            # plot first inflection only
            if len(inflections) > 0:
                d = inflections[0]
                idx = np.argmin(np.abs(z - d))

                plt.scatter(
                    y[idx],
                    z[idx],
                    color="red",
                    s=40,
                    zorder=6
                )

                plt.text(
                    y[idx],
                    z[idx],
                    f"  {d:.1f} m",
                    color="red",
                    fontsize=8,
                    va="center",
                    ha="left"
                )

        if not plotted:
            plt.close()
            continue

        plt.gca().invert_yaxis()
        plt.xlabel("Lateral Deflection (mm)")
        plt.ylabel("Depth (m)")
        plt.title(f"Deflection vs Depth with First Inflection Point ({lc})")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{prefix}_deflection_with_inflection_{lc}.png"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(output_dir, f"{prefix}_deflection_with_inflection_{lc}.pdf"), bbox_inches="tight")
        plt.show()
        plt.close()

# sediment
plot_deflection_with_inflections(
    sediment_defl_df,
    "Sediment Type",
    sediment_order,
    sediment_colors,
    sediment_out,
    "sediment"
)

# water depth
plot_deflection_with_inflections(
    water_defl_df,
    "Water Depth Numeric",
    water_order,
    water_colors,
    water_out,
    "water"
)

# embedment depth
plot_deflection_with_inflections(
    embedment_defl_df,
    "Embedment Depth Numeric",
    embedment_order,
    embedment_colors,
    embedment_out,
    "embedment"
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# OUTPUT DIRECTORY
# =========================================================
output_dir = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/fixity_depth_plots"
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# FILE PATHS
# =========================================================
files = {
    "sediment": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/gems_compare_output/LC_Defelction_Slope_Pressure.csv",
    "water_depth": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/downloaded_csv/deflection_slope_pressure.csv",
    "embedment": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/downloaded_csv/deflection_slope_pressure.csv"
}

# =========================================================
# COLOR MAPS
# =========================================================
sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

water_depth_colors = {
    "10 m": "#ADD8E6",
    "30 m": "#A6BEE3",
    "50 m": "#A0A4E0",
    "70 m": "#998ADE",
    "90 m": "#9370DB",
    "10.0 m": "#ADD8E6",
    "30.0 m": "#A6BEE3",
    "50.0 m": "#A0A4E0",
    "70.0 m": "#998ADE",
    "90.0 m": "#9370DB",
}

embedment_colors = {
    "15 m": "#FFD700",
    "25 m": "#FFBF00",
    "30 m": "#FFA500",
    "35 m": "#FF8C00",
    "40 m": "#FF6A00",
    "45 m": "#E34234",
    "55 m": "#CC0000",
    "15.0 m": "#FFD700",
    "25.0 m": "#FFBF00",
    "30.0 m": "#FFA500",
    "35.0 m": "#FF8C00",
    "40.0 m": "#FF6A00",
    "45.0 m": "#E34234",
    "55.0 m": "#CC0000",
}

default_color = "gray"

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def extract_lc(label):
    m = re.search(r"LC\s*-?\s*(\d+)", str(label), flags=re.IGNORECASE)
    return f"LC{m.group(1)}" if m else str(label).strip()

def extract_number(text):
    m = re.search(r"(\d+\.?\d*)", str(text))
    return float(m.group(1)) if m else np.nan

def sort_group_values(series):
    extracted = series.astype(str).str.extract(r'(\d+\.?\d*)', expand=False)
    numeric = pd.to_numeric(extracted, errors="coerce")
    if numeric.notna().any():
        return numeric
    return series.astype(str)

def find_zero_crossing_depth(depths, vals):
    """
    Prefer true zero crossing by interpolation.
    If no zero crossing exists, use the depth where deflection is closest to zero.
    """
    depths = pd.to_numeric(pd.Series(depths), errors="coerce").to_numpy(dtype=float)
    vals = pd.to_numeric(pd.Series(vals), errors="coerce").to_numpy(dtype=float)

    mask = np.isfinite(depths) & np.isfinite(vals)
    depths = depths[mask]
    vals = vals[mask]

    if len(depths) == 0:
        return np.nan, np.nan, "no_data"

    order = np.argsort(depths)
    depths = depths[order]
    vals = vals[order]

    for i in range(len(vals) - 1):
        y1, y2 = vals[i], vals[i + 1]
        z1, z2 = depths[i], depths[i + 1]

        if y1 == 0:
            return z1, y1, "exact_zero"

        if y1 * y2 < 0:
            z_zero = z1 - y1 * (z2 - z1) / (y2 - y1)
            return z_zero, 0.0, "interpolated_zero"

    idx = np.argmin(np.abs(vals))
    return depths[idx], vals[idx], "closest_to_zero"

def get_group_colors(groups, dataset_name):
    colors = []
    for g in groups:
        g = str(g).strip()
        if dataset_name == "sediment":
            colors.append(sediment_colors.get(g, default_color))
        elif dataset_name == "water_depth":
            colors.append(water_depth_colors.get(g, default_color))
        elif dataset_name == "embedment":
            colors.append(embedment_colors.get(g, default_color))
        else:
            colors.append(default_color)
    return colors

# =========================================================
# PARSING
# =========================================================
def parse_deflection_file(path, dataset_name):
    df = pd.read_csv(path)
    df = clean_columns(df)

    combo_col = None
    for c in df.columns:
        cl = c.lower()
        if "loadcase" in cl or "load case" in cl:
            combo_col = c
            break

    if combo_col is None:
        raise ValueError(f"Could not find combined LoadCase column in {dataset_name} file.")

    split_cols = df[combo_col].astype(str).str.split("-", n=1, expand=True)

    df["Load Case"] = split_cols[0].str.strip().apply(extract_lc)
    df["Parsed Group"] = split_cols[1].str.strip()

    # Assign true grouping label
    if dataset_name == "sediment":
        df["Group"] = df["Parsed Group"].astype(str).str.strip()
    elif dataset_name == "water_depth":
        df["Group_num"] = df["Parsed Group"].apply(extract_number)
        df = df[df["Group_num"].isin([10, 30, 50, 70, 90])].copy()
        df["Group"] = df["Group_num"].astype(int).astype(str) + " m"
    elif dataset_name == "embedment":
        df["Group_num"] = df["Parsed Group"].apply(extract_number)
        df = df[df["Group_num"].isin([15, 25, 30, 35, 40, 45, 55])].copy()
        df["Group"] = df["Group_num"].astype(int).astype(str) + " m"
    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name}")

    # Find numeric columns
    depth_col = None
    deflection_col = None
    for c in df.columns:
        if c.lower() == "depth (m)" or c.lower() == "depth":
            depth_col = c
        if "deflection" in c.lower():
            deflection_col = c

    if depth_col is None:
        raise ValueError(f"Could not detect depth column for {dataset_name}.")
    if deflection_col is None:
        raise ValueError(f"Could not detect deflection column for {dataset_name}.")

    df[depth_col] = pd.to_numeric(df[depth_col], errors="coerce")
    df[deflection_col] = pd.to_numeric(df[deflection_col], errors="coerce")

    df = df.dropna(subset=[depth_col, deflection_col, "Group", "Load Case"]).copy()

    df = df.rename(columns={
        depth_col: "Depth (m)",
        deflection_col: "Lateral Deflection (mm)"
    })

    return df

# =========================================================
# FIXITY CALCULATION
# =========================================================
def compute_fixity_by_loadcase(df):
    rows = []

    for (group, lc), sub in df.groupby(["Group", "Load Case"]):
        sub = sub.sort_values("Depth (m)").copy()

        z_fix, y_fix, method = find_zero_crossing_depth(
            sub["Depth (m)"],
            sub["Lateral Deflection (mm)"]
        )

        rows.append({
            "Group": group,
            "Load Case": lc,
            "Fixity Depth (m)": z_fix,
            "Deflection at Fixity": y_fix,
            "Method": method
        })

    out = pd.DataFrame(rows)
    out["_sort"] = sort_group_values(out["Group"])
    out = out.sort_values(["_sort", "Load Case"]).drop(columns="_sort")
    return out

def find_fixity_depth(depths, vals, embedment_depth):
    depths = np.array(depths, dtype=float)
    vals = np.array(vals, dtype=float)

    mask = np.isfinite(depths) & np.isfinite(vals)
    depths = depths[mask]
    vals = vals[mask]

    # 🔥 CRITICAL: restrict to within embedment
    mask = depths <= embedment_depth
    depths = depths[mask]
    vals = vals[mask]

    if len(depths) == 0:
        return np.nan, "no_data"

    # check for zero crossing WITHIN embedment
    for i in range(len(vals) - 1):
        y1, y2 = vals[i], vals[i + 1]
        z1, z2 = depths[i], depths[i + 1]

        if y1 * y2 < 0:
            z_zero = z1 - y1 * (z2 - z1) / (y2 - y1)
            return z_zero, "zero_crossing_within_embedment"

    # fallback: closest to zero within embedment
    idx = np.argmin(np.abs(vals))
    return depths[idx], "closest_within_embedment"
    
    for (group, lc), sub in df.groupby(["Group", "Load Case"]):

        sub = sub.sort_values("Depth (m)").copy()
    
        embedment_depth = float(group.split()[0])  # "30 m" → 30
    
        z_fix, method = find_fixity_depth(
            sub["Depth (m)"],
            sub["Lateral Deflection (mm)"],
            embedment_depth
        )
    
        rows.append({
            "Group": group,
            "Load Case": lc,
            "Fixity Depth (m)": z_fix,
            "Method": method
        })
def summarize_fixity(fixity_df):
    summary_df = (
        fixity_df.groupby("Group")["Fixity Depth (m)"]
        .agg(["mean", "min", "max", "std"])
        .reset_index()
    )

    summary_df = summary_df.rename(columns={
        "mean": "Mean Fixity Depth (m)",
        "min": "Min Fixity Depth (m)",
        "max": "Max Fixity Depth (m)",
        "std": "Std Dev (m)"
    })

    summary_df["Lower Error"] = summary_df["Mean Fixity Depth (m)"] - summary_df["Min Fixity Depth (m)"]
    summary_df["Upper Error"] = summary_df["Max Fixity Depth (m)"] - summary_df["Mean Fixity Depth (m)"]

    summary_df["_sort"] = sort_group_values(summary_df["Group"])
    summary_df = summary_df.sort_values("_sort").drop(columns="_sort")
    return summary_df

# =========================================================
# PLOTTING
# =========================================================
def plot_fixity_bar(summary_df, dataset_name, x_label, title, filename):
    colors = get_group_colors(summary_df["Group"], dataset_name)

    x = np.arange(len(summary_df))
    y = summary_df["Mean Fixity Depth (m)"].values
    yerr = np.vstack([
        summary_df["Lower Error"].values,
        summary_df["Upper Error"].values
    ])

    plt.figure(figsize=(10, 6))
    plt.bar(
        x,
        y,
        yerr=yerr,
        capsize=5,
        color=colors
    )

    plt.xticks(x, summary_df["Group"].astype(str), rotation=45, ha="right")
    plt.ylabel("Fixity Depth / Near-Zero Deflection Depth (m)")
    plt.xlabel(x_label)
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()

    out_png = os.path.join(output_dir, f"{filename}.png")
    out_pdf = os.path.join(output_dir, f"{filename}.pdf")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close()

# =========================================================
# RUN ONE DATASET
# =========================================================
def run_fixity_analysis(path, dataset_name, x_label, title, filename_base):
    df = parse_deflection_file(path, dataset_name)
    fixity_df = compute_fixity_by_loadcase(df)
    summary_df = summarize_fixity(fixity_df)

    fixity_csv = os.path.join(output_dir, f"{filename_base}_fixity_by_loadcase.csv")
    summary_csv = os.path.join(output_dir, f"{filename_base}_fixity_summary_mean_range.csv")

    fixity_df.to_csv(fixity_csv, index=False)
    summary_df.to_csv(summary_csv, index=False)

    plot_fixity_bar(
        summary_df=summary_df,
        dataset_name=dataset_name,
        x_label=x_label,
        title=title,
        filename=f"{filename_base}_fixity_depth_bar"
    )

    print(f"\nSaved detailed fixity table: {fixity_csv}")
    print(f"Saved summary table: {summary_csv}")
    print(summary_df)

    return df, fixity_df, summary_df

def check_zero_crossing(df):
    results = []

    for (grp, lc), sub in df.groupby(["Group", "Load Case"]):
        sub = sub.sort_values("Depth (m)")
        y = sub["Lateral Deflection (mm)"].values

        has_positive = np.any(y > 0)
        has_negative = np.any(y < 0)

        results.append({
            "Group": grp,
            "Load Case": lc,
            "Has Positive": has_positive,
            "Has Negative": has_negative,
            "Crosses Zero": has_positive and has_negative
        })

    return pd.DataFrame(results)

check_df = check_zero_crossing(embedment_df)
print(check_df)

# =========================================================
# RUN ALL THREE
# =========================================================
sediment_df, sediment_fixity_df, sediment_fixity_summary = run_fixity_analysis(
    files["sediment"],
    dataset_name="sediment",
    x_label="Sediment Type",
    title="Fixity Depth by Sediment Type",
    filename_base="sediment"
)

water_df, water_fixity_df, water_fixity_summary = run_fixity_analysis(
    files["water_depth"],
    dataset_name="water_depth",
    x_label="Water Depth",
    title="Fixity Depth by Water Depth",
    filename_base="water_depth"
)

embedment_df, embedment_fixity_df, embedment_fixity_summary = run_fixity_analysis(
    files["embedment"],
    dataset_name="embedment",
    x_label="Embedment Depth",
    title="Fixity Depth by Embedment Depth",
    filename_base="embedment_depth"
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

output_dir = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/fixity_depth_plots"
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def sort_group_values(series):
    extracted = series.astype(str).str.extract(r'(\d+\.?\d*)', expand=False)
    numeric = pd.to_numeric(extracted, errors="coerce")
    if numeric.notna().any():
        return numeric
    return series.astype(str)

def extract_embedment_from_group(group_label):
    m = re.search(r"(\d+\.?\d*)", str(group_label))
    return float(m.group(1)) if m else np.nan

sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

water_depth_colors = {
    "10 m": "#ADD8E6",
    "30 m": "#A6BEE3",
    "50 m": "#A0A4E0",
    "70 m": "#998ADE",
    "90 m": "#9370DB"
}

embedment_colors = {
    "15 m": "#FFD700",
    "25 m": "#FFBF00",
    "30 m": "#FFA500",
    "35 m": "#FF8C00",
    "40 m": "#FF6A00",
    "45 m": "#E34234",
    "55 m": "#CC0000"
}

def get_colors(groups, dataset_name):
    colors = []
    for g in groups:
        g = str(g).strip()
        if dataset_name == "sediment":
            colors.append(sediment_colors.get(g, "gray"))
        elif dataset_name == "water_depth":
            colors.append(water_depth_colors.get(g, "gray"))
        elif dataset_name == "embedment":
            colors.append(embedment_colors.get(g, "gray"))
        else:
            colors.append("gray")
    return colors

# --------------------------------------------------
# LOAD YOUR EXISTING SUMMARY TABLES
# --------------------------------------------------
sediment_summary = pd.read_csv(
    os.path.join(output_dir, "sediment_fixity_summary_mean_range.csv")
)

water_summary = pd.read_csv(
    os.path.join(output_dir, "water_depth_fixity_summary_mean_range.csv")
)

embedment_summary = pd.read_csv(
    os.path.join(output_dir, "embedment_depth_fixity_summary_mean_range.csv")
)

# Sort
for df in [sediment_summary, water_summary, embedment_summary]:
    df["_sort"] = sort_group_values(df["Group"])
    df.sort_values("_sort", inplace=True)
    df.drop(columns="_sort", inplace=True)

# --------------------------------------------------
# ADD REFERENCE DEPTHS
# --------------------------------------------------
# Sediment + water were all modeled with 45 m embedment
sediment_summary["Reference Depth (m)"] = 45.0
water_summary["Reference Depth (m)"] = 45.0

# Embedment plot gets its own embedment depth as reference
embedment_summary["Reference Depth (m)"] = embedment_summary["Group"].apply(extract_embedment_from_group)

# --------------------------------------------------
# PLOTTING FUNCTION
# --------------------------------------------------
def plot_fixity_with_reference(summary_df, dataset_name, x_label, title, filename):
    summary_df = summary_df.copy()

    x = np.arange(len(summary_df))
    y = summary_df["Mean Fixity Depth (m)"].values

    yerr = np.vstack([
        summary_df["Lower Error"].values,
        summary_df["Upper Error"].values
    ])

    ref = summary_df["Reference Depth (m)"].values
    colors = get_colors(summary_df["Group"], dataset_name)

    plt.figure(figsize=(10, 6))

    # Fixity bars
    plt.bar(
        x,
        y,
        yerr=yerr,
        capsize=5,
        color=colors,
        alpha=0.9,
        label="Mean fixity depth"
    )

    # Reference markers
    plt.scatter(
        x,
        ref,
        color="black",
        marker="D",
        s=55,
        zorder=5,
        label="Embedment depth reference"
    )

    # Optional dashed line from marker to top of bar
    for xi, yi, ri in zip(x, y, ref):
        plt.plot([xi, xi], [yi, ri], color="black", linestyle="--", linewidth=1, alpha=0.7)

    # Optional numeric labels for reference markers
    for xi, ri in zip(x, ref):
        plt.text(
            xi,
            ri,
            f" {ri:.0f} m",
            fontsize=8,
            va="bottom",
            ha="left",
            color="black"
        )

    plt.xticks(x, summary_df["Group"].astype(str), rotation=45, ha="right")
    plt.ylabel("Depth (m)")
    plt.xlabel(x_label)
    plt.title(title)
    plt.legend()
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(os.path.join(output_dir, filename + ".png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, filename + ".pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

# --------------------------------------------------
# MAKE THE THREE FIGURES
# --------------------------------------------------
plot_fixity_with_reference(
    sediment_summary,
    dataset_name="sediment",
    x_label="Sediment Type",
    title="Fixity Depth by Sediment Type with 45 m Embedment Reference",
    filename="sediment_fixity_with_45m_reference"
)

plot_fixity_with_reference(
    water_summary,
    dataset_name="water_depth",
    x_label="Water Depth",
    title="Fixity Depth by Water Depth with 45 m Embedment Reference",
    filename="water_depth_fixity_with_45m_reference"
)

plot_fixity_with_reference(
    embedment_summary,
    dataset_name="embedment",
    x_label="Embedment Depth",
    title="Fixity Depth by Embedment Depth with Embedment Reference",
    filename="embedment_depth_fixity_with_embedment_reference"
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, kruskal, levene, ttest_rel, wilcoxon

# =========================================================
# 1. PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"

FACTOR_DIRS = {
    "sediment": os.path.join(BASE_DIR, "Sediment Comparison", "organized"),
    "water_depth": os.path.join(BASE_DIR, "Water Depth Comparison", "organized"),
    "embedment": os.path.join(BASE_DIR, "Embedment Depth Comparison", "organized"),
}

OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. CONFIG
# =========================================================
RAW_LIKE_SKIP_THRESHOLD = 400

EXCLUDE_NUMERIC_COLS = {
    "Node", "No.", "Depth (m)", "z (m)", "Depth",
    "WaterDepth_num", "Water Depth Numeric",
    "Embedment_num", "Embedment Depth Numeric",
    "LoadCase_num"
}

GROUP_CANDIDATES = {
    "sediment": ["Sediment Type", "Group", "Parsed Group"],
    "water_depth": ["Water Depth", "WaterDepth_num", "Water Depth Numeric", "Group", "Parsed Group"],
    "embedment": ["Embedment Depth", "Embedment_num", "Embedment Depth Numeric", "Group", "Parsed Group"]
}

LOADCASE_CANDIDATES = ["Load Case", "Load-case", "LoadCase", "Load case"]

# =========================================================
# 3. HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def normalize_load_case(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"LC\s*-?\s*(\d+)", str(val), flags=re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    return str(val).strip()

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def guess_family(file_name):
    f = file_name.lower()

    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "pile_capacity" in f or "pile capacity" in f:
        return "pile_capacity"
    if "unit_resistance" in f or "unit resistance" in f:
        return "unit_resistance"
    if "pile_settlement" in f or "pile settlement" in f:
        return "pile_settlement"
    if "axial_settlement" in f or "axial settlement" in f:
        return "axial_settlement"
    if "axial_force" in f or "axial force" in f:
        return "axial_force"
    if "axial_stress" in f or "axial stress" in f:
        return "axial_stress"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "shear" in f or "bending" in f or "moment" in f:
        return "shear_bending"
    if "fixity" in f:
        return "fixity"
    return "other"

def find_group_col(df, factor_name):
    for c in GROUP_CANDIDATES[factor_name]:
        if c in df.columns:
            return c
    return None

def find_loadcase_col(df):
    for c in LOADCASE_CANDIDATES:
        if c in df.columns:
            return c
    return None

def is_raw_like(df):
    cols = set(df.columns)
    has_depth = "Depth (m)" in cols or "Depth" in cols
    has_row_id = "Node" in cols or "No." in cols
    many_rows = len(df) > RAW_LIKE_SKIP_THRESHOLD
    return has_depth and has_row_id and many_rows

def coerce_group_label(val, factor_name):
    if pd.isna(val):
        return np.nan

    if factor_name == "sediment":
        return str(val).strip()

    num = extract_number(val)
    if pd.isna(num):
        return str(val).strip()

    return f"{int(num)} m"

def sort_group_values(series):
    extracted = series.astype(str).str.extract(r'(\d+\.?\d*)', expand=False)
    numeric = pd.to_numeric(extracted, errors="coerce")
    if numeric.notna().any():
        return numeric
    return series.astype(str)

def is_depth_metric(metric_name):
    m = metric_name.lower()
    depth_keywords = [
        "depth ", " depth", "inflection depth", "fixity depth",
        "depth of max", "depth mmax", "depth vmax", "depth difference"
    ]
    return any(k in m for k in depth_keywords)

def guess_orientation(metric_name):
    m = metric_name.lower()

    if is_depth_metric(m):
        return None

    higher_keywords = [
        "stiffness", "capacity", "resistance",
        "q_max", "qmax", "p_max", "pmax", "max_t", "t_max",
        "area_under_curve", "area", "terminal q"
    ]

    lower_keywords = [
        "moment magnitude", "mmax magnitude",
        "vmax magnitude", "deflection", "slope", "pressure",
        "stress", "force", "settlement",
        "cv", "brittleness", "plateau_fraction",
        "mismatch", "difference"
    ]

    if any(k in m for k in higher_keywords):
        return "higher"

    if any(k in m for k in lower_keywords):
        return "lower"

    return None

def eta_squared_from_groups(groups):
    all_vals = np.concatenate(groups)
    grand_mean = np.mean(all_vals)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in groups)
    ss_total = np.sum((all_vals - grand_mean) ** 2)
    return ss_between / ss_total if ss_total != 0 else np.nan

def discover_csvs(folder):
    csvs = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.join(root, f))
    return sorted(csvs)

def split_metric_and_lc(metric_name):
    """
    Example:
    'Max Force (kN)_LC1' -> ('Max Force (kN)', 'LC1')
    """
    m = re.search(r"^(.*)_LC(\d+)$", str(metric_name).strip(), flags=re.IGNORECASE)
    if m:
        base = m.group(1).strip()
        lc = f"LC{m.group(2)}"
        return base, lc
    return str(metric_name).strip(), None

# =========================================================
# 4. BUILD metrics_long CORRECTLY
# =========================================================
def build_metrics_long():
    rows = []
    inventory_rows = []

    for factor_name, folder in FACTOR_DIRS.items():
        if not os.path.exists(folder):
            print(f"Missing folder: {folder}")
            continue

        files = discover_csvs(folder)

        for path in files:
            file_name = os.path.basename(path)
            family = guess_family(file_name)

            try:
                df = pd.read_csv(path)
                df = clean_columns(df)
            except Exception as e:
                inventory_rows.append({
                    "factor": factor_name,
                    "file_name": file_name,
                    "path": path,
                    "family": family,
                    "status": f"read_error: {e}"
                })
                continue

            if is_raw_like(df):
                inventory_rows.append({
                    "factor": factor_name,
                    "file_name": file_name,
                    "path": path,
                    "family": family,
                    "status": "skipped_raw_like"
                })
                continue

            group_col = find_group_col(df, factor_name)
            load_col = find_loadcase_col(df)

            if group_col is None:
                inventory_rows.append({
                    "factor": factor_name,
                    "file_name": file_name,
                    "path": path,
                    "family": family,
                    "status": "skipped_no_group_col"
                })
                continue

            df = df.copy()
            df["__group__"] = df[group_col].apply(lambda x: coerce_group_label(x, factor_name))

            if load_col is not None:
                df["__load_case__"] = df[load_col].apply(normalize_load_case)
            else:
                df["__load_case__"] = "ALL"

            numeric_cols = [
                c for c in df.columns
                if pd.api.types.is_numeric_dtype(df[c]) and c not in EXCLUDE_NUMERIC_COLS
            ]

            numeric_cols = [
                c for c in numeric_cols
                if c not in {"__group__", "__load_case__"}
                and not c.lower().endswith("_num")
            ]

            if len(numeric_cols) == 0:
                inventory_rows.append({
                    "factor": factor_name,
                    "file_name": file_name,
                    "path": path,
                    "family": family,
                    "status": "skipped_no_numeric_metrics"
                })
                continue

            inventory_rows.append({
                "factor": factor_name,
                "file_name": file_name,
                "path": path,
                "family": family,
                "status": "used",
                "group_col": group_col,
                "load_col": load_col if load_col is not None else "ALL",
                "n_rows": len(df),
                "n_metrics": len(numeric_cols)
            })

            for _, row in df.iterrows():
                for metric in numeric_cols:
                    val = row[metric]
                    if pd.isna(val):
                        continue

                    base_metric, lc_from_metric = split_metric_and_lc(metric)

                    final_lc = lc_from_metric if lc_from_metric is not None else row["__load_case__"]

                    rows.append({
                        "factor": factor_name,
                        "family": family,
                        "source_file": file_name,
                        "group": row["__group__"],
                        "load_case": final_lc,
                        "metric": base_metric,
                        "value": float(val)
                    })

    inventory_df = pd.DataFrame(inventory_rows)
    metrics_long = pd.DataFrame(rows)
    return metrics_long, inventory_df

metrics_long, inventory_df = build_metrics_long()

inventory_df.to_csv(os.path.join(OUTPUT_DIR, "file_inventory_used_for_master_pipeline.csv"), index=False)
metrics_long.to_csv(os.path.join(OUTPUT_DIR, "metrics_long_all_factors.csv"), index=False)

print("metrics_long shape:", metrics_long.shape)
print(metrics_long.head(10))
print("\nUnique load cases:")
print(sorted(metrics_long["load_case"].dropna().unique()))
print("\nSample metrics:")
print(sorted(metrics_long["metric"].dropna().unique())[:20])

# =========================================================
# 5. BETWEEN-GROUP STATS
# =========================================================
def run_between_group_stats(metrics_long):
    out_rows = []

    for (factor, family, metric, load_case), sub in metrics_long.groupby(["factor", "family", "metric", "load_case"]):
        work = sub[["group", "value"]].dropna().copy()

        groups = []
        group_names = []

        for g, gsub in work.groupby("group"):
            vals = gsub["value"].astype(float).values
            vals = vals[np.isfinite(vals)]
            if len(vals) >= 2:
                groups.append(vals)
                group_names.append(g)

        if len(groups) < 2:
            continue

        try:
            aov = f_oneway(*groups)
            aov_F, aov_p = aov.statistic, aov.pvalue
        except Exception:
            aov_F, aov_p = np.nan, np.nan

        try:
            kw = kruskal(*groups)
            kw_H, kw_p = kw.statistic, kw.pvalue
        except Exception:
            kw_H, kw_p = np.nan, np.nan

        try:
            lev = levene(*groups, center="median")
            lev_W, lev_p = lev.statistic, lev.pvalue
        except Exception:
            lev_W, lev_p = np.nan, np.nan

        eta_sq = eta_squared_from_groups(groups)

        out_rows.append({
            "factor": factor,
            "family": family,
            "metric": metric,
            "load_case": load_case,
            "n_groups": len(groups),
            "groups_tested": ", ".join(map(str, group_names)),
            "anova_F": aov_F,
            "anova_p": aov_p,
            "kruskal_H": kw_H,
            "kruskal_p": kw_p,
            "levene_W": lev_W,
            "levene_p": lev_p,
            "eta_squared": eta_sq
        })

    return pd.DataFrame(out_rows)

between_group_stats = run_between_group_stats(metrics_long)
between_group_stats.to_csv(os.path.join(OUTPUT_DIR, "between_group_stats_by_factor_family_metric_loadcase.csv"), index=False)

# =========================================================
# 6. LOAD CASE VS LC1 STATS
# =========================================================
def run_lc_vs_lc1_stats(metrics_long):
    out_rows = []

    for (factor, family, metric), sub in metrics_long.groupby(["factor", "family", "metric"]):
        pivot = (
            sub.groupby(["group", "load_case"], as_index=False)["value"]
            .mean()
            .pivot(index="group", columns="load_case", values="value")
        )

        if "LC1" not in pivot.columns:
            continue

        other_lcs = [c for c in pivot.columns if c != "LC1" and str(c).startswith("LC")]

        for lc in sorted(other_lcs, key=lambda x: int(re.search(r"\d+", x).group())):
            pair = pivot[["LC1", lc]].dropna().copy()
            if len(pair) < 2:
                continue

            x1 = pair["LC1"].astype(float).values
            x2 = pair[lc].astype(float).values

            try:
                t_res = ttest_rel(x1, x2, nan_policy="omit")
                t_stat, t_p = t_res.statistic, t_res.pvalue
            except Exception:
                t_stat, t_p = np.nan, np.nan

            try:
                w_res = wilcoxon(x1, x2)
                w_stat, w_p = w_res.statistic, w_res.pvalue
            except Exception:
                w_stat, w_p = np.nan, np.nan

            out_rows.append({
                "factor": factor,
                "family": family,
                "metric": metric,
                "compare_to_lc1": lc,
                "n_groups": len(pair),
                "paired_t_stat": t_stat,
                "paired_t_p": t_p,
                "wilcoxon_stat": w_stat,
                "wilcoxon_p": w_p,
                "mean_LC1": np.mean(x1),
                "mean_compare": np.mean(x2),
                "delta_compare_minus_lc1": np.mean(x2) - np.mean(x1)
            })

    return pd.DataFrame(out_rows)

lc_vs_lc1_stats = run_lc_vs_lc1_stats(metrics_long)
lc_vs_lc1_stats.to_csv(os.path.join(OUTPUT_DIR, "loadcase_vs_lc1_stats_by_factor_family_metric.csv"), index=False)

# =========================================================
# 7. RANKING TABLES
# =========================================================
def build_ranking_table(metrics_long, use_lc1_only=False):
    work = metrics_long.copy()

    if use_lc1_only:
        work = work[work["load_case"] == "LC1"].copy()

    agg = (
        work.groupby(["factor", "group", "family", "metric"], as_index=False)["value"]
        .mean()
    )

    agg["metric_key"] = agg["family"] + " | " + agg["metric"]

    wide = agg.pivot_table(
        index=["factor", "group"],
        columns="metric_key",
        values="value",
        aggfunc="mean"
    ).reset_index()

    metric_cols = [c for c in wide.columns if c not in ["factor", "group"]]

    rank_series = {}
    for col in metric_cols:
        direction = guess_orientation(col)
        if direction is None:
            continue

        ascending = True if direction == "lower" else False
        rank_col = f"RANK | {col}"

        rank_series[rank_col] = (
            wide.groupby("factor")[col]
            .rank(ascending=ascending, method="dense")
        )

    if rank_series:
        rank_df = pd.DataFrame(rank_series)
        wide = pd.concat([wide, rank_df], axis=1)
        rank_cols = list(rank_df.columns)
    else:
        rank_cols = []

    wide["Combined Raw Rank"] = wide[rank_cols].sum(axis=1, min_count=1)
    wide["Ranked Metric Count"] = wide[rank_cols].notna().sum(axis=1)

    wide["_sort"] = wide["group"].apply(lambda x: extract_number(x) if pd.notna(extract_number(x)) else np.nan)
    wide = wide.sort_values(["factor", "Combined Raw Rank", "_sort", "group"]).drop(columns="_sort")

    return wide

ranking_all_lc = build_ranking_table(metrics_long, use_lc1_only=False)
ranking_lc1 = build_ranking_table(metrics_long, use_lc1_only=True)

ranking_all_lc.to_csv(os.path.join(OUTPUT_DIR, "ranking_all_loadcases_raw_ranks.csv"), index=False)
ranking_lc1.to_csv(os.path.join(OUTPUT_DIR, "ranking_lc1_only_raw_ranks.csv"), index=False)

# =========================================================
# 8. SMALL SUMMARY TABLES
# =========================================================
def top_summary(ranking_df, label):
    cols = ["factor", "group", "Combined Raw Rank", "Ranked Metric Count"]
    out = ranking_df[cols].copy().sort_values(["factor", "Combined Raw Rank", "group"])
    out.to_csv(os.path.join(OUTPUT_DIR, f"top_summary_{label}.csv"), index=False)
    return out

top_all = top_summary(ranking_all_lc, "all_loadcases")
top_lc1 = top_summary(ranking_lc1, "lc1_only")

print("\nTop summary, all load cases:")
print(top_all.head(20))

print("\nTop summary, LC1 only:")
print(top_lc1.head(20))

# =========================================================
# 9. BAR GRAPHS
# =========================================================
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# COLOR MAPS
# =========================================================
sediment_order = [
    "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
    "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
    "sand_silt_sand", "f_silty_sand", "sandy_silt"
]

sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

water_colors = {
    "10 m": "#ADD8E6",
    "30 m": "#A6BEE3",
    "50 m": "#A0A4E0",
    "70 m": "#998ADE",
    "90 m": "#9370DB"
}

embedment_colors = {
    "15 m": "#FFD700",
    "25 m": "#FFBF00",
    "30 m": "#FFA500",
    "35 m": "#FF8C00",
    "40 m": "#FF6A00",
    "45 m": "#E34234",
    "55 m": "#CC0000"
}

# =========================================================
# HELPERS
# =========================================================
def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def get_factor_colors(groups, factor):
    colors = []

    for g in groups:
        g = str(g).strip()

        if factor == "sediment":
            colors.append(sediment_colors.get(g, "gray"))
        elif factor == "water_depth":
            colors.append(water_colors.get(g, "gray"))
        elif factor == "embedment":
            colors.append(embedment_colors.get(g, "gray"))
        else:
            colors.append("gray")

    return colors

def sort_groups_for_factor(df, factor):
    df = df.copy()

    if factor == "sediment":
        # keep your sediment palette order if desired, but still rank by score first
        df["group_order"] = df["group"].map({g: i for i, g in enumerate(sediment_order)})
    else:
        df["group_order"] = df["group"].apply(extract_number)

    df = df.sort_values(["Combined Raw Rank", "group_order", "group"])
    return df.drop(columns="group_order")

# =========================================================
# PLOT FUNCTION
# =========================================================
def plot_ranking_bars_colored(summary_df, title_suffix, filename_suffix, output_dir):
    factors = ["sediment", "water_depth", "embedment"]

    for factor in factors:
        sub = summary_df[summary_df["factor"] == factor].copy()
        if sub.empty:
            continue

        sub = sort_groups_for_factor(sub, factor)

        x = np.arange(len(sub))
        y = sub["Combined Raw Rank"].values
        groups = sub["group"].astype(str).tolist()
        colors = get_factor_colors(groups, factor)

        plt.figure(figsize=(10, 6))
        plt.bar(x, y, color=colors)

        plt.xticks(x, groups, rotation=45, ha="right")
        plt.ylabel("Combined Raw Rank")
        plt.xlabel(factor.replace("_", " ").title())
        plt.title(f"{factor.replace('_', ' ').title()} Ranking — {title_suffix}")
        plt.grid(True, axis="y", alpha=0.3)

        # add value labels
        for xi, yi in zip(x, y):
            plt.text(
                xi,
                yi,
                f"{yi:.0f}",
                ha="center",
                va="bottom",
                fontsize=8
            )

        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, f"{factor}_{filename_suffix}.png"),
            dpi=300,
            bbox_inches="tight"
        )
        plt.savefig(
            os.path.join(output_dir, f"{factor}_{filename_suffix}.pdf"),
            bbox_inches="tight"
        )
        plt.show()
        plt.close()

# =========================================================
# RUN ON YOUR EXISTING SUMMARY TABLES
# =========================================================
plot_ranking_bars_colored(
    top_all,
    title_suffix="All Load Cases",
    filename_suffix="ranking_bar_all_loadcases_colored",
    output_dir=OUTPUT_DIR
)

plot_ranking_bars_colored(
    top_lc1,
    title_suffix="LC1 Only",
    filename_suffix="ranking_bar_lc1_only_colored",
    output_dir=OUTPUT_DIR
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Sediment Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "sediment_load_scaled_ranking")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# CONTEXT / MODEL NOTES
# =========================================================
MODEL_NOTES = {
    "sediment_type_constant_embedment_m": 45.0,
    "sediment_type_constant_water_depth_m": 22.5,
    "water_depth_constant_embedment_m": 45.0,
    "embedment_depth_constant_water_depth_m": 22.5,
    "water_depth_and_embedment_use_medium_sand": True
}

# =========================================================
# SEDIMENT COLORS
# =========================================================
sediment_order = [
    "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
    "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
    "sand_silt_sand", "f_silty_sand", "sandy_silt"
]

sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

# =========================================================
# LOAD MAPS
# =========================================================
# Standard sediment load cases
standard_loads = {
    "LC1": {"axial": 20000.0, "lateral": 2500.0, "moment": 375000.0},
    "LC2": {"axial": 20000.0, "lateral": 3250.0, "moment": 375000.0},
    "LC3": {"axial": 20500.0, "lateral": 2500.0, "moment": 375000.0},
    "LC4": {"axial": 20000.0, "lateral": 2500.0, "moment": 375000.0},
    "LC5": {"axial": 20000.0, "lateral": 3250.0, "moment": 375000.0},
}

# Sandy silt custom loads
sandy_silt_loads = {
    "LC1": {"axial": 15750.0, "lateral": 700.0, "moment": 105000.0},
    "LC2": {"axial": 15750.0, "lateral": 800.0, "moment": 105000.0},
    "LC3": {"axial": 16250.0, "lateral": 700.0, "moment": 105000.0},
    "LC4": {"axial": 15750.0, "lateral": 700.0, "moment": 105000.0},
    "LC5": {"axial": 15750.0, "lateral": 800.0, "moment": 105000.0},
}

# Fine silty sand special baseline case
# LC6 should stand in for LC1-equivalent comparison, per your note
# Fine silty sand custom loads
f_silty_sand_loads = {
    "LC1": {"axial": 20000.0, "lateral": 1500.0, "moment": 225000.0},
    "LC2": {"axial": 20000.0, "lateral": 1950.0, "moment": 225000.0},
    "LC3": {"axial": 20500.0, "lateral": 1500.0, "moment": 225000.0},
    "LC4": {"axial": 20000.0, "lateral": 1200.0, "moment": 180000.0},
    "LC5": {"axial": 20000.0, "lateral": 1950.0, "moment": 225000.0},
    "LC6": {"axial": 20000.0, "lateral": 1200.0, "moment": 180000.0},
}

def get_loads_for_sediment(group, load_case):
    group = str(group).strip()
    load_case = str(load_case).strip().upper()

    if group == "sandy_silt":
        return sandy_silt_loads.get(load_case, np.nan)

    if group == "f_silty_sand":
        return f_silty_sand_loads.get(load_case, np.nan)

    return standard_loads.get(load_case, np.nan)


# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                paths.append(os.path.join(root, f))
    return sorted(paths)

def guess_family(file_name):
    f = file_name.lower()
    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "pile_capacity" in f:
        return "pile_capacity"
    if "unit_resistance" in f:
        return "unit_resistance"
    if "pile_settlement" in f:
        return "pile_settlement"
    if "axial_settlement" in f:
        return "axial_settlement"
    if "axial_force" in f:
        return "axial_force"
    if "axial_stress" in f:
        return "axial_stress"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "shear" in f or "bending" in f or "moment" in f:
        return "shear_bending"
    if "fixity" in f:
        return "fixity"
    return "other"

def normalize_load_case(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.search(r"LC\s*-?\s*(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    m2 = re.fullmatch(r"(\d+)", s)
    if m2:
        return f"LC{m2.group(1)}"
    return s

def split_metric_and_lc(metric_name):
    m = re.search(r"^(.*)_LC(\d+)$", str(metric_name).strip(), flags=re.IGNORECASE)
    if m:
        return m.group(1).strip(), f"LC{m.group(2)}"
    return str(metric_name).strip(), None

def find_group_col(df):
    for c in ["Sediment Type", "Group", "Parsed Group"]:
        if c in df.columns:
            return c
    return None

def find_loadcase_col(df):
    for c in ["Load Case", "Load-case", "LoadCase", "Load case"]:
        if c in df.columns:
            return c
    return None

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def is_depth_metric(metric_name):
    m = metric_name.lower()
    depth_keywords = [
        "depth ", " depth", "inflection depth", "fixity depth",
        "depth of max", "depth mmax", "depth vmax", "depth difference",
        "deepest depth", "depth contribution", "z_at_", "z50", "z_max"
    ]
    return any(k in m for k in depth_keywords)

def metric_scale_type(family, metric):
    m = metric.lower()

    # Skip metrics that should not be load-scaled
    if is_depth_metric(metric):
        return None

    # axial-demand-scaled
    if family in ["axial_force", "axial_stress", "axial_settlement"]:
        return "axial"

    # lateral-demand-scaled
    if family in ["deflection_slope_pressure", "p_y"]:
        return "lateral"

    # moment/lateral scaled structural demand
    if family == "shear_bending":
        if "moment" in m or "mmax" in m:
            return "moment"
        if "shear" in m or "vmax" in m or "deflection" in m:
            return "lateral"
        return None

    # capacity can be compared against applied load
    if family == "pile_capacity":
        if "capacity" in m or "shaft friction" in m or "base capacity" in m or "total capacity" in m:
            return "axial"
        return None

    if family == "pile_settlement":
        return "axial"

    if family in ["q_z", "t_z", "unit_resistance", "fixity", "other"]:
        return None

    return None

def guess_orientation(metric_name):
    m = metric_name.lower()

    if is_depth_metric(m):
        return None

    higher_keywords = [
        "stiffness", "capacity", "resistance",
        "q_max", "qmax", "p_max", "pmax", "max_t", "t_max",
        "area_under_curve", "area", "terminal q"
    ]
    lower_keywords = [
        "moment magnitude", "mmax magnitude", "vmax magnitude",
        "deflection", "slope", "pressure",
        "stress", "force", "settlement",
        "cv", "brittleness", "difference",
        "scaled"
    ]

    if any(k in m for k in higher_keywords):
        return "higher"
    if any(k in m for k in lower_keywords):
        return "lower"
    return None

# =========================================================
# BUILD LONG TABLE FOR SEDIMENT ONLY
# =========================================================
def build_sediment_metrics_long():
    rows = []
    inventory = []

    for path in discover_csvs(ORGANIZED_DIR):
        file_name = os.path.basename(path)
        family = guess_family(file_name)

        try:
            df = pd.read_csv(path)
            df = clean_columns(df)
        except Exception as e:
            inventory.append({"file": file_name, "status": f"read_error: {e}"})
            continue

        group_col = find_group_col(df)
        if group_col is None:
            inventory.append({"file": file_name, "status": "skipped_no_group"})
            continue

        load_col = find_loadcase_col(df)

        df = df.copy()
        df["__group__"] = df[group_col].astype(str).str.strip()

        if load_col is not None:
            df["__load_case__"] = df[load_col].apply(normalize_load_case)
        else:
            df["__load_case__"] = "ALL"

        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric_cols = [
            c for c in numeric_cols
            if c not in {"Node", "No.", "Depth (m)", "Depth", "Water Depth", "Embedment Depth",
                         "__group__", "__load_case__"}
            and not c.lower().endswith("_num")
        ]

        if not numeric_cols:
            inventory.append({"file": file_name, "status": "skipped_no_numeric"})
            continue

        inventory.append({"file": file_name, "status": "used", "family": family, "n_metrics": len(numeric_cols)})

        for _, row in df.iterrows():
            for metric in numeric_cols:
                val = row[metric]
                if pd.isna(val):
                    continue

                base_metric, lc_from_metric = split_metric_and_lc(metric)
                final_lc = lc_from_metric if lc_from_metric is not None else row["__load_case__"]

                rows.append({
                    "family": family,
                    "source_file": file_name,
                    "group": row["__group__"],
                    "load_case": final_lc,
                    "metric": base_metric,
                    "value": float(val)
                })

    return pd.DataFrame(rows), pd.DataFrame(inventory)

metrics_long, inventory_df = build_sediment_metrics_long()
inventory_df.to_csv(os.path.join(OUTPUT_DIR, "sediment_file_inventory.csv"), index=False)
metrics_long.to_csv(os.path.join(OUTPUT_DIR, "sediment_metrics_long_raw.csv"), index=False)

# =========================================================
# ADD LOADS + SCALED METRICS
# =========================================================
def attach_loads(df):
    df = df.copy()

    axial_list = []
    lateral_list = []
    moment_list = []

    for _, row in df.iterrows():
        loads = get_loads_for_sediment(row["group"], row["load_case"])

        if isinstance(loads, dict):
            axial_list.append(loads["axial"])
            lateral_list.append(loads["lateral"])
            moment_list.append(loads["moment"])
        else:
            axial_list.append(np.nan)
            lateral_list.append(np.nan)
            moment_list.append(np.nan)

    df["Applied Axial Load (kN)"] = axial_list
    df["Applied Lateral Load (kN)"] = lateral_list
    df["Applied Moment (kNm)"] = moment_list
    return df

def add_scaled_metrics(df):
    df = df.copy()

    scaled_values = []
    scaled_metric_names = []
    scaling_used = []

    for _, row in df.iterrows():
        scale_type = metric_scale_type(row["family"], row["metric"])

        if scale_type == "axial" and pd.notna(row["Applied Axial Load (kN)"]) and row["Applied Axial Load (kN)"] != 0:
            scaled_values.append(row["value"] / row["Applied Axial Load (kN)"])
            scaled_metric_names.append(f"{row['metric']} [scaled]")
            scaling_used.append("axial")
        elif scale_type == "lateral" and pd.notna(row["Applied Lateral Load (kN)"]) and row["Applied Lateral Load (kN)"] != 0:
            scaled_values.append(row["value"] / row["Applied Lateral Load (kN)"])
            scaled_metric_names.append(f"{row['metric']} [scaled]")
            scaling_used.append("lateral")
        elif scale_type == "moment" and pd.notna(row["Applied Moment (kNm)"]) and row["Applied Moment (kNm)"] != 0:
            scaled_values.append(row["value"] / row["Applied Moment (kNm)"])
            scaled_metric_names.append(f"{row['metric']} [scaled]")
            scaling_used.append("moment")
        else:
            scaled_values.append(np.nan)
            scaled_metric_names.append(np.nan)
            scaling_used.append(None)

    df["scaled_value"] = scaled_values
    df["scaled_metric"] = scaled_metric_names
    df["scaling_used"] = scaling_used
    return df

metrics_long = attach_loads(metrics_long)
metrics_long = add_scaled_metrics(metrics_long)
metrics_long.to_csv(os.path.join(OUTPUT_DIR, "sediment_metrics_long_with_scaled_values.csv"), index=False)

# =========================================================
# BUILD LC1-EQUIVALENT BASELINE TABLE
# =========================================================
# For f_silty_sand, use LC6 as the LC1-equivalent baseline
def keep_baseline_cases(df):
    df = df.copy()

    keep_mask = []
    baseline_label = []

    for _, row in df.iterrows():
        g = row["group"]
        lc = row["load_case"]

        if g == "f_silty_sand":
            keep = (lc == "LC6")
            base = "LC6_as_LC1"
        else:
            keep = (lc == "LC1")
            base = "LC1"

        keep_mask.append(keep)
        baseline_label.append(base)

    df["baseline_label"] = baseline_label
    return df[np.array(keep_mask)].copy()

baseline_df = keep_baseline_cases(metrics_long)

# only use scaled metrics for the fair sediment ranking
baseline_scaled = baseline_df.dropna(subset=["scaled_metric", "scaled_value"]).copy()

# =========================================================
# BUILD SCALED RANKING TABLE
# =========================================================
def build_scaled_ranking_table(df):
    agg = (
        df.groupby(["group", "family", "scaled_metric"], as_index=False)["scaled_value"]
        .mean()
    )

    agg["metric_key"] = agg["family"] + " | " + agg["scaled_metric"]

    wide = agg.pivot_table(
        index="group",
        columns="metric_key",
        values="scaled_value",
        aggfunc="mean"
    ).reset_index()

    metric_cols = [c for c in wide.columns if c != "group"]

    rank_cols = []
    for col in metric_cols:
        direction = guess_orientation(col)
        if direction is None:
            continue

        ascending = True if direction == "lower" else False
        rank_col = f"RANK | {col}"
        wide[rank_col] = wide[col].rank(ascending=ascending, method="dense")
        rank_cols.append(rank_col)

    wide["Combined Scaled Rank"] = wide[rank_cols].sum(axis=1, min_count=1)
    wide["Ranked Metric Count"] = wide[rank_cols].notna().sum(axis=1)

    order_map = {g: i for i, g in enumerate(sediment_order)}
    wide["order"] = wide["group"].map(order_map)
    wide = wide.sort_values(["Combined Scaled Rank", "order", "group"]).drop(columns="order")

    return wide

scaled_ranking = build_scaled_ranking_table(baseline_scaled)
scaled_ranking.to_csv(os.path.join(OUTPUT_DIR, "sediment_load_scaled_ranking.csv"), index=False)

scaled_summary = scaled_ranking[["group", "Combined Scaled Rank", "Ranked Metric Count"]].copy()
scaled_summary.to_csv(os.path.join(OUTPUT_DIR, "sediment_load_scaled_ranking_summary.csv"), index=False)

print("\nLoad-scaled sediment ranking summary:")
print(scaled_summary)

# =========================================================
# PLOT
# =========================================================
def plot_scaled_ranking(summary_df):
    df = summary_df.copy()
    order_map = {g: i for i, g in enumerate(sediment_order)}
    df["order"] = df["group"].map(order_map)
    df = df.sort_values(["Combined Scaled Rank", "order", "group"]).drop(columns="order")

    colors = [sediment_colors.get(g, "gray") for g in df["group"]]

    plt.figure(figsize=(10, 6))
    plt.bar(df["group"], df["Combined Scaled Rank"], color=colors)

    for i, v in enumerate(df["Combined Scaled Rank"]):
        plt.text(i, v, f"{v:.0f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Combined Load-Scaled Rank")
    plt.xlabel("Sediment Type")
    plt.title("Sediment Ranking Using Load-Scaled Baseline Response\n(LC1 for most sediments; LC6 baseline for f_silty_sand)")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(os.path.join(OUTPUT_DIR, "sediment_load_scaled_ranking_bar.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, "sediment_load_scaled_ranking_bar.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

plot_scaled_ranking(scaled_summary)

# =========================================================
# SAVE MODEL NOTES
# =========================================================
notes_df = pd.DataFrame([MODEL_NOTES])
notes_df.to_csv(os.path.join(OUTPUT_DIR, "model_notes_context.csv"), index=False)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Sediment Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "sediment_curated_ranking")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# SEDIMENT ORDER + COLORS
# =========================================================
sediment_order = [
    "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
    "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
    "sand_silt_sand", "f_silty_sand", "sandy_silt"
]

sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

# =========================================================
# CURATED METRICS
# =========================================================
# lower = better unless otherwise noted
CURATED_METRICS = {
    "lateral_performance": [
        {"family": "deflection_slope_pressure", "metric": "Max Deflection Magnitude (mm)", "direction": "lower"},
        {"family": "deflection_slope_pressure", "metric": "Slope (rad)", "direction": "lower"},
        {"family": "deflection_slope_pressure", "metric": "First Inflection Depth (m)", "direction": "lower"},
        {"family": "fixity", "metric": "Mean Fixity Depth (m)", "direction": "lower"},
    ],
    "axial_settlement_performance": [
        {"family": "pile_settlement", "metric": "Max Head (mm)", "direction": "lower"},
        {"family": "pile_settlement", "metric": "Pile shaft settlement (mm)", "direction": "lower"},
    ],
    "capacity_resistance": [
        {"family": "pile_capacity", "metric": "Ultimate total capacity (kN)", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_shaft", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_base", "direction": "higher"},
        {"family": "p_y", "metric": "initial_stiffness", "direction": "higher"},
        {"family": "q_z", "metric": "Q_max", "direction": "higher"},
        {"family": "t_z", "metric": "t_max", "direction": "higher"},
    ],
    "variability_risk": [
        {"family": "other", "metric": "CV", "direction": "lower"},
        {"family": "unit_resistance", "metric": "cv_unit_shaft", "direction": "lower"},
        {"family": "unit_resistance", "metric": "cv_unit_base", "direction": "lower"},
    ]
}

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                paths.append(os.path.join(root, f))
    return sorted(paths)

def guess_family(file_name):
    f = file_name.lower()
    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "pile_capacity" in f:
        return "pile_capacity"
    if "unit_resistance" in f:
        return "unit_resistance"
    if "pile_settlement" in f:
        return "pile_settlement"
    if "axial_settlement" in f:
        return "axial_settlement"
    if "axial_force" in f:
        return "axial_force"
    if "axial_stress" in f:
        return "axial_stress"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "shear" in f or "bending" in f or "moment" in f:
        return "shear_bending"
    if "fixity" in f:
        return "fixity"
    return "other"

def find_group_col(df):
    for c in ["Sediment Type", "Group", "Parsed Group"]:
        if c in df.columns:
            return c
    return None

def normalize_load_case(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.search(r"LC\s*-?\s*(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    m2 = re.fullmatch(r"(\d+)", s)
    if m2:
        return f"LC{m2.group(1)}"
    return s

def find_loadcase_col(df):
    for c in ["Load Case", "Load-case", "LoadCase", "Load case"]:
        if c in df.columns:
            return c
    return None

def split_metric_and_lc(metric_name):
    m = re.search(r"^(.*)_LC(\d+)$", str(metric_name).strip(), flags=re.IGNORECASE)
    if m:
        return m.group(1).strip(), f"LC{m.group(2)}"
    return str(metric_name).strip(), None

def make_metric_key(family, metric):
    return f"{family} | {metric}"

def get_color_list(groups):
    return [sediment_colors.get(g, "gray") for g in groups]

# =========================================================
# BUILD LONG TABLE
# =========================================================
def build_metrics_long():
    rows = []

    for path in discover_csvs(ORGANIZED_DIR):
        file_name = os.path.basename(path)
        family = guess_family(file_name)

        try:
            df = pd.read_csv(path)
            df = clean_columns(df)
        except Exception:
            continue

        group_col = find_group_col(df)
        if group_col is None:
            continue

        load_col = find_loadcase_col(df)

        df = df.copy()
        df["__group__"] = df[group_col].astype(str).str.strip()

        if load_col is not None:
            df["__load_case__"] = df[load_col].apply(normalize_load_case)
        else:
            df["__load_case__"] = "ALL"

        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric_cols = [
            c for c in numeric_cols
            if c not in {"Node", "No.", "Depth (m)", "Depth", "Water Depth", "Embedment Depth",
                         "__group__", "__load_case__"}
            and not c.lower().endswith("_num")
        ]

        for _, row in df.iterrows():
            for metric in numeric_cols:
                val = row[metric]
                if pd.isna(val):
                    continue

                base_metric, lc_from_metric = split_metric_and_lc(metric)
                final_lc = lc_from_metric if lc_from_metric is not None else row["__load_case__"]

                rows.append({
                    "family": family,
                    "source_file": file_name,
                    "group": row["__group__"],
                    "load_case": final_lc,
                    "metric": base_metric,
                    "metric_key": make_metric_key(family, base_metric),
                    "value": float(val)
                })

    return pd.DataFrame(rows)

metrics_long = build_metrics_long()

# =========================================================
# BASELINE FILTER
# =========================================================
# Use LC1 for the curated sediment ranking.
# If a metric does not have load case structure, keep ALL.


baseline_df = metrics_long[
    (metrics_long["load_case"] == "LC1") | (metrics_long["load_case"] == "ALL")
].copy()

(
    baseline_df.loc[
        baseline_df["family"] == "deflection_slope_pressure",
        "metric"
    ].dropna().unique().tolist()
)

# =========================================================
# EXTRACT CURATED METRICS
# =========================================================
curated_rows = []

for category, metrics in CURATED_METRICS.items():
    for item in metrics:
        fam = item["family"]
        met = item["metric"]
        direction = item["direction"]
        key = make_metric_key(fam, met)

        sub = baseline_df[
            (baseline_df["family"] == fam) &
            (baseline_df["metric"] == met)
        ].copy()

        if sub.empty:
            print(f"Missing metric: {key}")
            continue

        grouped = (
            sub.groupby("group", as_index=False)["value"]
            .mean()
        )

        # special extra column for slope in degrees
        if fam == "deflection_slope_pressure" and met == "Max Slope (rad)":
            grouped["value_deg"] = np.degrees(grouped["value"])

        grouped["category"] = category
        grouped["family"] = fam
        grouped["metric"] = met
        grouped["metric_key"] = key
        grouped["direction"] = direction

        curated_rows.append(grouped)

curated_values = pd.concat(curated_rows, ignore_index=True)
curated_values.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_metric_values.csv"), index=False)

# =========================================================
# RANK EACH METRIC
# =========================================================
rank_rows = []

for (category, metric_key, direction), sub in curated_values.groupby(["category", "metric_key", "direction"]):
    work = sub.copy()

    ascending = True if direction == "lower" else False
    work["metric_rank"] = work["value"].rank(ascending=ascending, method="dense")

    rank_rows.append(work)

curated_ranks = pd.concat(rank_rows, ignore_index=True)
curated_ranks.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_metric_ranks.csv"), index=False)

# =========================================================
# FACTOR SCORES
# =========================================================
factor_scores = (
    curated_ranks.groupby(["group", "category"], as_index=False)
    .agg(
        factor_rank_sum=("metric_rank", "sum"),
        metric_count=("metric_rank", "count"),
        factor_rank_mean=("metric_rank", "mean")
    )
)

factor_scores.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_factor_scores.csv"), index=False)

# =========================================================
# OVERALL RANKING
# =========================================================
overall_scores = (
    factor_scores.groupby("group", as_index=False)
    .agg(
        overall_rank_sum=("factor_rank_sum", "sum"),
        total_metric_count=("metric_count", "sum"),
        overall_mean_rank=("factor_rank_mean", "mean")
    )
)

overall_scores["overall_rank_position"] = overall_scores["overall_rank_sum"].rank(ascending=True, method="dense")
order_map = {g: i for i, g in enumerate(sediment_order)}
overall_scores["order"] = overall_scores["group"].map(order_map)
overall_scores = overall_scores.sort_values(["overall_rank_sum", "order", "group"]).drop(columns="order")

overall_scores.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_overall_ranking.csv"), index=False)

print("\nCurated overall sediment ranking:")
print(overall_scores)

# =========================================================
# PLOTTING
# =========================================================
def plot_factor_bar(factor_scores, category, title, filename):
    sub = factor_scores[factor_scores["category"] == category].copy()
    sub["order"] = sub["group"].map({g: i for i, g in enumerate(sediment_order)})
    sub = sub.sort_values(["factor_rank_sum", "order", "group"]).drop(columns="order")

    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["factor_rank_sum"], color=colors)

    for i, v in enumerate(sub["factor_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Factor Rank Sum")
    plt.xlabel("Sediment Type")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

def plot_overall_bar(overall_scores):
    sub = overall_scores.copy()
    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["overall_rank_sum"], color=colors)

    for i, v in enumerate(sub["overall_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Overall Curated Rank Sum")
    plt.xlabel("Sediment Type")
    plt.title("Overall Curated Sediment Ranking")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_sediment_ranking.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_sediment_ranking.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

plot_factor_bar(
    factor_scores,
    "lateral_performance",
    "Sediment Ranking: Lateral Performance",
    "lateral_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "axial_settlement_performance",
    "Sediment Ranking: Axial / Settlement Performance",
    "axial_settlement_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "capacity_resistance",
    "Sediment Ranking: Capacity / Resistance",
    "capacity_resistance_ranking"
)

plot_factor_bar(
    factor_scores,
    "variability_risk",
    "Sediment Ranking: Variability / Risk",
    "variability_risk_ranking"
)

plot_overall_bar(overall_scores)


# =========================================================
# OPTIONAL: MAKE A WIDE TABLE FOR READING
# =========================================================
metric_value_table = curated_values.pivot_table(
    index="group",
    columns="metric_key",
    values="value",
    aggfunc="mean"
).reset_index()

# try to find any slope metric that was actually included
slope_rows = curated_values[
    (curated_values["family"] == "deflection_slope_pressure") &
    (curated_values["metric"].astype(str).str.contains("Slope", case=False, na=False))
].copy()

if not slope_rows.empty:
    slope_rows["value_deg"] = np.degrees(slope_rows["value"])

    slope_deg_table = (
        slope_rows[["group", "metric", "value_deg"]]
        .drop_duplicates()
        .pivot_table(index="group", columns="metric", values="value_deg", aggfunc="mean")
        .reset_index()
    )

    # rename columns so they are obvious
    slope_deg_table = slope_deg_table.rename(
        columns={c: f"deflection_slope_pressure | {c.replace('(rad)', '(deg)')}"
                 for c in slope_deg_table.columns if c != "group"}
    )

    metric_value_table = metric_value_table.merge(slope_deg_table, on="group", how="left")
else:
    print("No slope metric found in curated_values, so degree conversion was skipped.")

metric_value_table.to_csv(
    os.path.join(OUTPUT_DIR, "sediment_curated_metric_values_wide.csv"),
    index=False
)

# # add slope in degrees as a separate readable table
# slope_deg_table = curated_values[
#     (curated_values["family"] == "deflection_slope_pressure") &
#     (curated_values["metric"] == "Max Slope (rad)")
# ][["group", "value_deg"]].drop_duplicates()

# metric_value_table = metric_value_table.merge(slope_deg_table, on="group", how="left")
# metric_value_table = metric_value_table.rename(columns={"value_deg": "deflection_slope_pressure | Max Slope (deg)"})
# metric_value_table.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_metric_values_wide.csv"), index=False)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Sediment Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "sediment_curated_ranking")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# SEDIMENT ORDER + COLORS
# =========================================================
sediment_order = [
    "f_sand", "vc_sand", "sand_slc_sand", "sand_psc_sand",
    "slc_sand_slc", "sand_dgs_grvl", "sand_ngs_grvl",
    "sand_silt_sand", "f_silty_sand", "sandy_silt"
]

sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

# =========================================================
# CURATED METRICS
# =========================================================
# lower = better unless otherwise noted
CURATED_METRICS = {
    "lateral_performance": [
        {"family": "deflection_slope_pressure", "metric": "Max Deflection Magnitude (mm)", "direction": "lower"},
        {"family": "deflection_slope_pressure", "metric": "Slope (rad)", "direction": "lower"},
        {"family": "deflection_slope_pressure", "metric": "First Inflection Depth (m)", "direction": "lower"},
        {"family": "fixity", "metric": "Mean Fixity Depth (m)", "direction": "lower"},
    ],
    "axial_settlement_performance": [
        {"family": "pile_settlement", "metric": "Max Head (mm)", "direction": "lower"},
        {"family": "pile_settlement", "metric": "Pile shaft settlement (mm)", "direction": "lower"},
    ],
    "capacity_resistance": [
        {"family": "pile_capacity", "metric": "Ultimate total capacity (kN)", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_shaft", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_base", "direction": "higher"},
        {"family": "p_y", "metric": "initial_stiffness", "direction": "higher"},
        {"family": "q_z", "metric": "Q_max", "direction": "higher"},
        {"family": "t_z", "metric": "t_max", "direction": "higher"},
    ],
    "variability_risk": [
        {"family": "other", "metric": "CV", "direction": "lower"},
        {"family": "unit_resistance", "metric": "cv_unit_shaft", "direction": "lower"},
        {"family": "unit_resistance", "metric": "cv_unit_base", "direction": "lower"},
    ]
}

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                paths.append(os.path.join(root, f))
    return sorted(paths)

def guess_family(file_name):
    f = file_name.lower()
    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "pile_capacity" in f:
        return "pile_capacity"
    if "unit_resistance" in f:
        return "unit_resistance"
    if "pile_settlement" in f:
        return "pile_settlement"
    if "axial_settlement" in f:
        return "axial_settlement"
    if "axial_force" in f:
        return "axial_force"
    if "axial_stress" in f:
        return "axial_stress"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "shear" in f or "bending" in f or "moment" in f:
        return "shear_bending"
    if "fixity" in f:
        return "fixity"
    return "other"

def find_group_col(df):
    for c in ["Sediment Type", "Group", "Parsed Group"]:
        if c in df.columns:
            return c
    return None

def normalize_load_case(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.search(r"LC\s*-?\s*(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    m2 = re.fullmatch(r"(\d+)", s)
    if m2:
        return f"LC{m2.group(1)}"
    return s

def find_loadcase_col(df):
    for c in ["Load Case", "Load-case", "LoadCase", "Load case"]:
        if c in df.columns:
            return c
    return None

def split_metric_and_lc(metric_name):
    m = re.search(r"^(.*)_LC(\d+)$", str(metric_name).strip(), flags=re.IGNORECASE)
    if m:
        return m.group(1).strip(), f"LC{m.group(2)}"
    return str(metric_name).strip(), None

def make_metric_key(family, metric):
    return f"{family} | {metric}"

def get_color_list(groups):
    return [sediment_colors.get(g, "gray") for g in groups]

# =========================================================
# BUILD LONG TABLE
# =========================================================
def build_metrics_long():
    rows = []

    for path in discover_csvs(ORGANIZED_DIR):
        file_name = os.path.basename(path)
        family = guess_family(file_name)

        try:
            df = pd.read_csv(path)
            df = clean_columns(df)
        except Exception:
            continue

        group_col = find_group_col(df)
        if group_col is None:
            continue

        load_col = find_loadcase_col(df)

        df = df.copy()
        df["__group__"] = df[group_col].astype(str).str.strip()

        if load_col is not None:
            df["__load_case__"] = df[load_col].apply(normalize_load_case)
        else:
            df["__load_case__"] = "ALL"

        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric_cols = [
            c for c in numeric_cols
            if c not in {"Node", "No.", "Depth (m)", "Depth", "Water Depth", "Embedment Depth",
                         "__group__", "__load_case__"}
            and not c.lower().endswith("_num")
        ]

        for _, row in df.iterrows():
            for metric in numeric_cols:
                val = row[metric]
                if pd.isna(val):
                    continue

                base_metric, lc_from_metric = split_metric_and_lc(metric)
                final_lc = lc_from_metric if lc_from_metric is not None else row["__load_case__"]

                rows.append({
                    "family": family,
                    "source_file": file_name,
                    "group": row["__group__"],
                    "load_case": final_lc,
                    "metric": base_metric,
                    "metric_key": make_metric_key(family, base_metric),
                    "value": float(val)
                })

    return pd.DataFrame(rows)

metrics_long = build_metrics_long()

# =========================================================
# BASELINE FILTER
# =========================================================
# Use LC1 for the curated sediment ranking.
# If a metric does not have load case structure, keep ALL.


baseline_df = metrics_long[
    (metrics_long["load_case"] == "LC1") | (metrics_long["load_case"] == "ALL")
].copy()

(
    baseline_df.loc[
        baseline_df["family"] == "deflection_slope_pressure",
        "metric"
    ].dropna().unique().tolist()
)

# =========================================================
# EXTRACT CURATED METRICS
# =========================================================
curated_rows = []

for category, metrics in CURATED_METRICS.items():
    for item in metrics:
        fam = item["family"]
        met = item["metric"]
        direction = item["direction"]
        key = make_metric_key(fam, met)

        sub = baseline_df[
            (baseline_df["family"] == fam) &
            (baseline_df["metric"] == met)
        ].copy()

        if sub.empty:
            print(f"Missing metric: {key}")
            continue

        grouped = (
            sub.groupby("group", as_index=False)["value"]
            .mean()
        )

        # special extra column for slope in degrees
        if fam == "deflection_slope_pressure" and met == "Max Slope (rad)":
            grouped["value_deg"] = np.degrees(grouped["value"])

        grouped["category"] = category
        grouped["family"] = fam
        grouped["metric"] = met
        grouped["metric_key"] = key
        grouped["direction"] = direction

        curated_rows.append(grouped)

curated_values = pd.concat(curated_rows, ignore_index=True)
curated_values.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_metric_values.csv"), index=False)

# =========================================================
# RANK EACH METRIC
# =========================================================
rank_rows = []

for (category, metric_key, direction), sub in curated_values.groupby(["category", "metric_key", "direction"]):
    work = sub.copy()

    ascending = True if direction == "lower" else False
    work["metric_rank"] = work["value"].rank(ascending=ascending, method="dense")

    rank_rows.append(work)

curated_ranks = pd.concat(rank_rows, ignore_index=True)
curated_ranks.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_metric_ranks.csv"), index=False)

# =========================================================
# FACTOR SCORES
# =========================================================
factor_scores = (
    curated_ranks.groupby(["group", "category"], as_index=False)
    .agg(
        factor_rank_sum=("metric_rank", "sum"),
        metric_count=("metric_rank", "count"),
        factor_rank_mean=("metric_rank", "mean")
    )
)

factor_scores.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_factor_scores.csv"), index=False)

# =========================================================
# OVERALL RANKING
# =========================================================
overall_scores = (
    factor_scores.groupby("group", as_index=False)
    .agg(
        overall_rank_sum=("factor_rank_sum", "sum"),
        total_metric_count=("metric_count", "sum"),
        overall_mean_rank=("factor_rank_mean", "mean")
    )
)

overall_scores["overall_rank_position"] = overall_scores["overall_rank_sum"].rank(ascending=True, method="dense")
order_map = {g: i for i, g in enumerate(sediment_order)}
overall_scores["order"] = overall_scores["group"].map(order_map)
overall_scores = overall_scores.sort_values(["overall_rank_sum", "order", "group"]).drop(columns="order")

overall_scores.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_overall_ranking.csv"), index=False)

print("\nCurated overall sediment ranking:")
print(overall_scores)

# =========================================================
# PLOTTING
# =========================================================
def plot_factor_bar(factor_scores, category, title, filename):
    sub = factor_scores[factor_scores["category"] == category].copy()
    sub["order"] = sub["group"].map({g: i for i, g in enumerate(sediment_order)})
    sub = sub.sort_values(["factor_rank_sum", "order", "group"]).drop(columns="order")

    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["factor_rank_sum"], color=colors)

    for i, v in enumerate(sub["factor_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Factor Rank Sum")
    plt.xlabel("Sediment Type")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

def plot_overall_bar(overall_scores):
    sub = overall_scores.copy()
    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["overall_rank_sum"], color=colors)

    for i, v in enumerate(sub["overall_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Overall Curated Rank Sum")
    plt.xlabel("Sediment Type")
    plt.title("Overall Curated Sediment Ranking")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_sediment_ranking.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_sediment_ranking.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

plot_factor_bar(
    factor_scores,
    "lateral_performance",
    "Sediment Ranking: Lateral Performance",
    "lateral_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "axial_settlement_performance",
    "Sediment Ranking: Axial / Settlement Performance",
    "axial_settlement_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "capacity_resistance",
    "Sediment Ranking: Capacity / Resistance",
    "capacity_resistance_ranking"
)

plot_factor_bar(
    factor_scores,
    "variability_risk",
    "Sediment Ranking: Variability / Risk",
    "variability_risk_ranking"
)

plot_overall_bar(overall_scores)


# =========================================================
# OPTIONAL: MAKE A WIDE TABLE FOR READING
# =========================================================
metric_value_table = curated_values.pivot_table(
    index="group",
    columns="metric_key",
    values="value",
    aggfunc="mean"
).reset_index()

# try to find any slope metric that was actually included
slope_rows = curated_values[
    (curated_values["family"] == "deflection_slope_pressure") &
    (curated_values["metric"].astype(str).str.contains("Slope", case=False, na=False))
].copy()

if not slope_rows.empty:
    slope_rows["value_deg"] = np.degrees(slope_rows["value"])

    slope_deg_table = (
        slope_rows[["group", "metric", "value_deg"]]
        .drop_duplicates()
        .pivot_table(index="group", columns="metric", values="value_deg", aggfunc="mean")
        .reset_index()
    )

    # rename columns so they are obvious
    slope_deg_table = slope_deg_table.rename(
        columns={c: f"deflection_slope_pressure | {c.replace('(rad)', '(deg)')}"
                 for c in slope_deg_table.columns if c != "group"}
    )

    metric_value_table = metric_value_table.merge(slope_deg_table, on="group", how="left")
else:
    print("No slope metric found in curated_values, so degree conversion was skipped.")

metric_value_table.to_csv(
    os.path.join(OUTPUT_DIR, "sediment_curated_metric_values_wide.csv"),
    index=False
)

# # add slope in degrees as a separate readable table
# slope_deg_table = curated_values[
#     (curated_values["family"] == "deflection_slope_pressure") &
#     (curated_values["metric"] == "Max Slope (rad)")
# ][["group", "value_deg"]].drop_duplicates()

# metric_value_table = metric_value_table.merge(slope_deg_table, on="group", how="left")
# metric_value_table = metric_value_table.rename(columns={"value_deg": "deflection_slope_pressure | Max Slope (deg)"})
# metric_value_table.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_metric_values_wide.csv"), index=False)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Water Depth Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "water_depth_curated_ranking")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# WATER DEPTH ORDER + COLORS
# =========================================================
water_depth_order = ["10 m", "30 m", "50 m", "70 m", "90 m"]

water_cmap = LinearSegmentedColormap.from_list(
    "waterdepth_gradient",
    ["#ADD8E6", "#9370DB"]
)

water_depth_colors = {
    wd: to_hex(water_cmap(i / (len(water_depth_order) - 1)))
    for i, wd in enumerate(water_depth_order)
}

# =========================================================
# CURATED METRICS
# =========================================================
# lower = better unless otherwise noted
CURATED_METRICS = {
    "lateral_performance": [
        {"family": "deflection_slope_pressure", "metric": "Lateral deflection (mm)", "direction": "lower"},
        {"family": "deflection_slope_pressure", "metric": "Slope (rad)", "direction": "lower"},
        {"family": "fixity", "metric": "Mean Fixity Depth (m)", "direction": "lower"},
    ],
    "axial_settlement_performance": [
        {"family": "pile_settlement", "metric": "Max Head (mm)", "direction": "lower"},
        {"family": "pile_settlement", "metric": "Pile shaft settlement (mm)", "direction": "lower"},
    ],
    "capacity_resistance": [
        {"family": "pile_capacity", "metric": "Ultimate total capacity (kN)", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_shaft", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_base", "direction": "higher"},
        {"family": "p_y", "metric": "initial_stiffness", "direction": "higher"},
        {"family": "q_z", "metric": "Q_max", "direction": "higher"},
        {"family": "t_z", "metric": "t_max", "direction": "higher"},
    ],
    "variability_risk": [
        {"family": "unit_resistance", "metric": "cv_unit_shaft", "direction": "lower"},
        {"family": "unit_resistance", "metric": "cv_unit_base", "direction": "lower"},
    ]
}

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                paths.append(os.path.join(root, f))
    return sorted(paths)

def guess_family(file_name):
    f = file_name.lower()

    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "pile_capacity" in f:
        return "pile_capacity"
    if "unit_resistance" in f:
        return "unit_resistance"
    if "pile_settlement" in f:
        return "pile_settlement"
    if "axial_settlement" in f:
        return "axial_settlement"
    if "axial_force" in f:
        return "axial_force"
    if "axial_stress" in f:
        return "axial_stress"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "shear" in f or "bending" in f or "moment" in f:
        return "shear_bending"
    if "fixity" in f:
        return "fixity"
    return "other"

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def normalize_water_depth_label(val):
    if pd.isna(val):
        return np.nan
    num = extract_number(val)
    if pd.isna(num):
        return str(val).strip()
    return f"{int(num)} m"

def find_group_col(df):
    for c in ["Water Depth", "Water Depth (m)", "WaterDepth", "Group", "Parsed Group"]:
        if c in df.columns:
            return c
    return None

def normalize_load_case(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.search(r"LC\s*-?\s*(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    m2 = re.fullmatch(r"(\d+)", s)
    if m2:
        return f"LC{m2.group(1)}"
    return s

def find_loadcase_col(df):
    for c in ["Load Case", "Load-case", "LoadCase", "Load case"]:
        if c in df.columns:
            return c
    return None

def split_metric_and_lc(metric_name):
    m = re.search(r"^(.*)_LC(\d+)$", str(metric_name).strip(), flags=re.IGNORECASE)
    if m:
        return m.group(1).strip(), f"LC{m.group(2)}"
    return str(metric_name).strip(), None

def make_metric_key(family, metric):
    return f"{family} | {metric}"

def get_color_list(groups):
    return [water_depth_colors.get(g, "gray") for g in groups]

def sort_groups_by_water_depth(df, group_col="group"):
    df = df.copy()
    df["_sort"] = df[group_col].apply(extract_number)
    df = df.sort_values(["_sort", group_col]).drop(columns="_sort")
    return df

# =========================================================
# BUILD LONG TABLE
# =========================================================
def build_metrics_long():
    rows = []

    for path in discover_csvs(ORGANIZED_DIR):
        file_name = os.path.basename(path)
        family = guess_family(file_name)

        try:
            df = pd.read_csv(path)
            df = clean_columns(df)
        except Exception:
            continue

        group_col = find_group_col(df)
        if group_col is None:
            continue

        load_col = find_loadcase_col(df)

        df = df.copy()
        df["__group__"] = df[group_col].apply(normalize_water_depth_label)

        if load_col is not None:
            df["__load_case__"] = df[load_col].apply(normalize_load_case)
        else:
            df["__load_case__"] = "ALL"

        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric_cols = [
            c for c in numeric_cols
            if c not in {
                "Node", "No.", "Depth (m)", "Depth",
                "Water Depth", "Water Depth (m)", "WaterDepth",
                "Embedment Depth", "__group__", "__load_case__"
            }
            and not c.lower().endswith("_num")
        ]

        for _, row in df.iterrows():
            for metric in numeric_cols:
                val = row[metric]
                if pd.isna(val):
                    continue

                base_metric, lc_from_metric = split_metric_and_lc(metric)
                final_lc = lc_from_metric if lc_from_metric is not None else row["__load_case__"]

                rows.append({
                    "family": family,
                    "source_file": file_name,
                    "group": row["__group__"],
                    "load_case": final_lc,
                    "metric": base_metric,
                    "metric_key": make_metric_key(family, base_metric),
                    "value": float(val)
                })

    return pd.DataFrame(rows)

metrics_long = build_metrics_long()

# =========================================================
# BASELINE FILTER
# =========================================================
# Use LC1 for the curated water-depth ranking.
# If a metric does not have load-case structure, keep ALL.
baseline_df = metrics_long[
    (metrics_long["load_case"] == "LC1") | (metrics_long["load_case"] == "ALL")
].copy()

print("\nAvailable deflection_slope_pressure metrics in baseline_df:")
print(sorted(
    baseline_df.loc[
        baseline_df["family"] == "deflection_slope_pressure",
        "metric"
    ].dropna().unique().tolist()
))

# =========================================================
# EXTRACT CURATED METRICS
# =========================================================
curated_rows = []

for category, metrics in CURATED_METRICS.items():
    for item in metrics:
        fam = item["family"]
        met = item["metric"]
        direction = item["direction"]
        key = make_metric_key(fam, met)

        sub = baseline_df[
            (baseline_df["family"] == fam) &
            (baseline_df["metric"] == met)
        ].copy()

        if sub.empty:
            print(f"Missing metric: {key}")
            continue

        grouped = (
            sub.groupby("group", as_index=False)["value"]
            .mean()
        )

        # add slope in degrees
        if fam == "deflection_slope_pressure" and met == "Slope (rad)":
            grouped["value_deg"] = np.degrees(grouped["value"])

        grouped["category"] = category
        grouped["family"] = fam
        grouped["metric"] = met
        grouped["metric_key"] = key
        grouped["direction"] = direction

        curated_rows.append(grouped)

curated_values = pd.concat(curated_rows, ignore_index=True)
curated_values = sort_groups_by_water_depth(curated_values, "group")
curated_values.to_csv(os.path.join(OUTPUT_DIR, "water_depth_curated_metric_values.csv"), index=False)

# =========================================================
# RANK EACH METRIC
# =========================================================
rank_rows = []

for (category, metric_key, direction), sub in curated_values.groupby(["category", "metric_key", "direction"]):
    work = sub.copy()

    ascending = True if direction == "lower" else False
    work["metric_rank"] = work["value"].rank(ascending=ascending, method="dense")

    rank_rows.append(work)

curated_ranks = pd.concat(rank_rows, ignore_index=True)
curated_ranks = sort_groups_by_water_depth(curated_ranks, "group")
curated_ranks.to_csv(os.path.join(OUTPUT_DIR, "water_depth_curated_metric_ranks.csv"), index=False)

# =========================================================
# FACTOR SCORES
# =========================================================
factor_scores = (
    curated_ranks.groupby(["group", "category"], as_index=False)
    .agg(
        factor_rank_sum=("metric_rank", "sum"),
        metric_count=("metric_rank", "count"),
        factor_rank_mean=("metric_rank", "mean")
    )
)

factor_scores = sort_groups_by_water_depth(factor_scores, "group")
factor_scores.to_csv(os.path.join(OUTPUT_DIR, "water_depth_curated_factor_scores.csv"), index=False)

# =========================================================
# OVERALL RANKING
# =========================================================
overall_scores = (
    factor_scores.groupby("group", as_index=False)
    .agg(
        overall_rank_sum=("factor_rank_sum", "sum"),
        total_metric_count=("metric_count", "sum"),
        overall_mean_rank=("factor_rank_mean", "mean")
    )
)

overall_scores["overall_rank_position"] = overall_scores["overall_rank_sum"].rank(ascending=True, method="dense")
overall_scores["_sort"] = overall_scores["group"].apply(extract_number)
overall_scores = overall_scores.sort_values(["overall_rank_sum", "_sort", "group"]).drop(columns="_sort")

overall_scores.to_csv(os.path.join(OUTPUT_DIR, "water_depth_curated_overall_ranking.csv"), index=False)

print("\nCurated overall water-depth ranking:")
print(overall_scores)

# =========================================================
# PLOTTING
# =========================================================
def plot_factor_bar(factor_scores, category, title, filename):
    sub = factor_scores[factor_scores["category"] == category].copy()
    sub["_sort"] = sub["group"].apply(extract_number)
    sub = sub.sort_values(["factor_rank_sum", "_sort", "group"]).drop(columns="_sort")

    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["factor_rank_sum"], color=colors)

    for i, v in enumerate(sub["factor_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Factor Rank Sum")
    plt.xlabel("Water Depth")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

def plot_overall_bar(overall_scores):
    sub = overall_scores.copy()
    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["overall_rank_sum"], color=colors)

    for i, v in enumerate(sub["overall_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Overall Curated Rank Sum")
    plt.xlabel("Water Depth")
    plt.title("Overall Curated Water-Depth Ranking")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_water_depth_ranking.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_water_depth_ranking.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

plot_factor_bar(
    factor_scores,
    "lateral_performance",
    "Water-Depth Ranking: Lateral Performance",
    "lateral_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "axial_settlement_performance",
    "Water-Depth Ranking: Axial / Settlement Performance",
    "axial_settlement_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "capacity_resistance",
    "Water-Depth Ranking: Capacity / Resistance",
    "capacity_resistance_ranking"
)

plot_factor_bar(
    factor_scores,
    "variability_risk",
    "Water-Depth Ranking: Variability / Risk",
    "variability_risk_ranking"
)

plot_overall_bar(overall_scores)

# =========================================================
# OPTIONAL: MAKE A WIDE TABLE FOR READING
# =========================================================
metric_value_table = curated_values.pivot_table(
    index="group",
    columns="metric_key",
    values="value",
    aggfunc="mean"
).reset_index()

slope_rows = curated_values[
    (curated_values["family"] == "deflection_slope_pressure") &
    (curated_values["metric"] == "Slope (rad)")
].copy()

if not slope_rows.empty:
    slope_rows["value_deg"] = np.degrees(slope_rows["value"])

    slope_deg_table = (
        slope_rows[["group", "value_deg"]]
        .drop_duplicates()
    )

    metric_value_table = metric_value_table.merge(slope_deg_table, on="group", how="left")
    metric_value_table = metric_value_table.rename(
        columns={"value_deg": "deflection_slope_pressure | Slope (deg)"}
    )
else:
    print("No curated slope metric found, so degree conversion was skipped.")

metric_value_table = sort_groups_by_water_depth(metric_value_table, "group")
metric_value_table.to_csv(
    os.path.join(OUTPUT_DIR, "water_depth_curated_metric_values_wide.csv"),
    index=False
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Embedment Depth Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "embedment_depth_curated_ranking")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# EMBEDMENT ORDER + COLORS
# =========================================================
embedment_order = ["15 m", "25 m", "30 m", "35 m", "40 m", "45 m", "55 m"]

embedment_cmap = LinearSegmentedColormap.from_list(
    "embedment_gradient",
    ["#FFD700", "#FF8C00", "#CC0000"]
)

embedment_colors = {
    ed: to_hex(embedment_cmap(i / (len(embedment_order) - 1)))
    for i, ed in enumerate(embedment_order)
}

# =========================================================
# CURATED METRICS
# =========================================================
CURATED_METRICS = {
    "lateral_performance": [
        {"family": "deflection_slope_pressure", "metric": "Lateral deflection (mm)", "direction": "lower"},
        {"family": "deflection_slope_pressure", "metric": "Slope (rad)", "direction": "lower"},
        {"family": "fixity", "metric": "Mean Fixity Depth (m)", "direction": "lower"},
    ],
    "axial_settlement_performance": [
        {"family": "pile_settlement", "metric": "Max Head (mm)", "direction": "lower"},
        {"family": "pile_settlement", "metric": "Pile shaft settlement (mm)", "direction": "lower"},
    ],
    "capacity_resistance": [
        {"family": "pile_capacity", "metric": "Ultimate total capacity (kN)", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_shaft", "direction": "higher"},
        {"family": "unit_resistance", "metric": "mean_unit_base", "direction": "higher"},
        {"family": "p_y", "metric": "initial_stiffness", "direction": "higher"},
        {"family": "q_z", "metric": "Q_max", "direction": "higher"},
        {"family": "t_z", "metric": "t_max", "direction": "higher"},
    ],
    "variability_risk": [
        {"family": "unit_resistance", "metric": "cv_unit_shaft", "direction": "lower"},
        {"family": "unit_resistance", "metric": "cv_unit_base", "direction": "lower"},
    ]
}

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                paths.append(os.path.join(root, f))
    return sorted(paths)

def guess_family(file_name):
    f = file_name.lower()

    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "pile_capacity" in f:
        return "pile_capacity"
    if "unit_resistance" in f:
        return "unit_resistance"
    if "pile_settlement" in f:
        return "pile_settlement"
    if "axial_settlement" in f:
        return "axial_settlement"
    if "axial_force" in f:
        return "axial_force"
    if "axial_stress" in f:
        return "axial_stress"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "shear" in f or "bending" in f or "moment" in f:
        return "shear_bending"
    if "fixity" in f:
        return "fixity"
    return "other"

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def normalize_embedment_label(val):
    if pd.isna(val):
        return np.nan
    num = extract_number(val)
    if pd.isna(num):
        return str(val).strip()
    return f"{int(num)} m"

def find_group_col(df):
    for c in ["Embedment Depth", "Embedment Depth (m)", "Embedment", "Group", "Parsed Group"]:
        if c in df.columns:
            return c
    return None

def normalize_load_case(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.search(r"LC\s*-?\s*(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    m2 = re.fullmatch(r"(\d+)", s)
    if m2:
        return f"LC{m2.group(1)}"
    return s

def find_loadcase_col(df):
    for c in ["Load Case", "Load-case", "LoadCase", "Load case"]:
        if c in df.columns:
            return c
    return None

def split_metric_and_lc(metric_name):
    m = re.search(r"^(.*)_LC(\d+)$", str(metric_name).strip(), flags=re.IGNORECASE)
    if m:
        return m.group(1).strip(), f"LC{m.group(2)}"
    return str(metric_name).strip(), None

def make_metric_key(family, metric):
    return f"{family} | {metric}"

def get_color_list(groups):
    return [embedment_colors.get(g, "gray") for g in groups]

def sort_groups_by_embedment(df, group_col="group"):
    df = df.copy()
    df["_sort"] = df[group_col].apply(extract_number)
    df = df.sort_values(["_sort", group_col]).drop(columns="_sort")
    return df

# =========================================================
# BUILD LONG TABLE
# =========================================================
def build_metrics_long():
    rows = []

    for path in discover_csvs(ORGANIZED_DIR):
        file_name = os.path.basename(path)
        family = guess_family(file_name)

        try:
            df = pd.read_csv(path)
            df = clean_columns(df)
        except Exception:
            continue

        group_col = find_group_col(df)
        if group_col is None:
            continue

        load_col = find_loadcase_col(df)

        df = df.copy()
        df["__group__"] = df[group_col].apply(normalize_embedment_label)

        if load_col is not None:
            df["__load_case__"] = df[load_col].apply(normalize_load_case)
        else:
            df["__load_case__"] = "ALL"

        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric_cols = [
            c for c in numeric_cols
            if c not in {
                "Node", "No.", "Depth (m)", "Depth",
                "Water Depth", "Embedment Depth", "Embedment Depth (m)", "Embedment",
                "__group__", "__load_case__"
            }
            and not c.lower().endswith("_num")
        ]

        for _, row in df.iterrows():
            for metric in numeric_cols:
                val = row[metric]
                if pd.isna(val):
                    continue

                base_metric, lc_from_metric = split_metric_and_lc(metric)
                final_lc = lc_from_metric if lc_from_metric is not None else row["__load_case__"]

                rows.append({
                    "family": family,
                    "source_file": file_name,
                    "group": row["__group__"],
                    "load_case": final_lc,
                    "metric": base_metric,
                    "metric_key": make_metric_key(family, base_metric),
                    "value": float(val)
                })

    return pd.DataFrame(rows)

metrics_long = build_metrics_long()

# =========================================================
# BASELINE FILTER
# =========================================================
baseline_df = metrics_long[
    (metrics_long["load_case"] == "LC1") | (metrics_long["load_case"] == "ALL")
].copy()

print("\nAvailable deflection_slope_pressure metrics in baseline_df:")
print(sorted(
    baseline_df.loc[
        baseline_df["family"] == "deflection_slope_pressure",
        "metric"
    ].dropna().unique().tolist()
))

print("\nAvailable fixity metrics in baseline_df:")
print(sorted(
    baseline_df.loc[
        baseline_df["family"] == "fixity",
        "metric"
    ].dropna().unique().tolist()
))

print("\nAvailable pile_settlement metrics in baseline_df:")
print(sorted(
    baseline_df.loc[
        baseline_df["family"] == "pile_settlement",
        "metric"
    ].dropna().unique().tolist()
))

# =========================================================
# EXTRACT CURATED METRICS
# =========================================================
curated_rows = []

for category, metrics in CURATED_METRICS.items():
    for item in metrics:
        fam = item["family"]
        met = item["metric"]
        direction = item["direction"]
        key = make_metric_key(fam, met)

        sub = baseline_df[
            (baseline_df["family"] == fam) &
            (baseline_df["metric"] == met)
        ].copy()

        if sub.empty:
            print(f"Missing metric: {key}")
            continue

        grouped = (
            sub.groupby("group", as_index=False)["value"]
            .mean()
        )

        if fam == "deflection_slope_pressure" and met == "Slope (rad)":
            grouped["value_deg"] = np.degrees(grouped["value"])

        grouped["category"] = category
        grouped["family"] = fam
        grouped["metric"] = met
        grouped["metric_key"] = key
        grouped["direction"] = direction

        curated_rows.append(grouped)

curated_values = pd.concat(curated_rows, ignore_index=True)
curated_values = sort_groups_by_embedment(curated_values, "group")
curated_values.to_csv(os.path.join(OUTPUT_DIR, "embedment_curated_metric_values.csv"), index=False)

# =========================================================
# RANK EACH METRIC
# =========================================================
rank_rows = []

for (category, metric_key, direction), sub in curated_values.groupby(["category", "metric_key", "direction"]):
    work = sub.copy()
    ascending = True if direction == "lower" else False
    work["metric_rank"] = work["value"].rank(ascending=ascending, method="dense")
    rank_rows.append(work)

curated_ranks = pd.concat(rank_rows, ignore_index=True)
curated_ranks = sort_groups_by_embedment(curated_ranks, "group")
curated_ranks.to_csv(os.path.join(OUTPUT_DIR, "embedment_curated_metric_ranks.csv"), index=False)

# =========================================================
# FACTOR SCORES
# =========================================================
factor_scores = (
    curated_ranks.groupby(["group", "category"], as_index=False)
    .agg(
        factor_rank_sum=("metric_rank", "sum"),
        metric_count=("metric_rank", "count"),
        factor_rank_mean=("metric_rank", "mean")
    )
)

factor_scores = sort_groups_by_embedment(factor_scores, "group")
factor_scores.to_csv(os.path.join(OUTPUT_DIR, "embedment_curated_factor_scores.csv"), index=False)

# =========================================================
# OVERALL RANKING
# =========================================================
overall_scores = (
    factor_scores.groupby("group", as_index=False)
    .agg(
        overall_rank_sum=("factor_rank_sum", "sum"),
        total_metric_count=("metric_count", "sum"),
        overall_mean_rank=("factor_rank_mean", "mean")
    )
)

overall_scores["overall_rank_position"] = overall_scores["overall_rank_sum"].rank(ascending=True, method="dense")
overall_scores["_sort"] = overall_scores["group"].apply(extract_number)
overall_scores = overall_scores.sort_values(["overall_rank_sum", "_sort", "group"]).drop(columns="_sort")

overall_scores.to_csv(os.path.join(OUTPUT_DIR, "embedment_curated_overall_ranking.csv"), index=False)

print("\nCurated overall embedment-depth ranking:")
print(overall_scores)

# =========================================================
# PLOTTING
# =========================================================
def plot_factor_bar(factor_scores, category, title, filename):
    sub = factor_scores[factor_scores["category"] == category].copy()
    sub["_sort"] = sub["group"].apply(extract_number)
    sub = sub.sort_values(["factor_rank_sum", "_sort", "group"]).drop(columns="_sort")

    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["factor_rank_sum"], color=colors)

    for i, v in enumerate(sub["factor_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Factor Rank Sum")
    plt.xlabel("Embedment Depth")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

def plot_overall_bar(overall_scores):
    sub = overall_scores.copy()
    colors = get_color_list(sub["group"])

    plt.figure(figsize=(10, 6))
    plt.bar(sub["group"], sub["overall_rank_sum"], color=colors)

    for i, v in enumerate(sub["overall_rank_sum"]):
        plt.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Overall Curated Rank Sum")
    plt.xlabel("Embedment Depth")
    plt.title("Overall Curated Embedment-Depth Ranking")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_embedment_depth_ranking.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, "overall_curated_embedment_depth_ranking.pdf"), bbox_inches="tight")
    plt.show()
    plt.close()

plot_factor_bar(
    factor_scores,
    "lateral_performance",
    "Embedment-Depth Ranking: Lateral Performance",
    "lateral_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "axial_settlement_performance",
    "Embedment-Depth Ranking: Axial / Settlement Performance",
    "axial_settlement_performance_ranking"
)

plot_factor_bar(
    factor_scores,
    "capacity_resistance",
    "Embedment-Depth Ranking: Capacity / Resistance",
    "capacity_resistance_ranking"
)

plot_factor_bar(
    factor_scores,
    "variability_risk",
    "Embedment-Depth Ranking: Variability / Risk",
    "variability_risk_ranking"
)

plot_overall_bar(overall_scores)

# =========================================================
# OPTIONAL: MAKE A WIDE TABLE FOR READING
# =========================================================
metric_value_table = curated_values.pivot_table(
    index="group",
    columns="metric_key",
    values="value",
    aggfunc="mean"
).reset_index()

slope_rows = curated_values[
    (curated_values["family"] == "deflection_slope_pressure") &
    (curated_values["metric"] == "Slope (rad)")
].copy()

if not slope_rows.empty:
    slope_rows["value_deg"] = np.degrees(slope_rows["value"])

    slope_deg_table = (
        slope_rows[["group", "value_deg"]]
        .drop_duplicates()
    )

    metric_value_table = metric_value_table.merge(slope_deg_table, on="group", how="left")
    metric_value_table = metric_value_table.rename(
        columns={"value_deg": "deflection_slope_pressure | Slope (deg)"}
    )
else:
    print("No curated slope metric found, so degree conversion was skipped.")

metric_value_table = sort_groups_by_embedment(metric_value_table, "group")
metric_value_table.to_csv(
    os.path.join(OUTPUT_DIR, "embedment_curated_metric_values_wide.csv"),
    index=False
)

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy import stats
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_factor_ranking_stats_FIXED"

sediment_file = os.path.join(BASE_DIR, "sediment_curated_ranking", "sediment_curated_metric_values.csv")
water_file = os.path.join(BASE_DIR, "water_depth_curated_ranking", "water_depth_curated_metric_values.csv")
embedment_file = os.path.join(BASE_DIR, "embedment_depth_curated_ranking", "embedment_curated_metric_values.csv")

output_dir = os.path.join(BASE_DIR, "factor_importance_stats")
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# LOAD DATA
# =========================================================
sed = pd.read_csv(sediment_file)
water = pd.read_csv(water_file)
embed = pd.read_csv(embedment_file)

sed["Factor"] = "Sediment"
water["Factor"] = "Water Depth"
embed["Factor"] = "Embedment Depth"

df = pd.concat([sed, water, embed], ignore_index=True)

# =========================================================
# CLEAN
# =========================================================
df = df.dropna(subset=["value", "metric"])

# =========================================================
# STAT FUNCTIONS
# =========================================================
def compute_eta_squared(groups):
    all_vals = np.concatenate(groups)
    grand_mean = np.mean(all_vals)

    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum((all_vals - grand_mean)**2)

    return ss_between / ss_total if ss_total != 0 else np.nan

# =========================================================
# MAIN STATS LOOP
# =========================================================
results = []

for metric in sorted(df["metric"].unique()):

    sub = df[df["metric"] == metric]

    grouped = [g["value"].values for _, g in sub.groupby("Factor")]

    if len(grouped) < 2:
        continue

    try:
        f_stat, p_anova = stats.f_oneway(*grouped)
    except:
        p_anova = np.nan

    try:
        h_stat, p_kw = stats.kruskal(*grouped)
    except:
        p_kw = np.nan

    try:
        stat, p_levene = stats.levene(*grouped)
    except:
        p_levene = np.nan

    eta = compute_eta_squared(grouped)

    cv = np.std(sub["value"]) / np.mean(sub["value"]) if np.mean(sub["value"]) != 0 else np.nan

    results.append({
        "Metric": metric,
        "ANOVA p-value": p_anova,
        "Kruskal p-value": p_kw,
        "Levene p-value": p_levene,
        "Eta-squared": eta,
        "CV": cv
    })

stats_df = pd.DataFrame(results).sort_values("Eta-squared", ascending=False)

stats_df.to_csv(os.path.join(output_dir, "factor_importance_statistics.csv"), index=False)

print("\n=== STATISTICS SUMMARY ===")
print(stats_df)

# =========================================================
# AGGREGATE FACTOR IMPORTANCE
# =========================================================
factor_effects = df.groupby("Factor").agg(
    mean_value=("value", "mean"),
    std_value=("value", "std")
).reset_index()

# =========================================================
# COLORS
# =========================================================
color_map = {
    "Sediment": "green",
    "Water Depth": "purple",
    "Embedment Depth": "orange"
}

colors = [color_map[f] for f in factor_effects["Factor"]]

# =========================================================
# BAR PLOT (MEAN RESPONSE)
# =========================================================
plt.figure()
plt.bar(factor_effects["Factor"], factor_effects["mean_value"], color=colors)

for i, v in enumerate(factor_effects["mean_value"]):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom")

plt.ylabel("Mean Metric Value")
plt.title("Average Influence of Each Factor on Pile Behavior")
plt.grid(axis="y", alpha=0.3)

plt.savefig(os.path.join(output_dir, "factor_mean_comparison.png"), dpi=300)
plt.show()

# =========================================================
# EFFECT SIZE BAR PLOT
# =========================================================
plt.figure()
plt.bar(stats_df["Metric"], stats_df["Eta-squared"], color="gray")

plt.xticks(rotation=90)
plt.ylabel("Eta-squared (Effect Size)")
plt.title("Effect Size of Each Metric")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "eta_squared_by_metric.png"), dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_factor_ranking_stats_FIXED"

sed_file = os.path.join(BASE_DIR, "sediment_true_statistics", "sediment_metric_significance_and_effect_sizes.csv")
water_file = os.path.join(BASE_DIR, "water_true_statistics", "water_metric_stats.csv")
embed_file = os.path.join(BASE_DIR, "embedment_true_statistics", "embedment_metric_stats.csv")

output_dir = os.path.join(BASE_DIR, "corrected_factor_control")
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# LOAD
# =========================================================
sed = pd.read_csv(sed_file)
water = pd.read_csv(water_file)
embed = pd.read_csv(embed_file)

sed["Factor"] = "Sediment"
water["Factor"] = "Water Depth"
embed["Factor"] = "Embedment Depth"

df = pd.concat([sed, water, embed], ignore_index=True)

# =========================================================
# CLEAN
# =========================================================
df = df.dropna(subset=["Metric", "Eta-squared"])

# =========================================================
# PIVOT (KEY STEP)
# =========================================================
pivot = df.pivot_table(
    index="Metric",
    columns="Factor",
    values="Eta-squared",
    aggfunc="mean"
).reset_index()

pivot = pivot.fillna(0)

# =========================================================
# DETERMINE DOMINANT FACTOR
# =========================================================
pivot["Dominant Factor"] = pivot[
    ["Sediment", "Water Depth", "Embedment Depth"]
].idxmax(axis=1)

pivot["Max Effect Size"] = pivot[
    ["Sediment", "Water Depth", "Embedment Depth"]
].max(axis=1)

# =========================================================
# SAVE
# =========================================================
pivot.to_csv(os.path.join(output_dir, "factor_control_by_metric.csv"), index=False)

print("\n=== WHICH FACTOR CONTROLS EACH METRIC ===")
print(pivot.sort_values("Max Effect Size", ascending=False))

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import stats

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Sediment Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "sediment_true_statistics")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# CURATED METRICS TO TEST
# =========================================================
CURATED_METRICS = {
    "lateral_performance": [
        ("deflection_slope_pressure", "Max Deflection Magnitude (mm)"),
        ("deflection_slope_pressure", "Slope (rad)"),
        ("deflection_slope_pressure", "First Inflection Depth (m)"),
        ("fixity", "Mean Fixity Depth (m)")
    ],
    "axial_settlement_performance": [
        ("pile_settlement", "Max Head (mm)"),
        ("pile_settlement", "Pile shaft settlement (mm)")
    ],
    "capacity_resistance": [
        ("pile_capacity", "Ultimate total capacity (kN)"),
        ("unit_resistance", "mean_unit_shaft"),
        ("unit_resistance", "mean_unit_base"),
        ("p_y", "initial_stiffness"),
        ("q_z", "Q_max"),
        ("t_z", "t_max")
    ],
    "variability_risk": [
        ("other", "CV"),
        ("unit_resistance", "cv_unit_shaft"),
        ("unit_resistance", "cv_unit_base")
    ]
}

# flatten
TARGETS = {(fam, met): cat for cat, vals in CURATED_METRICS.items() for fam, met in vals}

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                out.append(os.path.join(root, f))
    return sorted(out)

def guess_family(file_name):
    f = file_name.lower()
    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "pile_capacity" in f:
        return "pile_capacity"
    if "unit_resistance" in f:
        return "unit_resistance"
    if "pile_settlement" in f:
        return "pile_settlement"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "fixity" in f:
        return "fixity"
    if "other" in f or "comparison" in f or "summary" in f:
        return "other"
    return None

def normalize_load_case(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.search(r"LC\s*-?\s*(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"LC{m.group(1)}"
    m2 = re.fullmatch(r"(\d+)", s)
    if m2:
        return f"LC{m2.group(1)}"
    return s

def split_metric_and_lc(metric_name):
    m = re.search(r"^(.*)_LC(\d+)$", str(metric_name).strip(), flags=re.IGNORECASE)
    if m:
        return m.group(1).strip(), f"LC{m.group(2)}"
    return str(metric_name).strip(), None

def find_group_col(df):
    for c in ["Sediment Type", "Group", "Parsed Group"]:
        if c in df.columns:
            return c
    return None

def find_loadcase_col(df):
    for c in ["Load Case", "Load-case", "LoadCase", "Load case"]:
        if c in df.columns:
            return c
    return None

def eta_squared_anova(groups):
    all_vals = np.concatenate(groups)
    grand_mean = np.mean(all_vals)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in groups)
    ss_total = np.sum((all_vals - grand_mean) ** 2)
    return ss_between / ss_total if ss_total != 0 else np.nan

def epsilon_squared_kruskal(H, n, k):
    # nonparametric effect size
    if n <= k:
        return np.nan
    return (H - k + 1) / (n - k)

# =========================================================
# BUILD LESS-AGGREGATED LONG TABLE
# =========================================================
rows = []

for path in discover_csvs(ORGANIZED_DIR):
    file_name = os.path.basename(path)
    family = guess_family(file_name)
    if family is None:
        continue

    try:
        df = pd.read_csv(path)
        df = clean_columns(df)
    except Exception:
        continue

    group_col = find_group_col(df)
    if group_col is None:
        continue

    load_col = find_loadcase_col(df)

    df = df.copy()
    df["__group__"] = df[group_col].astype(str).str.strip()

    if load_col is not None:
        df["__load_case__"] = df[load_col].apply(normalize_load_case)
    else:
        df["__load_case__"] = "ALL"

    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    numeric_cols = [
        c for c in numeric_cols
        if c not in {"Node", "No.", "Depth (m)", "Depth", "Embedment Depth", "Water Depth",
                     "__group__", "__load_case__"}
        and not c.lower().endswith("_num")
    ]

    for _, row in df.iterrows():
        for metric in numeric_cols:
            val = row[metric]
            if pd.isna(val):
                continue

            base_metric, lc_from_metric = split_metric_and_lc(metric)
            final_lc = lc_from_metric if lc_from_metric is not None else row["__load_case__"]

            if (family, base_metric) not in TARGETS:
                continue

            rows.append({
                "category": TARGETS[(family, base_metric)],
                "family": family,
                "metric": base_metric,
                "group": row["__group__"],
                "load_case": final_lc,
                "value": float(val)
            })

long_df = pd.DataFrame(rows)
long_df.to_csv(os.path.join(OUTPUT_DIR, "sediment_curated_replicated_long.csv"), index=False)

print("Long df shape:", long_df.shape)
print(long_df.head())

# =========================================================
# OPTIONAL: baseline only OR all replicated values
# Choose one:
# 1) LC1/ALL only
# 2) all rows (replication across LC etc.)
# =========================================================
USE_BASELINE_ONLY = False

if USE_BASELINE_ONLY:
    stat_df = long_df[(long_df["load_case"] == "LC1") | (long_df["load_case"] == "ALL")].copy()
else:
    stat_df = long_df.copy()

# =========================================================
# STATS PER METRIC
# =========================================================
results = []

for (category, family, metric), sub in stat_df.groupby(["category", "family", "metric"]):
    grouped = []
    group_names = []

    for g, gsub in sub.groupby("group"):
        vals = pd.to_numeric(gsub["value"], errors="coerce").dropna().values
        if len(vals) >= 2:
            grouped.append(vals)
            group_names.append(g)

    if len(grouped) < 2:
        continue

    try:
        f_stat, p_anova = stats.f_oneway(*grouped)
    except Exception:
        f_stat, p_anova = np.nan, np.nan

    try:
        h_stat, p_kw = stats.kruskal(*grouped)
    except Exception:
        h_stat, p_kw = np.nan, np.nan

    try:
        lev_stat, p_levene = stats.levene(*grouped)
    except Exception:
        lev_stat, p_levene = np.nan, np.nan

    n_total = sum(len(g) for g in grouped)
    k = len(grouped)

    eta_sq = eta_squared_anova(grouped)
    eps_sq = epsilon_squared_kruskal(h_stat, n_total, k) if pd.notna(h_stat) else np.nan

    all_vals = pd.to_numeric(sub["value"], errors="coerce").dropna().values
    cv = np.std(all_vals) / np.mean(all_vals) if np.mean(all_vals) != 0 else np.nan

    results.append({
        "Category": category,
        "Family": family,
        "Metric": metric,
        "N sediment groups": k,
        "Total N": n_total,
        "ANOVA F": f_stat,
        "ANOVA p-value": p_anova,
        "Kruskal H": h_stat,
        "Kruskal p-value": p_kw,
        "Levene W": lev_stat,
        "Levene p-value": p_levene,
        "Eta-squared": eta_sq,
        "Epsilon-squared": eps_sq,
        "CV": cv
    })

results_df = pd.DataFrame(results)

if results_df.empty:
    print("No valid sediment statistics were produced.")
else:
    results_df = results_df.sort_values(["Category", "Eta-squared"], ascending=[True, False])
    results_df.to_csv(os.path.join(OUTPUT_DIR, "sediment_metric_significance_and_effect_sizes.csv"), index=False)

    print("\n=== TRUE SEDIMENT STATISTICS ===")
    print(results_df)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_factor_ranking_stats_FIXED"
input_file = os.path.join(BASE_DIR, "sediment_true_statistics", "sediment_metric_significance_and_effect_sizes.csv")
output_dir = os.path.join(BASE_DIR, "sediment_true_statistics")

df = pd.read_csv(input_file)

ranked_df = df.sort_values("Eta-squared", ascending=False).reset_index(drop=True)
ranked_df.to_csv(os.path.join(output_dir, "sediment_metrics_ranked_by_influence.csv"), index=False)

ranked_df = ranked_df[ranked_df["Metric"] != "First Inflection Depth (m)"].copy()
ranked_df = ranked_df.sort_values("Eta-squared", ascending=False).reset_index(drop=True)



print("\n=== METRICS RANKED BY HOW MUCH SEDIMENT CHANGES THEM ===")
print(ranked_df[[
    "Category", "Family", "Metric",
    "ANOVA p-value", "Kruskal p-value",
    "Eta-squared", "Epsilon-squared", "CV"
]])

plt.figure(figsize=(10, 7))
plt.barh(ranked_df["Metric"], ranked_df["Eta-squared"], color="green")
plt.xlabel("Eta-squared")
plt.ylabel("Metric")
plt.title("How Strongly Sediment Type Influences Each Metric")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "sediment_metric_influence_eta_squared.png"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(output_dir, "sediment_metric_influence_eta_squared.pdf"), bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import stats

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Water Depth Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "water_true_statistics")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# CURATED METRICS
# =========================================================
CURATED_METRICS = {
    "lateral_performance": [
        ("deflection_slope_pressure", "Max Deflection Magnitude (mm)"),
        ("deflection_slope_pressure", "Slope (rad)"),
        ("deflection_slope_pressure", "First Inflection Depth (m)"),
        ("fixity", "Mean Fixity Depth (m)")
    ],
    "axial_settlement_performance": [
        ("pile_settlement", "Max Head (mm)"),
        ("pile_settlement", "Pile shaft settlement (mm)")
    ],
    "capacity_resistance": [
        ("pile_capacity", "Ultimate total capacity (kN)"),
        ("unit_resistance", "mean_unit_shaft"),
        ("unit_resistance", "mean_unit_base"),
        ("p_y", "initial_stiffness"),
        ("q_z", "Q_max"),
        ("t_z", "t_max")
    ],
    "variability_risk": [
        ("other", "CV"),
        ("unit_resistance", "cv_unit_shaft"),
        ("unit_resistance", "cv_unit_base")
    ]
}

TARGETS = {(fam, met): cat for cat, vals in CURATED_METRICS.items() for fam, met in vals}

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                out.append(os.path.join(root, f))
    return sorted(out)

def guess_family(name):
    f = name.lower()
    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "capacity" in f:
        return "pile_capacity"
    if "unit" in f:
        return "unit_resistance"
    if "settlement" in f:
        return "pile_settlement"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "fixity" in f:
        return "fixity"
    return "other"

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def eta_squared(groups):
    all_vals = np.concatenate(groups)
    grand_mean = np.mean(all_vals)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in groups)
    ss_total = np.sum((all_vals - grand_mean) ** 2)
    return ss_between / ss_total if ss_total != 0 else np.nan

def epsilon_squared(H, n, k):
    return (H - k + 1) / (n - k) if n > k else np.nan

def find_first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# =========================================================
# BUILD LONG DATA FROM ORGANIZED FOLDER
# =========================================================
rows = []

for path in discover_csvs(ORGANIZED_DIR):
    try:
        df = pd.read_csv(path)
        df = clean_columns(df)
    except Exception:
        continue

    family = guess_family(os.path.basename(path))

    group_col = None
    for c in ["Water Depth", "WaterDepth", "Group"]:
        if c in df.columns:
            group_col = c
            break
    if group_col is None:
        continue

    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

    for _, r in df.iterrows():
        group = extract_number(r[group_col])

        if pd.isna(group):
            continue

        for col in numeric_cols:
            if col.lower() in ["depth", "depth (m)", "water depth numeric"]:
                continue

            val = r[col]
            if pd.isna(val):
                continue

            metric = col
            if (family, metric) not in TARGETS:
                continue

            rows.append({
                "category": TARGETS[(family, metric)],
                "family": family,
                "metric": metric,
                "group": group,
                "value": float(val),
                "source": "organized"
            })

long_df = pd.DataFrame(rows)

print("Initial long_df shape from organized folder:", long_df.shape)

# =========================================================
# ADD INFLECTION + DEFLECTION + FIXITY MANUALLY
# =========================================================
extra_rows = []

# --- INFLECTION DEPTH ---
inflection_file = os.path.join(
    BASE_DIR,
    "Water Depth Comparison",
    "shear_bending_plots",
    "first_inflection_depth_by_water_depth.csv"
)

if os.path.exists(inflection_file):
    inf_df = pd.read_csv(inflection_file)
    inf_df = clean_columns(inf_df)

    wd_col = find_first_existing_column(inf_df, ["Water Depth", "WaterDepth"])
    inf_col = find_first_existing_column(inf_df, ["First Inflection Depth (m)"])

    if wd_col and inf_col:
        for _, row in inf_df.iterrows():
            wd = extract_number(row[wd_col])
            val = pd.to_numeric(pd.Series([row[inf_col]]), errors="coerce").iloc[0]

            if pd.notna(wd) and pd.notna(val):
                extra_rows.append({
                    "category": "lateral_performance",
                    "family": "deflection_slope_pressure",
                    "metric": "First Inflection Depth (m)",
                    "group": wd,
                    "value": float(val),
                    "source": "manual_inflection"
                })
        print("Added First Inflection Depth rows from:", inflection_file)
    else:
        print("Inflection file found, but required columns were missing.")
else:
    print("Inflection file not found:", inflection_file)

# --- DEFLECTION (MAX) ---
defl_file = os.path.join(
    BASE_DIR,
    "Water Depth Comparison",
    "deflection_slope_pressure_plots",
    "deflection_slope_pressure_stats_by_water_depth.csv"
)

if os.path.exists(defl_file):
    defl_df = pd.read_csv(defl_file)
    defl_df = clean_columns(defl_df)

    wd_col = find_first_existing_column(defl_df, ["Water Depth", "WaterDepth"])
    defl_col = find_first_existing_column(
        defl_df,
        ["Max Deflection Magnitude (mm)", "Max Deflection (mm)", "Max Deflection"]
    )

    if wd_col and defl_col:
        for _, row in defl_df.iterrows():
            wd = extract_number(row[wd_col])
            val = pd.to_numeric(pd.Series([row[defl_col]]), errors="coerce").iloc[0]

            if pd.notna(wd) and pd.notna(val):
                extra_rows.append({
                    "category": "lateral_performance",
                    "family": "deflection_slope_pressure",
                    "metric": "Max Deflection Magnitude (mm)",
                    "group": wd,
                    "value": float(abs(val)),
                    "source": "manual_deflection"
                })
        print("Added Max Deflection Magnitude rows from:", defl_file)
    else:
        print("Deflection stats file found, but required columns were missing.")
else:
    print("Deflection stats file not found:", defl_file)

# --- FIXITY DEPTH ---
# tries multiple possible filenames
fixity_candidates = [
    os.path.join(BASE_DIR, "Water Depth Comparison", "deflection_slope_pressure_plots", "fixity_depth_by_water_depth.csv"),
    os.path.join(BASE_DIR, "Water Depth Comparison", "deflection_plots", "fixity_depth_by_water_depth.csv"),
    os.path.join(BASE_DIR, "Water Depth Comparison", "fixity_depth_plots", "water_depth_fixity_by_loadcase.csv"),
    os.path.join(BASE_DIR, "Water Depth Comparison", "fixity_depth_plots", "water_depth_fixity_summary_mean_range.csv"),
]

fixity_file = None
for fp in fixity_candidates:
    if os.path.exists(fp):
        fixity_file = fp
        break

if fixity_file is not None:
    fix_df = pd.read_csv(fixity_file)
    fix_df = clean_columns(fix_df)

    wd_col = find_first_existing_column(fix_df, ["Water Depth", "WaterDepth", "Group"])
    fix_col = find_first_existing_column(
        fix_df,
        ["Mean Fixity Depth (m)", "Fixity Depth (m)", "Mean Fixity Depth"]
    )

    if wd_col and fix_col:
        for _, row in fix_df.iterrows():
            wd = extract_number(row[wd_col])
            val = pd.to_numeric(pd.Series([row[fix_col]]), errors="coerce").iloc[0]

            if pd.notna(wd) and pd.notna(val):
                extra_rows.append({
                    "category": "lateral_performance",
                    "family": "fixity",
                    "metric": "Mean Fixity Depth (m)",
                    "group": wd,
                    "value": float(val),
                    "source": "manual_fixity"
                })
        print("Added Fixity rows from:", fixity_file)
    else:
        print("Fixity file found, but required columns were missing.")
else:
    print("No fixity file found in expected locations.")

# combine
if extra_rows:
    extra_df = pd.DataFrame(extra_rows)
    long_df = pd.concat([long_df, extra_df], ignore_index=True)

print("Final long_df shape after manual additions:", long_df.shape)

# save the replicated long table
long_df.to_csv(os.path.join(OUTPUT_DIR, "water_curated_replicated_long.csv"), index=False)

# =========================================================
# STATS
# =========================================================
results = []

for (cat, fam, metric), sub in long_df.groupby(["category", "family", "metric"]):
    # require at least 2 repeated values per water depth for inferential stats
    grouped = [g["value"].dropna().values for _, g in sub.groupby("group") if len(g) >= 2]

    if len(grouped) < 2:
        print(f"Skipping {metric}: fewer than 2 water-depth groups with replication")
        continue

    try:
        f, p_a = stats.f_oneway(*grouped)
    except Exception:
        f, p_a = np.nan, np.nan

    try:
        h, p_k = stats.kruskal(*grouped)
    except Exception:
        h, p_k = np.nan, np.nan

    n = sum(len(g) for g in grouped)
    k = len(grouped)

    results.append({
        "Category": cat,
        "Family": fam,
        "Metric": metric,
        "N water-depth groups": k,
        "Total N": n,
        "ANOVA p-value": p_a,
        "Kruskal p-value": p_k,
        "Eta-squared": eta_squared(grouped),
        "Epsilon-squared": epsilon_squared(h, n, k) if pd.notna(h) else np.nan
    })

results_df = pd.DataFrame(results)

if results_df.empty:
    print("\nNo valid water-depth statistics were produced.")
else:
    results_df = results_df.sort_values("Eta-squared", ascending=False)
    results_df.to_csv(os.path.join(OUTPUT_DIR, "water_metric_stats.csv"), index=False)

    print("\n=== WATER DEPTH EFFECT ===")
    print(results_df)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import stats

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"
ORGANIZED_DIR = os.path.join(BASE_DIR, "Embedment Depth Comparison", "organized")
OUTPUT_DIR = os.path.join(BASE_DIR, "master_factor_ranking_stats_FIXED", "embedment_true_statistics")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# CURATED METRICS
# =========================================================
CURATED_METRICS = {
    "lateral_performance": [
        ("deflection_slope_pressure", "Max Deflection Magnitude (mm)"),
        ("deflection_slope_pressure", "Slope (rad)"),
        ("deflection_slope_pressure", "First Inflection Depth (m)"),
        ("fixity", "Mean Fixity Depth (m)")
    ],
    "axial_settlement_performance": [
        ("pile_settlement", "Max Head (mm)"),
        ("pile_settlement", "Pile shaft settlement (mm)")
    ],
    "capacity_resistance": [
        ("pile_capacity", "Ultimate total capacity (kN)"),
        ("unit_resistance", "mean_unit_shaft"),
        ("unit_resistance", "mean_unit_base"),
        ("p_y", "initial_stiffness"),
        ("q_z", "Q_max"),
        ("t_z", "t_max")
    ],
    "variability_risk": [
        ("other", "CV"),
        ("unit_resistance", "cv_unit_shaft"),
        ("unit_resistance", "cv_unit_base")
    ]
}

TARGETS = {(fam, met): cat for cat, vals in CURATED_METRICS.items() for fam, met in vals}

# =========================================================
# HELPERS
# =========================================================
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace('"', "", regex=False)
    )
    return df

def discover_csvs(folder):
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".csv"):
                out.append(os.path.join(root, f))
    return sorted(out)

def guess_family(name):
    f = name.lower()
    if "t_z" in f or "t-z" in f or "tz" in f:
        return "t_z"
    if "q_z" in f or "q-z" in f or "qz" in f:
        return "q_z"
    if "p_y" in f or "p-y" in f or "py" in f:
        return "p_y"
    if "capacity" in f:
        return "pile_capacity"
    if "unit" in f:
        return "unit_resistance"
    if "settlement" in f:
        return "pile_settlement"
    if "deflection" in f or "slope" in f or "pressure" in f:
        return "deflection_slope_pressure"
    if "fixity" in f:
        return "fixity"
    return "other"

def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def eta_squared(groups):
    all_vals = np.concatenate(groups)
    grand_mean = np.mean(all_vals)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in groups)
    ss_total = np.sum((all_vals - grand_mean) ** 2)
    return ss_between / ss_total if ss_total != 0 else np.nan

def epsilon_squared(H, n, k):
    return (H - k + 1) / (n - k) if n > k else np.nan

def find_first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# =========================================================
# BUILD LONG DATA FROM ORGANIZED FOLDER
# =========================================================
rows = []

for path in discover_csvs(ORGANIZED_DIR):
    try:
        df = pd.read_csv(path)
        df = clean_columns(df)
    except Exception:
        continue

    family = guess_family(os.path.basename(path))

    group_col = None
    for c in ["Embedment Depth", "Embedment", "Group"]:
        if c in df.columns:
            group_col = c
            break
    if group_col is None:
        continue

    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

    for _, r in df.iterrows():
        group = extract_number(r[group_col])

        if pd.isna(group):
            continue

        for col in numeric_cols:
            if col.lower() in ["depth", "depth (m)", "embedment depth numeric"]:
                continue

            val = r[col]
            if pd.isna(val):
                continue

            metric = col
            if (family, metric) not in TARGETS:
                continue

            rows.append({
                "category": TARGETS[(family, metric)],
                "family": family,
                "metric": metric,
                "group": group,
                "value": float(val),
                "source": "organized"
            })

long_df = pd.DataFrame(rows)

print("Initial long_df shape from organized folder:", long_df.shape)

# =========================================================
# ADD INFLECTION + DEFLECTION + FIXITY MANUALLY
# =========================================================
extra_rows = []

# --- INFLECTION DEPTH ---
inflection_candidates = [
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "shear_bending_plots", "first_inflection_depth_by_embedment_depth.csv"),
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "shear_bending_plots", "first_inflection_depth_by_embedment.csv"),
]

inflection_file = None
for fp in inflection_candidates:
    if os.path.exists(fp):
        inflection_file = fp
        break

if inflection_file is not None:
    inf_df = pd.read_csv(inflection_file)
    inf_df = clean_columns(inf_df)

    emb_col = find_first_existing_column(inf_df, ["Embedment Depth", "Embedment", "Group"])
    inf_col = find_first_existing_column(inf_df, ["First Inflection Depth (m)"])

    if emb_col and inf_col:
        for _, row in inf_df.iterrows():
            emb = extract_number(row[emb_col])
            val = pd.to_numeric(pd.Series([row[inf_col]]), errors="coerce").iloc[0]

            if pd.notna(emb) and pd.notna(val):
                extra_rows.append({
                    "category": "lateral_performance",
                    "family": "deflection_slope_pressure",
                    "metric": "First Inflection Depth (m)",
                    "group": emb,
                    "value": float(val),
                    "source": "manual_inflection"
                })
        print("Added First Inflection Depth rows from:", inflection_file)
    else:
        print("Inflection file found, but required columns were missing.")
else:
    print("No inflection file found in expected locations.")

# --- DEFLECTION (MAX) ---
defl_candidates = [
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "deflection_slope_pressure_plots", "deflection_slope_pressure_stats_by_embedment_depth.csv"),
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "deflection_plots", "deflection_stats_by_embedment_depth.csv"),
]

defl_file = None
for fp in defl_candidates:
    if os.path.exists(fp):
        defl_file = fp
        break

if defl_file is not None:
    defl_df = pd.read_csv(defl_file)
    defl_df = clean_columns(defl_df)

    emb_col = find_first_existing_column(defl_df, ["Embedment Depth", "Embedment", "Group"])
    defl_col = find_first_existing_column(
        defl_df,
        ["Max Deflection Magnitude (mm)", "Max Deflection (mm)", "Max Deflection"]
    )

    if emb_col and defl_col:
        for _, row in defl_df.iterrows():
            emb = extract_number(row[emb_col])
            val = pd.to_numeric(pd.Series([row[defl_col]]), errors="coerce").iloc[0]

            if pd.notna(emb) and pd.notna(val):
                extra_rows.append({
                    "category": "lateral_performance",
                    "family": "deflection_slope_pressure",
                    "metric": "Max Deflection Magnitude (mm)",
                    "group": emb,
                    "value": float(abs(val)),
                    "source": "manual_deflection"
                })
        print("Added Max Deflection Magnitude rows from:", defl_file)
    else:
        print("Deflection stats file found, but required columns were missing.")
else:
    print("No deflection stats file found in expected locations.")

# --- FIXITY DEPTH ---
fixity_candidates = [
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "deflection_slope_pressure_plots", "fixity_depth_by_embedment_depth.csv"),
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "deflection_plots", "fixity_depth_by_embedment_depth.csv"),
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "fixity_depth_plots", "embedment_depth_fixity_by_loadcase.csv"),
    os.path.join(BASE_DIR, "Embedment Depth Comparison", "fixity_depth_plots", "embedment_depth_fixity_summary_mean_range.csv"),
]

fixity_file = None
for fp in fixity_candidates:
    if os.path.exists(fp):
        fixity_file = fp
        break

if fixity_file is not None:
    fix_df = pd.read_csv(fixity_file)
    fix_df = clean_columns(fix_df)

    emb_col = find_first_existing_column(fix_df, ["Embedment Depth", "Embedment", "Group"])
    fix_col = find_first_existing_column(
        fix_df,
        ["Mean Fixity Depth (m)", "Fixity Depth (m)", "Mean Fixity Depth"]
    )

    if emb_col and fix_col:
        for _, row in fix_df.iterrows():
            emb = extract_number(row[emb_col])
            val = pd.to_numeric(pd.Series([row[fix_col]]), errors="coerce").iloc[0]

            if pd.notna(emb) and pd.notna(val):
                extra_rows.append({
                    "category": "lateral_performance",
                    "family": "fixity",
                    "metric": "Mean Fixity Depth (m)",
                    "group": emb,
                    "value": float(val),
                    "source": "manual_fixity"
                })
        print("Added Fixity rows from:", fixity_file)
    else:
        print("Fixity file found, but required columns were missing.")
else:
    print("No fixity file found in expected locations.")

# combine
if extra_rows:
    extra_df = pd.DataFrame(extra_rows)
    long_df = pd.concat([long_df, extra_df], ignore_index=True)

print("Final long_df shape after manual additions:", long_df.shape)

# save the replicated long table
long_df.to_csv(os.path.join(OUTPUT_DIR, "embedment_curated_replicated_long.csv"), index=False)

# =========================================================
# STATS
# =========================================================
results = []

for (cat, fam, metric), sub in long_df.groupby(["category", "family", "metric"]):
    grouped = [g["value"].dropna().values for _, g in sub.groupby("group") if len(g) >= 2]

    if len(grouped) < 2:
        print(f"Skipping {metric}: fewer than 2 embedment-depth groups with replication")
        continue

    try:
        f, p_a = stats.f_oneway(*grouped)
    except Exception:
        f, p_a = np.nan, np.nan

    try:
        h, p_k = stats.kruskal(*grouped)
    except Exception:
        h, p_k = np.nan, np.nan

    n = sum(len(g) for g in grouped)
    k = len(grouped)

    results.append({
        "Category": cat,
        "Family": fam,
        "Metric": metric,
        "N embedment-depth groups": k,
        "Total N": n,
        "ANOVA p-value": p_a,
        "Kruskal p-value": p_k,
        "Eta-squared": eta_squared(grouped),
        "Epsilon-squared": epsilon_squared(h, n, k) if pd.notna(h) else np.nan
    })

results_df = pd.DataFrame(results)

if results_df.empty:
    print("\nNo valid embedment-depth statistics were produced.")
else:
    results_df = results_df.sort_values("Eta-squared", ascending=False)
    results_df.to_csv(os.path.join(OUTPUT_DIR, "embedment_metric_stats.csv"), index=False)

    print("\n=== EMBEDMENT DEPTH EFFECT ===")
    print(results_df)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_factor_ranking_stats_FIXED"

water_long_file = os.path.join(BASE_DIR, "water_true_statistics", "water_curated_replicated_long.csv")
water_stats_file = os.path.join(BASE_DIR, "water_true_statistics", "water_metric_stats.csv")

embed_long_file = os.path.join(BASE_DIR, "embedment_true_statistics", "embedment_curated_replicated_long.csv")
embed_stats_file = os.path.join(BASE_DIR, "embedment_true_statistics", "embedment_metric_stats.csv")

output_dir = os.path.join(BASE_DIR, "descriptive_stats_for_skipped_metrics")
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================
def extract_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

def load_existing_metrics(stats_file):
    if os.path.exists(stats_file):
        stats_df = pd.read_csv(stats_file)
        if "Metric" in stats_df.columns:
            return set(stats_df["Metric"].dropna().astype(str).tolist())
    return set()

def summarize_skipped_metrics(long_df, tested_metrics, factor_label):
    # metrics that exist in long_df but were not successfully tested inferentially
    all_metrics = set(long_df["metric"].dropna().astype(str).tolist())
    skipped_metrics = sorted(all_metrics - tested_metrics)

    print(f"\n{factor_label} skipped metrics:")
    print(skipped_metrics)

    summary_rows = []

    for metric in skipped_metrics:
        sub = long_df[long_df["metric"] == metric].copy()
        if sub.empty:
            continue

        # summarize by group
        group_summary = (
            sub.groupby("group", as_index=False)["value"]
            .agg(["count", "mean", "std", "min", "max"])
            .reset_index()
        )

        group_summary["range"] = group_summary["max"] - group_summary["min"]
        group_summary["cv"] = np.where(
            group_summary["mean"] != 0,
            group_summary["std"] / group_summary["mean"],
            np.nan
        )

        group_summary["factor"] = factor_label
        group_summary["metric"] = metric

        # sort numeric groups
        group_summary["_sort"] = group_summary["group"].apply(extract_number)
        group_summary = group_summary.sort_values(["_sort", "group"]).drop(columns="_sort")

        # overall descriptive comparison across groups
        valid_means = group_summary["mean"].dropna()
        if len(valid_means) >= 2:
            lowest_mean = valid_means.min()
            highest_mean = valid_means.max()

            if lowest_mean != 0:
                pct_diff = ((highest_mean - lowest_mean) / abs(lowest_mean)) * 100
            else:
                pct_diff = np.nan
        else:
            pct_diff = np.nan

        for _, row in group_summary.iterrows():
            summary_rows.append({
                "Factor": factor_label,
                "Metric": metric,
                "Group": row["group"],
                "Count": row["count"],
                "Mean": row["mean"],
                "Std Dev": row["std"],
                "Min": row["min"],
                "Max": row["max"],
                "Range": row["range"],
                "CV": row["cv"],
                "Percent Difference Across Groups": pct_diff
            })

        # plot
        plt.figure(figsize=(9, 5))
        plt.bar(group_summary["group"].astype(str), group_summary["mean"])
        plt.errorbar(
            x=np.arange(len(group_summary)),
            y=group_summary["mean"],
            yerr=group_summary["std"].fillna(0),
            fmt="none",
            capsize=4
        )
        plt.xticks(rotation=45, ha="right")
        plt.ylabel(metric)
        plt.xlabel(factor_label)
        plt.title(f"{factor_label}: {metric} (descriptive only)")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()

        safe_metric = re.sub(r"[^A-Za-z0-9_]+", "_", metric).strip("_")
        plt.savefig(
            os.path.join(output_dir, f"{factor_label.lower().replace(' ', '_')}_{safe_metric}_descriptive.png"),
            dpi=300,
            bbox_inches="tight"
        )
        plt.close()

    summary_df = pd.DataFrame(summary_rows)
    return summary_df, skipped_metrics

# =========================================================
# LOAD FILES
# =========================================================
water_long = pd.read_csv(water_long_file)
embed_long = pd.read_csv(embed_long_file)

water_tested = load_existing_metrics(water_stats_file)
embed_tested = load_existing_metrics(embed_stats_file)

# =========================================================
# RUN DESCRIPTIVE SUMMARIES
# =========================================================
water_desc_df, water_skipped = summarize_skipped_metrics(
    water_long,
    water_tested,
    "Water Depth"
)

embed_desc_df, embed_skipped = summarize_skipped_metrics(
    embed_long,
    embed_tested,
    "Embedment Depth"
)

# =========================================================
# SAVE
# =========================================================
if not water_desc_df.empty:
    water_desc_df.to_csv(
        os.path.join(output_dir, "water_depth_skipped_metrics_descriptive_stats.csv"),
        index=False
    )

if not embed_desc_df.empty:
    embed_desc_df.to_csv(
        os.path.join(output_dir, "embedment_depth_skipped_metrics_descriptive_stats.csv"),
        index=False
    )

combined_desc_df = pd.concat([water_desc_df, embed_desc_df], ignore_index=True)
combined_desc_df.to_csv(
    os.path.join(output_dir, "combined_skipped_metrics_descriptive_stats.csv"),
    index=False
)

print("\nSaved descriptive stats to:")
print(os.path.join(output_dir, "combined_skipped_metrics_descriptive_stats.csv"))

print("\nWater depth skipped metrics:")
print(water_skipped)

print("\nEmbedment depth skipped metrics:")
print(embed_skipped)

print("\nPreview of descriptive stats:")
print(combined_desc_df.head(20))

In [ ]:
import pandas as pd
import os

# =========================================================
# LOAD
# =========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_factor_ranking_stats_FIXED"

sed = pd.read_csv(os.path.join(BASE_DIR, "sediment_true_statistics", "sediment_metric_significance_and_effect_sizes.csv"))
water = pd.read_csv(os.path.join(BASE_DIR, "water_true_statistics", "water_metric_stats.csv"))
embed = pd.read_csv(os.path.join(BASE_DIR, "embedment_true_statistics", "embedment_metric_stats.csv"))

sed["Factor"] = "Sediment"
water["Factor"] = "Water Depth"
embed["Factor"] = "Embedment Depth"

df = pd.concat([sed, water, embed], ignore_index=True)

# =========================================================
# CLASSIFY EFFECT SIZE
# =========================================================
def classify_effect(val):
    if pd.isna(val):
        return "○"
    elif val > 0.5:
        return "●●●"
    elif val > 0.15:
        return "●●"
    elif val > 0.05:
        return "●"
    else:
        return "○"

df["Control Strength"] = df["Eta-squared"].apply(classify_effect)

# =========================================================
# PIVOT
# =========================================================
matrix = df.pivot_table(
    index="Metric",
    columns="Factor",
    values="Control Strength",
    aggfunc="first"
).fillna("○")

# sort nicely
matrix = matrix.sort_index()

# =========================================================
# SAVE
# =========================================================
out_path = os.path.join(BASE_DIR, "control_matrix_summary.csv")
matrix.to_csv(out_path)

print("\nCONTROL MATRIX:")
print(matrix)

print("\nSaved to:", out_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# =========================================================
# LOAD YOUR FINAL TABLE
# =========================================================
df = pd.read_csv(
    "/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_factor_ranking_stats_FIXED/corrected_factor_control/factor_control_by_metric.csv"
)

# DROP BAD METRIC
df = df[df["Metric"] != "First Inflection Depth (m)"]

# =========================================================
# COLOR MAP
# =========================================================
color_map = {
    "Sediment": "green",
    "Water Depth": "purple",
    "Embedment Depth": "orange"
}

df["Color"] = df["Dominant Factor"].map(color_map)

# =========================================================
# SORT FOR CLEAN VISUAL
# =========================================================
df = df.sort_values("Max Effect Size", ascending=True)

# =========================================================
# PLOT
# =========================================================
plt.figure(figsize=(10, 6))

plt.barh(
    df["Metric"],
    df["Max Effect Size"],
    color=df["Color"]
)

plt.xlabel("Maximum Effect Size (η²)")
plt.title("Dominant Controls on Monopile Response Metrics")

# legend
handles = [
    plt.Line2D([0], [0], color="green", lw=6, label="Sediment"),
    plt.Line2D([0], [0], color="purple", lw=6, label="Water Depth"),
    plt.Line2D([0], [0], color="orange", lw=6, label="Embedment Depth")
]
plt.legend(handles=handles)

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

out_path = "/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/master_factor_ranking_stats_FIXED/corrected_factor_control/dominant_factor_plot.png"
plt.savefig(out_path, dpi=300)
plt.show()

print("Saved to:", out_path)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ==========================================================
# 1. PATHS
# ==========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"

FACTOR_FILES = {
    "Sediment": {
        "base": os.path.join(BASE_DIR, "Sediment Comparison", "gems_compare_output"),
        "files": {
            "Deflection_Slope_Pressure": "LC_Defelction_Slope_Pressure.csv",
            "Shear_Force_Bending_Moment": "LC_Shear_Force_Bending_Moment.csv",
            "Axial_Force": "Axial_Force.csv",
            "Axial_Stress": "Axial_Stress.csv",
            "Unit_Resistance": "Unit_Resistance.csv",
        }
    },
    "Water Depth": {
        "base": os.path.join(BASE_DIR, "Water Depth Comparison", "downloaded_csv"),
        "files": {
            "Deflection_Slope_Pressure": "deflection_slope_pressure.csv",
            "Shear_Force_Bending_Moment": "shear_force_bending_moment.csv",
            "Axial_Force": "axial_force.csv",
            "Axial_Stress": "axial_stress.csv",
            "Unit_Resistance": "unit_resistance.csv",
        }
    },
    "Embedment Depth": {
        "base": os.path.join(BASE_DIR, "Embedment Depth Comparison", "downloaded_csv"),
        "files": {
            "Deflection_Slope_Pressure": "deflection_slope_pressure.csv",
            "Shear_Force_Bending_Moment": "shear_force_bending_moment.csv",
            "Axial_Force": "axial_force.csv",
            "Axial_Stress": "axial_stress.csv",
            "Unit_Resistance": "unit_resistance.csv",
        }
    }
}

OUTPUT_DIR = os.path.join(BASE_DIR, "depth_bin_factor_influence")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# 2. DEPTH BINS
# ==========================================================
# Includes 60–70 so the full profile is covered.
DEPTH_BINS = [0, 10, 20, 30, 40, 50, 60, 70, np.inf]
DEPTH_LABELS = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-60", "60-70", "70+"]

# ==========================================================
# 3. HELPERS
# ==========================================================


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return df

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"[-+]?\d*\.?\d+", str(val))
    return float(m.group()) if m else np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    num = extract_numeric_value(val)
    if pd.notna(num):
        if float(num).is_integer():
            return str(int(num))
        return str(float(num))
    return str(val).strip()

def guess_depth_column(df: pd.DataFrame):
    cols = list(df.columns)
    priority_patterns = [
        r"^z\s*\(m\)$",
        r"^z$",
        r"depth",
        r"embedment",
    ]
    for patt in priority_patterns:
        for c in cols:
            if re.search(patt, c.lower()):
                return c
    return None

def guess_group_column_from_factor(df: pd.DataFrame, factor_name: str, depth_col: str):
    candidates = [c for c in df.columns if c != depth_col]

    factor_patterns = {
        "Sediment": [r"sediment", r"soil"],
        "Water Depth": [r"water depth", r"seafloor depth", r"depth"],
        "Embedment Depth": [r"embedment depth", r"embedment"],
    }

    for patt in factor_patterns.get(factor_name, []):
        for c in candidates:
            if re.search(patt, c.lower()):
                return c

    # fallback: first non-numeric-looking column
    for c in candidates:
        as_num = pd.to_numeric(df[c], errors="coerce")
        if as_num.notna().sum() < len(df) * 0.8:
            return c

    # final fallback
    return candidates[0] if candidates else None

def read_factor_metric_csv(factor_name, metric_family, csv_path):
    if not os.path.exists(csv_path):
        print(f"Missing file: {csv_path}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    df = clean_columns(df)

    # ----------------------------
    # Identify depth column first
    # ----------------------------
    depth_col = guess_depth_column(df)
    if depth_col is None:
        print(f"Skipped {csv_path}: no depth column found.")
        return pd.DataFrame()

    # ----------------------------
    # Drop combined metadata cols first
    # ----------------------------
    drop_cols = []
    for c in df.columns:
        cl = c.lower()
        if c == depth_col:
            continue
        if "loadcase-" in cl or "loadcase -" in cl:
            drop_cols.append(c)

    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")

    # ----------------------------
    # Now find the real group column
    # ----------------------------
    group_col = guess_group_column_from_factor(df, factor_name, depth_col)
    if group_col is None or group_col not in df.columns:
        print(f"Skipped {csv_path}: no valid group column found.")
        return pd.DataFrame()

    # ----------------------------
    # Convert depth
    # ----------------------------
    df[depth_col] = pd.to_numeric(df[depth_col].apply(extract_numeric_value), errors="coerce")

    # ----------------------------
    # Remove other metadata columns
    # ----------------------------
    meta_cols = []
    for c in df.columns:
        cl = c.lower()
        if c in [depth_col, group_col]:
            continue
        if "load case" in cl or cl == "loadcase":
            meta_cols.append(c)

    if meta_cols:
        df = df.drop(columns=meta_cols, errors="ignore")

    # ----------------------------
    # Find numeric value columns
    # ----------------------------
    value_cols = []
    for c in df.columns:
        if c in [depth_col, group_col]:
            continue
        as_num = pd.to_numeric(df[c], errors="coerce")
        if as_num.notna().sum() > 0:
            value_cols.append(c)

    if not value_cols:
        print(f"Skipped {csv_path}: no numeric value columns found.")
        return pd.DataFrame()

    # ----------------------------
    # Clean group labels
    # ----------------------------
    df[group_col] = df[group_col].apply(normalize_group_label)

    # ----------------------------
    # Melt to long
    # ----------------------------
    long_df = df.melt(
        id_vars=[group_col, depth_col],
        value_vars=value_cols,
        var_name="Metric",
        value_name="Value"
    )

    long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")
    long_df = long_df.dropna(subset=[depth_col, "Value"]).copy()

    long_df = long_df.rename(columns={
        group_col: "Group",
        depth_col: "Depth_m"
    })

    long_df["Factor"] = factor_name
    long_df["Family"] = metric_family

    long_df["DepthBin"] = pd.cut(
        long_df["Depth_m"],
        bins=DEPTH_BINS,
        labels=DEPTH_LABELS,
        include_lowest=True,
        right=False
    )

    return long_df[["Factor", "Family", "Metric", "Group", "Depth_m", "DepthBin", "Value"]]

# ==========================================================
# 5. BUILD MASTER LONG TABLE
# ==========================================================
all_long = []

for factor_name, info in FACTOR_FILES.items():
    base = info["base"]
    for family, fname in info["files"].items():
        path = os.path.join(base, fname)
        tmp = read_factor_metric_csv(factor_name, family, path)
        if not tmp.empty:
            all_long.append(tmp)

if not all_long:
    raise ValueError("No data were loaded. Check file paths and column names.")

long_df = pd.concat(all_long, ignore_index=True)
long_df.to_csv(os.path.join(OUTPUT_DIR, "depth_factor_master_long.csv"), index=False)

print("Loaded rows:", len(long_df))
print(long_df.head())

# ==========================================================
# 6. METRIC-LEVEL FACTOR INFLUENCE BY DEPTH BIN
# ==========================================================
metric_rows = []

# aggregate over load cases by keeping all values;
# this lets the within-bin variability contribute to the effect size
for (depth_bin, family, metric, factor), sub in long_df.groupby(["DepthBin", "Family", "Metric", "Factor"], observed=False):
    if pd.isna(depth_bin):
        continue

    grouped_values = []
    group_means = []

    for gname, gsub in sub.groupby("Group", observed=False):
        vals = gsub["Value"].dropna().values
        if len(vals) >= 2:
            grouped_values.append(vals)
        if len(vals) >= 1:
            group_means.append(np.mean(vals))

    eta = compute_eta_squared(grouped_values)
    norm_range = compute_normalized_range(group_means)

    metric_rows.append({
        "DepthBin": str(depth_bin),
        "Family": family,
        "Metric": metric,
        "Factor": factor,
        "EtaSquared": eta,
        "NormalizedRange": norm_range,
        "N_Groups": len(group_means)
    })

metric_df = pd.DataFrame(metric_rows)
metric_df.to_csv(os.path.join(OUTPUT_DIR, "metric_level_factor_influence_by_depth_bin.csv"), index=False)

# ==========================================================
# 7. DETERMINE DOMINANT FACTOR FOR EACH METRIC IN EACH BIN
# ==========================================================
dominance_rows = []

for (depth_bin, family, metric), sub in metric_df.groupby(["DepthBin", "Family", "Metric"], observed=False):
    sub_eta = sub.dropna(subset=["EtaSquared"]).copy()
    sub_rng = sub.dropna(subset=["NormalizedRange"]).copy()

    dominant_eta = sub_eta.loc[sub_eta["EtaSquared"].idxmax(), "Factor"] if not sub_eta.empty else np.nan
    dominant_rng = sub_rng.loc[sub_rng["NormalizedRange"].idxmax(), "Factor"] if not sub_rng.empty else np.nan

    dominance_rows.append({
        "DepthBin": depth_bin,
        "Family": family,
        "Metric": metric,
        "DominantFactor_Eta": dominant_eta,
        "DominantFactor_Range": dominant_rng
    })

dominance_df = pd.DataFrame(dominance_rows)
dominance_df.to_csv(os.path.join(OUTPUT_DIR, "dominant_factor_by_metric_and_depth_bin.csv"), index=False)

# ==========================================================
# 8. OVERALL DOMINANT FACTOR PER DEPTH BIN
# ==========================================================
summary_rows = []

for depth_bin, sub in metric_df.groupby("DepthBin", observed=False):
    if pd.isna(depth_bin):
        continue

    # average effect size / range across all metric families
    score = (
        sub.groupby("Factor", observed=False)
        .agg(
            MeanEta=("EtaSquared", "mean"),
            MeanRange=("NormalizedRange", "mean"),
            MetricCount=("Metric", "nunique")
        )
        .reset_index()
    )

    score["EtaRank"] = score["MeanEta"].rank(ascending=False, method="min")
    score["RangeRank"] = score["MeanRange"].rank(ascending=False, method="min")

    # dominant factor by mean eta
    eta_valid = score.dropna(subset=["MeanEta"])
    rng_valid = score.dropna(subset=["MeanRange"])

    dominant_eta = eta_valid.loc[eta_valid["MeanEta"].idxmax(), "Factor"] if not eta_valid.empty else np.nan
    dominant_rng = rng_valid.loc[rng_valid["MeanRange"].idxmax(), "Factor"] if not rng_valid.empty else np.nan

    # how many metrics each factor "won"
    wins = dominance_df[dominance_df["DepthBin"] == depth_bin]["DominantFactor_Eta"].value_counts(dropna=True).to_dict()

    row = {
        "DepthBin": depth_bin,
        "DominantFactor_MeanEta": dominant_eta,
        "DominantFactor_MeanRange": dominant_rng,
    }

    for factor in ["Sediment", "Water Depth", "Embedment Depth"]:
        row[f"{factor}_MeanEta"] = score.loc[score["Factor"] == factor, "MeanEta"].values[0] if factor in score["Factor"].values else np.nan
        row[f"{factor}_MeanRange"] = score.loc[score["Factor"] == factor, "MeanRange"].values[0] if factor in score["Factor"].values else np.nan
        row[f"{factor}_MetricWins"] = wins.get(factor, 0)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df["DepthBin"] = pd.Categorical(summary_df["DepthBin"], categories=DEPTH_LABELS, ordered=True)
summary_df = summary_df.sort_values("DepthBin")
summary_df.to_csv(os.path.join(OUTPUT_DIR, "overall_dominant_factor_by_depth_bin.csv"), index=False)

print("\n=== OVERALL DOMINANT FACTOR BY DEPTH BIN ===")
print(summary_df)

# ==========================================================
# 9. PLOT: HEATMAP-LIKE MATRIX OF MEAN ETA
# ==========================================================
plot_df = summary_df.set_index("DepthBin")[
    ["Sediment_MeanEta", "Water Depth_MeanEta", "Embedment Depth_MeanEta"]
].copy()
plot_df.columns = ["Sediment", "Water Depth", "Embedment Depth"]

plt.figure(figsize=(8, 6))
plt.imshow(plot_df.T, aspect="auto")
plt.yticks(range(len(plot_df.columns)), plot_df.columns)
plt.xticks(range(len(plot_df.index)), plot_df.index, rotation=45, ha="right")
plt.colorbar(label=r"Mean $\eta^2$")
plt.title(r"Main Influencing Factor by Depth Range (Mean $\eta^2$)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mean_eta_heatmap_by_depth_bin.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# ==========================================================
# 10. PLOT: DOMINANT FACTOR LABELS
# ==========================================================
factor_to_num = {"Sediment": 0, "Water Depth": 1, "Embedment Depth": 2}
num_to_factor = {v: k for k, v in factor_to_num.items()}

dom_nums = [factor_to_num.get(x, np.nan) for x in summary_df["DominantFactor_MeanEta"]]

plt.figure(figsize=(8, 3))
plt.scatter(range(len(summary_df)), dom_nums, s=120)
plt.yticks([0, 1, 2], ["Sediment", "Water Depth", "Embedment Depth"])
plt.xticks(range(len(summary_df)), summary_df["DepthBin"], rotation=45, ha="right")
plt.title("Dominant Factor in Each Depth Range")
plt.ylabel("Dominant Factor")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "dominant_factor_by_depth_bin.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"\nSaved outputs to: {OUTPUT_DIR}")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. PATHS
# ==========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"

FACTOR_FILES = {
    "Sediment": {
        "base": os.path.join(BASE_DIR, "Sediment Comparison", "gems_compare_output"),
        "files": {
            "Deflection_Slope_Pressure": "LC_Defelction_Slope_Pressure.csv",
            "Shear_Force_Bending_Moment": "LC_Shear_Force_Bending_Moment.csv",
            "Axial_Force": "Axial_Force.csv",
            "Axial_Stress": "Axial_Stress.csv",
            "Unit_Resistance": "Unit_Resistance.csv",
        }
    },
    "Water Depth": {
        "base": os.path.join(BASE_DIR, "Water Depth Comparison", "downloaded_csv"),
        "files": {
            "Deflection_Slope_Pressure": "deflection_slope_pressure.csv",
            "Shear_Force_Bending_Moment": "shear_force_bending_moment.csv",
            "Axial_Force": "axial_force.csv",
            "Axial_Stress": "axial_stress.csv",
            "Unit_Resistance": "unit_resistance.csv",
        }
    },
    "Embedment Depth": {
        "base": os.path.join(BASE_DIR, "Embedment Depth Comparison", "downloaded_csv"),
        "files": {
            "Deflection_Slope_Pressure": "deflection_slope_pressure.csv",
            "Shear_Force_Bending_Moment": "shear_force_bending_moment.csv",
            "Axial_Force": "axial_force.csv",
            "Axial_Stress": "axial_stress.csv",
            "Unit_Resistance": "unit_resistance.csv",
        }
    }
}

OUTPUT_DIR = os.path.join(BASE_DIR, "depth_bin_factor_influence_with_variables")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# 2. DEPTH BINS
# ==========================================================
DEPTH_BINS = [0, 10, 20, 30, 40, 50, 60, 70, np.inf]
DEPTH_LABELS = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-60", "60-70", "70+"]

# ==========================================================
# 3. HELPERS
# ==========================================================
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return df

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"[-+]?\d*\.?\d+", str(val))
    return float(m.group()) if m else np.nan

def normalize_group_label(val):
    if pd.isna(val):
        return np.nan
    num = extract_numeric_value(val)
    if pd.notna(num):
        if float(num).is_integer():
            return str(int(num))
        return str(float(num))
    return str(val).strip()

def guess_depth_column(df: pd.DataFrame):
    cols = list(df.columns)
    priority_patterns = [
        r"^z\s*\(m\)$",
        r"^z$",
        r"depth",
        r"embedment",
    ]
    for patt in priority_patterns:
        for c in cols:
            if re.search(patt, c.lower()):
                return c
    return None

def guess_group_column_from_factor(df: pd.DataFrame, factor_name: str, depth_col: str):
    candidates = [c for c in df.columns if c != depth_col]

    factor_patterns = {
        "Sediment": [r"sediment", r"soil"],
        "Water Depth": [r"water depth", r"seafloor depth"],
        "Embedment Depth": [r"embedment depth", r"embedment"],
    }

    for patt in factor_patterns.get(factor_name, []):
        for c in candidates:
            if re.search(patt, c.lower()):
                return c

    for c in candidates:
        as_num = pd.to_numeric(df[c], errors="coerce")
        if as_num.notna().sum() < len(df) * 0.8:
            return c

    return candidates[0] if candidates else None

def compute_eta_squared(groups):
    valid = [np.asarray(g, dtype=float) for g in groups if len(g) > 1]
    if len(valid) < 2:
        return np.nan
    all_vals = np.concatenate(valid)
    grand_mean = np.mean(all_vals)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in valid)
    ss_total = np.sum((all_vals - grand_mean) ** 2)
    return ss_between / ss_total if ss_total != 0 else np.nan

def standardize_metric_name(family, metric):
    m = str(metric).strip()
    ml = m.lower()

    # remove units only for matching
    m_nounit = re.sub(r"\(.*?\)", "", ml).strip()
    m_nounit = m_nounit.replace("_", " ").replace("-", " ")
    m_nounit = re.sub(r"\s+", " ", m_nounit).strip()

    if "max deflection" in m_nounit:
        return "Max Deflection"
    if m_nounit == "deflection":
        return "Deflection"
    if "slope" in m_nounit or "rotation" in m_nounit:
        return "Slope"
    if "soil pressure" in m_nounit:
        return "Soil Pressure"
    if "shear force" in m_nounit:
        return "Shear Force"
    if "bending moment" in m_nounit or m_nounit == "moment":
        return "Bending Moment"
    if "axial force" in m_nounit:
        return "Axial Force"
    if "axial stress" in m_nounit or m_nounit == "stress":
        return "Axial Stress"
    if "unit shaft" in m_nounit:
        return "Unit Shaft Resistance"
    if "unit base" in m_nounit:
        return "Unit Base Resistance"
    if m_nounit == "t":
        return "t-z response"
    if m_nounit == "p":
        return "p-y response"
    if m_nounit == "q":
        return "q-z response"

    return f"{family}: {m}"

def read_factor_metric_csv(factor_name, metric_family, csv_path):
    if not os.path.exists(csv_path):
        print(f"Missing file: {csv_path}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    df = clean_columns(df)

    depth_col = guess_depth_column(df)
    if depth_col is None:
        print(f"Skipped {csv_path}: no depth column found.")
        return pd.DataFrame()

    drop_cols = []
    for c in df.columns:
        cl = c.lower()
        if c == depth_col:
            continue
        if "loadcase-" in cl or "loadcase -" in cl:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")

    group_col = guess_group_column_from_factor(df, factor_name, depth_col)
    if group_col is None or group_col not in df.columns:
        print(f"Skipped {csv_path}: no valid group column found.")
        return pd.DataFrame()

    df[depth_col] = pd.to_numeric(df[depth_col].apply(extract_numeric_value), errors="coerce")

    meta_cols = []
    for c in df.columns:
        cl = c.lower()
        if c in [depth_col, group_col]:
            continue
        if "load case" in cl or cl == "loadcase":
            meta_cols.append(c)
    if meta_cols:
        df = df.drop(columns=meta_cols, errors="ignore")

    value_cols = []
    for c in df.columns:
        if c in [depth_col, group_col]:
            continue
        as_num = pd.to_numeric(df[c], errors="coerce")
        if as_num.notna().sum() > 0:
            value_cols.append(c)

    if not value_cols:
        print(f"Skipped {csv_path}: no numeric value columns found.")
        return pd.DataFrame()

    df[group_col] = df[group_col].apply(normalize_group_label)

    long_df = df.melt(
        id_vars=[group_col, depth_col],
        value_vars=value_cols,
        var_name="Metric",
        value_name="Value"
    )

    long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")
    long_df = long_df.dropna(subset=[depth_col, "Value"]).copy()

    long_df = long_df.rename(columns={
        group_col: "Group",
        depth_col: "Depth_m"
    })

    long_df["Factor"] = factor_name
    long_df["Family"] = metric_family
    long_df["MetricClean"] = long_df["Metric"].apply(lambda x: standardize_metric_name(metric_family, x))
    long_df["DepthBin"] = pd.cut(
        long_df["Depth_m"],
        bins=DEPTH_BINS,
        labels=DEPTH_LABELS,
        include_lowest=True,
        right=False
    )

    return long_df[["Factor", "Family", "Metric", "MetricClean", "Group", "Depth_m", "DepthBin", "Value"]]

# ==========================================================
# 4. BUILD MASTER LONG TABLE
# ==========================================================
all_long = []

for factor_name, info in FACTOR_FILES.items():
    for family, fname in info["files"].items():
        path = os.path.join(info["base"], fname)
        tmp = read_factor_metric_csv(factor_name, family, path)
        if not tmp.empty:
            all_long.append(tmp)

if not all_long:
    raise ValueError("No data loaded. Check file paths and columns.")

long_df = pd.concat(all_long, ignore_index=True)
long_df.to_csv(os.path.join(OUTPUT_DIR, "depth_factor_master_long.csv"), index=False)

# ==========================================================
# 5. METRIC-LEVEL ETA BY DEPTH BIN AND FACTOR
# ==========================================================
metric_rows = []

for (depth_bin, family, metric_clean, factor), sub in long_df.groupby(
    ["DepthBin", "Family", "MetricClean", "Factor"], observed=False
):
    if pd.isna(depth_bin):
        continue

    grouped_values = []
    for _, gsub in sub.groupby("Group", observed=False):
        vals = gsub["Value"].dropna().values
        if len(vals) >= 2:
            grouped_values.append(vals)

    eta = compute_eta_squared(grouped_values)

    metric_rows.append({
        "DepthBin": str(depth_bin),
        "Family": family,
        "Metric": metric_clean,
        "Factor": factor,
        "EtaSquared": eta
    })

metric_df = pd.DataFrame(metric_rows)
metric_df.to_csv(os.path.join(OUTPUT_DIR, "metric_level_eta_by_depth_bin.csv"), index=False)

# ==========================================================
# 6. TOP VARIABLES DRIVING EACH FACTOR IN EACH DEPTH BIN
# ==========================================================
top_variable_rows = []

for (depth_bin, factor), sub in metric_df.groupby(["DepthBin", "Factor"], observed=False):
    sub = sub.dropna(subset=["EtaSquared"]).sort_values("EtaSquared", ascending=False)
    if sub.empty:
        continue

    top_n = sub.head(5).copy()
    for rank, (_, row) in enumerate(top_n.iterrows(), start=1):
        top_variable_rows.append({
            "DepthBin": depth_bin,
            "Factor": factor,
            "Rank": rank,
            "Metric": row["Metric"],
            "Family": row["Family"],
            "EtaSquared": row["EtaSquared"]
        })

top_variables_df = pd.DataFrame(top_variable_rows)
top_variables_df.to_csv(os.path.join(OUTPUT_DIR, "top_variables_by_factor_and_depth_bin.csv"), index=False)

# ==========================================================
# 7. MAIN FACTOR BY DEPTH BIN + WHICH VARIABLES DROVE IT
# ==========================================================
summary_rows = []

for depth_bin, sub in metric_df.groupby("DepthBin", observed=False):
    if pd.isna(depth_bin):
        continue

    score = (
        sub.groupby("Factor", observed=False)
        .agg(MeanEta=("EtaSquared", "mean"))
        .reset_index()
    )

    score = score.dropna(subset=["MeanEta"])
    if score.empty:
        continue

    dominant_factor = score.loc[score["MeanEta"].idxmax(), "Factor"]

    dominant_sub = (
        sub[sub["Factor"] == dominant_factor]
        .dropna(subset=["EtaSquared"])
        .sort_values("EtaSquared", ascending=False)
        .head(3)
    )

    driver_text = "; ".join(
        [f"{r['Metric']} ({r['EtaSquared']:.2f})" for _, r in dominant_sub.iterrows()]
    )

    row = {
        "DepthBin": depth_bin,
        "DominantFactor": dominant_factor,
        "DriverVariables": driver_text
    }

    for factor in ["Sediment", "Water Depth", "Embedment Depth"]:
        val = score.loc[score["Factor"] == factor, "MeanEta"]
        row[f"{factor}_MeanEta"] = val.values[0] if len(val) > 0 else np.nan

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df["DepthBin"] = pd.Categorical(summary_df["DepthBin"], categories=DEPTH_LABELS, ordered=True)
summary_df = summary_df.sort_values("DepthBin")
summary_df.to_csv(os.path.join(OUTPUT_DIR, "main_factor_and_driver_variables_by_depth_bin.csv"), index=False)

print("\n=== MAIN FACTOR + DRIVER VARIABLES BY DEPTH BIN ===")
print(summary_df[["DepthBin", "DominantFactor", "DriverVariables"]])

# ==========================================================
# 8. HEATMAP OF MEAN ETA
# ==========================================================
plot_df = summary_df.set_index("DepthBin")[
    ["Sediment_MeanEta", "Water Depth_MeanEta", "Embedment Depth_MeanEta"]
].copy()
plot_df.columns = ["Sediment", "Water Depth", "Embedment Depth"]

plt.figure(figsize=(8, 6))
plt.imshow(plot_df.T, aspect="auto")
plt.yticks(range(len(plot_df.columns)), plot_df.columns)
plt.xticks(range(len(plot_df.index)), plot_df.index, rotation=45, ha="right")
plt.colorbar(label=r"Mean $\eta^2$")
plt.title(r"Main Influencing Factor by Depth Range (Mean $\eta^2$)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mean_eta_heatmap_by_depth_bin.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()


# ==========================================================
# 9. VERTICAL PROFILE PLOT (DEPTH ON Y-AXIS)
# ==========================================================
# ==========================================================
# ROBUST ABBREVIATION LOGIC
# ==========================================================
def clean_metric_for_matching(metric):
    m = str(metric).strip().lower()

    # remove family prefixes like "Deflection_Slope_Pressure: "
    if ":" in m:
        m = m.split(":", 1)[1].strip()

    # remove units in parentheses
    m = re.sub(r"\(.*?\)", "", m).strip()

    # normalize separators
    m = m.replace("_", " ").replace("-", " ")
    m = re.sub(r"\s+", " ", m).strip()

    return m

def abbreviate_metric_name(metric):
    m = clean_metric_for_matching(metric)

    # ordered matching from most specific to most general
    if "max deflection" in m:
        return "DEF"
    if m == "deflection":
        return "DEF"
    if "slope" in m or "rotation" in m:
        return "SLP"
    if "soil pressure" in m:
        return "SP"
    if "shear force" in m:
        return "SF"
    if "bending moment" in m or m == "moment":
        return "BM"
    if "axial force" in m:
        return "AF"
    if "axial stress" in m or m == "stress":
        return "AS"
    if "unit shaft" in m:
        return "USR"
    if "unit base" in m:
        return "UBR"
    if "mean unit shaft" in m:
        return "USR"
    if "mean unit base" in m:
        return "UBR"
    if "initial stiffness" in m and "p y" in m:
        return "PYs"
    if "p max" in m or ("ultimate p" in m):
        return "PYm"
    if "t max" in m:
        return "TZt"
    if "initial stiffness" in m and "t z" in m:
        return "TZs"
    if "q max" in m:
        return "QZq"
    if "initial stiffness" in m and "q z" in m:
        return "QZs"
    if "p y response" in m or m == "p":
        return "PY"
    if "t z response" in m or m == "t":
        return "TZ"
    if "q z response" in m or m == "q":
        return "QZ"

    # cleaner fallback than metric[:4].upper()
    words = m.split()
    if len(words) == 1:
        return words[0][:3].upper()
    return "".join(w[0].upper() for w in words[:3])

# Convert depth bins to numeric centers
def depth_bin_center(label):
    if "+" in label:
        return 75  # midpoint for 70+
    low, high = label.split("-")
    return (float(low) + float(high)) / 2

# Build rows with MULTIPLE FACTORS PER DEPTH
rows = []

for depth_bin in DEPTH_LABELS:
    sub = metric_df[metric_df["DepthBin"] == depth_bin].dropna(subset=["EtaSquared"])
    if sub.empty:
        continue

    # Compute mean eta per factor
    score = (
        sub.groupby("Factor")
        .agg(MeanEta=("EtaSquared", "mean"))
        .reset_index()
    )

    if score.empty:
        continue

    # Normalize to compare importance
    max_eta = score["MeanEta"].max()

    # Keep ALL factors within 80% of max (this controls multiple dots)
    threshold = 0.8 * max_eta
    important = score[score["MeanEta"] >= threshold]

    for _, row in important.iterrows():
        factor = row["Factor"]

        # get top drivers for THIS factor at THIS depth
        drivers = (
            sub[sub["Factor"] == factor]
            .sort_values("EtaSquared", ascending=False)
            .head(2)
        )

        driver_codes = []
        for _, d in drivers.iterrows():
            prefix = factor_prefix.get(factor, "?")
            code = abbreviate_metric_name(d["Metric"])
            driver_codes.append(f"{prefix}-{code}")

        rows.append({
            "DepthBin": depth_bin,
            "Depth": depth_bin_center(depth_bin),
            "Factor": factor,
            "X": factor_to_x[factor],
            "Color": factor_colors[factor],
            "Drivers": ", ".join(driver_codes)
        })

plot_df = pd.DataFrame(rows)

# ==========================================================
# 10. PLOT
# ==========================================================
plt.figure(figsize=(6, 8))

plt.scatter(
    plot_df["X"],
    plot_df["Depth"],
    c=plot_df["Color"],
    s=180,
    edgecolor="black"
)

# Add labels
for _, r in plot_df.iterrows():
    plt.text(
        r["X"] + 0.05,
        r["Depth"],
        r["Drivers"],
        fontsize=8,
        va="center"
    )

plt.xticks([0, 1, 2], ["Sediment", "Water Depth", "Embedment Depth"])
plt.ylabel("Depth (m)")
plt.title(r"Main Influencing Factors by Depth (Mean $\eta^2$)")

# 🔥 invert y-axis so 0 is at the top
plt.gca().invert_yaxis()

# Legend
from matplotlib.lines import Line2D
legend_text = [
    "S = Sediment",
    "W = Water Depth",
    "E = Embedment Depth",
    "",
    "DEF = Max Deflection",
    "SLP = Slope",
    "SP  = Soil Pressure",
    "BM  = Bending Moment",
    "SF  = Shear Force",
    "AF  = Axial Force",
    "AS  = Axial Stress",
    "USR = Unit Shaft Resistance",
    "UBR = Unit Base Resistance",
    "PYs = p-y stiffness",
    "PYm = p-y max",
    "TZt = t-z max",
    "TZs = t-z stiffness",
    "QZq = q-z max",
    "QZs = q-z stiffness"
]

legend_str = "\n".join(legend_text)

# Add text box to right side
plt.gcf().text(
    1.02, 0.5, legend_str,
    fontsize=8,
    va='center',
    ha='left',
    family='monospace'
)

# Make room for legend
plt.subplots_adjust(right=0.75)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "dominant_factor_profile_with_multiple_factors.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

legend_rows = []

for factor_name, prefix in factor_prefix.items():
    for full_name, short_code in abbr_map.items():
        legend_rows.append({
            "Factor": factor_name,
            "Prefix": prefix,
            "Variable": full_name,
            "Code": f"{prefix}-{short_code}"
        })

legend_df = pd.DataFrame(legend_rows)
legend_df.to_csv(os.path.join(OUTPUT_DIR, "prefixed_driver_code_legend.csv"), index=False)

print("\n=== DRIVER CODE LEGEND ===")
print(legend_df.head(20))



print(f"\nSaved outputs to: {OUTPUT_DIR}")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ==========================================================
# 1. PATHS
# ==========================================================
BASE_DIR = r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS"

SEDIMENT_FILES = {
    "base": os.path.join(BASE_DIR, "Sediment Comparison", "gems_compare_output"),
    "files": {
        "Deflection_Slope_Pressure": "LC_Defelction_Slope_Pressure.csv",
        "Shear_Force_Bending_Moment": "LC_Shear_Force_Bending_Moment.csv",
        "Axial_Force": "Axial_Force.csv",
        "Axial_Stress": "Axial_Stress.csv",
        "Unit_Resistance": "Unit_Resistance.csv",
    }
}

OUTPUT_DIR = os.path.join(BASE_DIR, "sediment_type_depth_profile")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# 2. DEPTH BINS
# ==========================================================
DEPTH_BINS = [0, 10, 20, 30, 40, 50, 60, 70, np.inf]
DEPTH_LABELS = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-60", "60-70", "70+"]

# ==========================================================
# 3. COLOR MAP
# ==========================================================
SEDIMENT_COLORS = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf",
}

# ==========================================================
# 4. HELPERS
# ==========================================================
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return df

def extract_numeric_value(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"[-+]?\d*\.?\d+", str(val))
    return float(m.group()) if m else np.nan

def guess_depth_column(df: pd.DataFrame):
    cols = list(df.columns)
    patterns = [r"^z\s*\(m\)$", r"^z$", r"depth", r"embedment"]
    for patt in patterns:
        for c in cols:
            if re.search(patt, c.lower()):
                return c
    return None

def get_sediment_column(metric_family: str, df: pd.DataFrame, depth_col: str):
    preferred = {
        "Deflection_Slope_Pressure": "Sediment Type",
        "Shear_Force_Bending_Moment": "Sediment Type",
        "Axial_Force": "Sediment Type",
        "Axial_Stress": "Sediment Type",
        "Unit_Resistance": "Sediment Type",
    }

    preferred_col = preferred.get(metric_family)
    if preferred_col in df.columns:
        return preferred_col

    candidates = [c for c in df.columns if c != depth_col]
    for patt in [r"sediment type", r"sediment", r"soil type", r"soil"]:
        for c in candidates:
            if re.search(patt, c.lower()):
                return c

    return None

def clean_sediment_label(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().strip(",")
    return s

def is_bad_sediment_label(s):
    if pd.isna(s):
        return True
    s = str(s).strip()

    # purely numeric
    if re.fullmatch(r"[-+]?\d*\.?\d+", s):
        return True

    # very short junk
    if len(s) <= 2:
        return True

    # labels that look like measurements
    if re.search(r"\d+\s*(?:mm|kpa|kn|m)\b", s, flags=re.IGNORECASE):
        return True

    return False

def standardize_metric_name(family, metric):
    m = str(metric).strip()
    ml = m.lower()

    m_nounit = re.sub(r"\(.*?\)", "", ml).strip()
    m_nounit = m_nounit.replace("_", " ").replace("-", " ")
    m_nounit = re.sub(r"\s+", " ", m_nounit).strip()

    if "max deflection" in m_nounit:
        return "Max Deflection"
    if m_nounit == "deflection":
        return "Deflection"
    if "slope" in m_nounit or "rotation" in m_nounit:
        return "Slope"
    if "soil pressure" in m_nounit:
        return "Soil Pressure"
    if "shear force" in m_nounit:
        return "Shear Force"
    if "bending moment" in m_nounit or m_nounit == "moment":
        return "Bending Moment"
    if "axial force" in m_nounit:
        return "Axial Force"
    if "axial stress" in m_nounit or m_nounit == "stress":
        return "Axial Stress"
    if "unit shaft" in m_nounit or "mean unit shaft" in m_nounit:
        return "Unit Shaft Resistance"
    if "unit base" in m_nounit or "mean unit base" in m_nounit:
        return "Unit Base Resistance"
    if m_nounit == "t":
        return "t-z response"
    if m_nounit == "p":
        return "p-y response"
    if m_nounit == "q":
        return "q-z response"

    return f"{family}: {m}"

def abbreviate_metric_name(metric):
    m = str(metric).strip().lower()

    if ":" in m:
        m = m.split(":", 1)[1].strip()

    m = re.sub(r"\(.*?\)", "", m).strip()
    m = m.replace("_", " ").replace("-", " ")
    m = re.sub(r"\s+", " ", m).strip()

    if "max deflection" in m or m == "deflection":
        return "DEF"
    if "slope" in m or "rotation" in m:
        return "SLP"
    if "soil pressure" in m:
        return "SP"
    if "shear force" in m:
        return "SF"
    if "bending moment" in m or m == "moment":
        return "BM"
    if "axial force" in m:
        return "AF"
    if "axial stress" in m or m == "stress":
        return "AS"
    if "unit shaft" in m:
        return "USR"
    if "unit base" in m:
        return "UBR"
    if "t z response" in m or m == "t":
        return "TZ"
    if "p y response" in m or m == "p":
        return "PY"
    if "q z response" in m or m == "q":
        return "QZ"

    words = m.split()
    if len(words) == 1:
        return words[0][:3].upper()
    return "".join(w[0].upper() for w in words[:3])

def depth_bin_center(label):
    if "+" in label:
        return 75.0
    low, high = label.split("-")
    return (float(low) + float(high)) / 2.0

# ==========================================================
# 5. READ SEDIMENT FILES INTO LONG FORMAT
# ==========================================================
def read_sediment_metric_csv(metric_family, csv_path):
    if not os.path.exists(csv_path):
        print(f"Missing file: {csv_path}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    df = clean_columns(df)

    depth_col = guess_depth_column(df)
    if depth_col is None:
        print(f"Skipped {csv_path}: no depth column found.")
        return pd.DataFrame()

    # remove weird combined columns
    drop_cols = []
    for c in df.columns:
        cl = c.lower()
        if c == depth_col:
            continue
        if "loadcase-" in cl or "loadcase -" in cl:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")

    sed_col = get_sediment_column(metric_family, df, depth_col)
    if sed_col is None or sed_col not in df.columns:
        print(f"Skipped {csv_path}: no sediment column found.")
        return pd.DataFrame()

    # clean depth and sediment label
    df[depth_col] = pd.to_numeric(df[depth_col].apply(extract_numeric_value), errors="coerce")
    df[sed_col] = df[sed_col].apply(clean_sediment_label)

    # remove rows where sediment label is junk
    df = df[~df[sed_col].apply(is_bad_sediment_label)].copy()

    print(f"\nUsing sediment column for {metric_family}: {sed_col}")
    print("Unique sediment types preview:")
    print(sorted(df[sed_col].dropna().astype(str).unique().tolist())[:15])

    # remove metadata columns
    meta_cols = []
    for c in df.columns:
        cl = c.lower()
        if c in [depth_col, sed_col]:
            continue
        if "load case" in cl or cl == "loadcase":
            meta_cols.append(c)
    if meta_cols:
        df = df.drop(columns=meta_cols, errors="ignore")

    # keep numeric value columns only
    value_cols = []
    for c in df.columns:
        if c in [depth_col, sed_col]:
            continue
        as_num = pd.to_numeric(df[c], errors="coerce")
        if as_num.notna().sum() > 0:
            value_cols.append(c)

    if not value_cols:
        print(f"Skipped {csv_path}: no numeric value columns found.")
        return pd.DataFrame()

    long_df = df.melt(
        id_vars=[sed_col, depth_col],
        value_vars=value_cols,
        var_name="Metric",
        value_name="Value"
    )

    long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")
    long_df = long_df.dropna(subset=[depth_col, "Value"]).copy()

    long_df = long_df.rename(columns={
        sed_col: "SedimentType",
        depth_col: "Depth_m"
    })

    long_df["Family"] = metric_family
    long_df["MetricClean"] = long_df["Metric"].apply(lambda x: standardize_metric_name(metric_family, x))
    long_df["DepthBin"] = pd.cut(
        long_df["Depth_m"],
        bins=DEPTH_BINS,
        labels=DEPTH_LABELS,
        include_lowest=True,
        right=False
    )

    return long_df[["Family", "Metric", "MetricClean", "SedimentType", "Depth_m", "DepthBin", "Value"]]

# ==========================================================
# 6. BUILD MASTER LONG TABLE
# ==========================================================
all_long = []

for family, fname in SEDIMENT_FILES["files"].items():
    path = os.path.join(SEDIMENT_FILES["base"], fname)
    tmp = read_sediment_metric_csv(family, path)
    if not tmp.empty:
        all_long.append(tmp)

if not all_long:
    raise ValueError("No sediment data loaded.")

long_df = pd.concat(all_long, ignore_index=True)
long_df.to_csv(os.path.join(OUTPUT_DIR, "sediment_depth_master_long.csv"), index=False)

print("\nlong_df shape:", long_df.shape)
print("Unique sediments in long_df:", sorted(long_df["SedimentType"].dropna().unique().tolist()))

# ==========================================================
# 7. COMPUTE SEDIMENT DISTINCTIVENESS BY DEPTH BIN
# ==========================================================
metric_mean = (
    long_df.groupby(["DepthBin", "Family", "MetricClean", "SedimentType"], observed=False)
    .agg(MeanValue=("Value", "mean"))
    .reset_index()
)

metric_mean["MetricZ"] = np.nan

for (depth_bin, family, metric), sub in metric_mean.groupby(
    ["DepthBin", "Family", "MetricClean"], observed=False
):
    valid = sub[np.isfinite(sub["MeanValue"])].copy()

    if len(valid) < 2:
        continue

    mu = valid["MeanValue"].mean()
    sd = valid["MeanValue"].std(ddof=0)

    if not np.isfinite(sd) or sd == 0:
        metric_mean.loc[valid.index, "MetricZ"] = 0.0
    else:
        metric_mean.loc[valid.index, "MetricZ"] = (valid["MeanValue"] - mu) / sd

metric_mean["AbsMetricZ"] = metric_mean["MetricZ"].abs()
metric_mean.to_csv(os.path.join(OUTPUT_DIR, "sediment_metric_zscores_by_depth_bin.csv"), index=False)

print("metric_mean non-null AbsMetricZ:", metric_mean["AbsMetricZ"].notna().sum())

# ==========================================================
# 8. SCORE SEDIMENT TYPES BY DEPTH BIN
# ==========================================================
sediment_score_df = (
    metric_mean.dropna(subset=["AbsMetricZ"])
    .groupby(["DepthBin", "SedimentType"], observed=False)
    .agg(
        MeanAbsZ=("AbsMetricZ", "mean"),
        MetricCount=("MetricClean", "nunique")
    )
    .reset_index()
)

sediment_score_df.to_csv(os.path.join(OUTPUT_DIR, "sediment_scores_by_depth_bin.csv"), index=False)
print("sediment_score_df shape:", sediment_score_df.shape)

# ==========================================================
# 9. DRIVER TABLE
# ==========================================================
driver_rows = []

for (depth_bin, sediment), sub in metric_mean.dropna(subset=["AbsMetricZ"]).groupby(
    ["DepthBin", "SedimentType"], observed=False
):
    sub = sub.sort_values("AbsMetricZ", ascending=False)
    if sub.empty:
        continue

    top2 = sub.head(2)

    driver_rows.append({
        "DepthBin": depth_bin,
        "SedimentType": sediment,
        "DriverCodes": ", ".join([abbreviate_metric_name(x) for x in top2["MetricClean"]]),
        "DriverDetail": "; ".join(
            [f"{abbreviate_metric_name(m)} ({z:.2f})" for m, z in zip(top2["MetricClean"], top2["AbsMetricZ"])]
        )
    })

driver_df = pd.DataFrame(driver_rows)
driver_df.to_csv(os.path.join(OUTPUT_DIR, "sediment_driver_codes_by_depth_bin.csv"), index=False)
print("driver_df shape:", driver_df.shape)

# ==========================================================
# 10. SELECT TOP 3 SEDIMENTS PER DEPTH BIN
# ==========================================================
plot_rows = []

for depth_bin in DEPTH_LABELS:
    sub = sediment_score_df[
        (sediment_score_df["DepthBin"] == depth_bin) &
        (np.isfinite(sediment_score_df["MeanAbsZ"]))
    ].copy()

    if sub.empty:
        print(f"No valid sediments for depth bin {depth_bin}")
        continue

    sub = sub[sub["MetricCount"] >= 1].copy()
    if sub.empty:
        print(f"No sediments with MetricCount >= 1 for {depth_bin}")
        continue

    important = sub.sort_values("MeanAbsZ", ascending=False).head(3)

    for _, r in important.iterrows():
        sed = r["SedimentType"]
        driver_match = driver_df[
            (driver_df["DepthBin"] == depth_bin) &
            (driver_df["SedimentType"] == sed)
        ]

        drivers = driver_match["DriverCodes"].iloc[0] if len(driver_match) > 0 else "NA"

        plot_rows.append({
            "DepthBin": depth_bin,
            "Depth": depth_bin_center(depth_bin),
            "SedimentType": sed,
            "MeanAbsZ": r["MeanAbsZ"],
            "DriverCodes": drivers
        })

plot_df = pd.DataFrame(plot_rows)
plot_df.to_csv(os.path.join(OUTPUT_DIR, "sediment_types_to_plot_by_depth_bin.csv"), index=False)

print("plot_df shape:", plot_df.shape)
print(plot_df.head(20))

if plot_df.empty:
    raise ValueError("plot_df is empty. No sediment types survived filtering.")

# ==========================================================
# 11. SEDIMENT ORDER + COLORS
# ==========================================================
sediment_order = sorted(plot_df["SedimentType"].dropna().unique().tolist())
sediment_to_x = {s: i for i, s in enumerate(sediment_order)}

plot_df["X"] = plot_df["SedimentType"].map(sediment_to_x)
plot_df["Color"] = plot_df["SedimentType"].map(SEDIMENT_COLORS).fillna("#333333")

# ==========================================================
# 12. PLOT
# ==========================================================
plt.figure(figsize=(12, 8))

plt.scatter(
    plot_df["X"],
    plot_df["Depth"],
    c=plot_df["Color"].tolist(),
    s=180,
    edgecolor="black"
)

for _, r in plot_df.iterrows():
    plt.text(
        r["X"] + 0.08,
        r["Depth"],
        f'{r["SedimentType"]}\n{r["DriverCodes"]}',
        fontsize=7,
        va="center"
    )

plt.xticks(range(len(sediment_order)), sediment_order, rotation=45, ha="right")
plt.ylabel("Depth (m)")
plt.title("Most Distinctive Sediment Types by Depth Range")
plt.gca().invert_yaxis()

# sediment legend
legend_lines = []
legend_text = []
for s in sediment_order:
    legend_lines.append(
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor=SEDIMENT_COLORS.get(s, "#333333"),
            markeredgecolor='black',
            markersize=8
        )
    )
    legend_text.append(s)

plt.legend(
    legend_lines,
    legend_text,
    title="Sediment Type",
    bbox_to_anchor=(1.02, 1.0),
    loc="upper left",
    frameon=True
)

# driver code legend
driver_legend_text = "\n".join([
    "DEF = Deflection",
    "SLP = Slope",
    "SP  = Soil Pressure",
    "BM  = Bending Moment",
    "SF  = Shear Force",
    "AF  = Axial Force",
    "AS  = Axial Stress",
    "USR = Unit Shaft Resistance",
    "UBR = Unit Base Resistance",
    "TZ  = t-z response",
    "PY  = p-y response",
    "QZ  = q-z response",
])

plt.gcf().text(
    1.02, 0.42,
    driver_legend_text,
    fontsize=8,
    va="top",
    ha="left",
    family="monospace"
)

plt.subplots_adjust(right=0.78)
plt.savefig(
    os.path.join(OUTPUT_DIR, "sediment_types_by_depth_profile.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

print(f"\nSaved outputs to: {OUTPUT_DIR}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ==========================================================
# 1. DRIVER CODE + FAMILY MAP
# ==========================================================
def abbreviate_metric_name(metric):
    m = str(metric).strip().lower()

    if ":" in m:
        m = m.split(":", 1)[1].strip()

    m = re.sub(r"\(.*?\)", "", m).strip()
    m = m.replace("_", " ").replace("-", " ")
    m = re.sub(r"\s+", " ", m).strip()

    if "max deflection" in m or m == "deflection":
        return "DEF"
    if "slope" in m or "rotation" in m:
        return "SLP"
    if "soil pressure" in m:
        return "SP"
    if "shear force" in m:
        return "SF"
    if "bending moment" in m or m == "moment":
        return "BM"
    if "axial force" in m:
        return "AF"
    if "axial stress" in m or m == "stress":
        return "AS"
    if "unit shaft" in m:
        return "USR"
    if "unit base" in m:
        return "UBR"
    if "t z response" in m or m == "t":
        return "TZ"
    if "p y response" in m or m == "p":
        return "PY"
    if "q z response" in m or m == "q":
        return "QZ"

    words = m.split()
    if len(words) == 1:
        return words[0][:3].upper()
    return "".join(w[0].upper() for w in words[:3])

def classify_metric_family(metric):
    code = abbreviate_metric_name(metric)

    if code in ["DEF", "SLP", "SP"]:
        return "Deflection/Slope/Pressure"
    if code in ["SF", "BM"]:
        return "Shear/Bending"
    if code in ["AF", "AS"]:
        return "Axial"
    if code in ["USR", "UBR"]:
        return "Unit Resistance"
    if code == "TZ":
        return "t-z"
    if code == "PY":
        return "p-y"
    if code == "QZ":
        return "q-z"
    return "Other"

family_color_map = {
    "Deflection/Slope/Pressure": "#8dd3c7",
    "Shear/Bending": "#ffffb3",
    "Axial": "#bebada",
    "Unit Resistance": "#fb8072",
    "t-z": "#80b1d3",
    "p-y": "#fdb462",
    "q-z": "#b3de69",
    "Other": "#d9d9d9",
}

# ==========================================================
# 2. FIND DOMINANT DRIVER FOR EACH SEDIMENT x DEPTH BIN
# ==========================================================
# metric_mean should already exist from your previous code
# Required columns:
#   DepthBin, SedimentType, MetricClean, AbsMetricZ

driver_pick = (
    metric_mean.dropna(subset=["AbsMetricZ"])
    .sort_values(["DepthBin", "SedimentType", "AbsMetricZ"], ascending=[True, True, False])
    .groupby(["DepthBin", "SedimentType"], observed=False)
    .first()
    .reset_index()
)

driver_pick["DriverCode"] = driver_pick["MetricClean"].apply(abbreviate_metric_name)
driver_pick["DriverFamily"] = driver_pick["MetricClean"].apply(classify_metric_family)

driver_pick.to_csv(
    os.path.join(OUTPUT_DIR, "dominant_driver_by_sediment_and_depth.csv"),
    index=False
)

print("\n=== DOMINANT DRIVER TABLE ===")
print(driver_pick[["SedimentType", "DepthBin", "MetricClean", "DriverCode", "AbsMetricZ"]].head(20))

# ==========================================================
# 3. PIVOT FOR TEXT + COLOR
# ==========================================================
sediment_order = [
    "f_sand",
    "vc_sand",
    "sand_slc_sand",
    "sand_psc_sand",
    "slc_sand_slc",
    "sand_dgs_grvl",
    "sand_ngs_grvl",
    "sand_silt_sand",
    "f_silty_sand",
    "sandy_silt",
]
sediment_order = [s for s in sediment_order if s in driver_pick["SedimentType"].unique()]

depth_order = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-60", "60-70", "70+"]

text_pivot = driver_pick.pivot_table(
    index="SedimentType",
    columns="DepthBin",
    values="DriverCode",
    aggfunc="first"
).reindex(index=sediment_order, columns=depth_order)

family_pivot = driver_pick.pivot_table(
    index="SedimentType",
    columns="DepthBin",
    values="DriverFamily",
    aggfunc="first"
).reindex(index=sediment_order, columns=depth_order)

strength_pivot = driver_pick.pivot_table(
    index="SedimentType",
    columns="DepthBin",
    values="AbsMetricZ",
    aggfunc="first"
).reindex(index=sediment_order, columns=depth_order)

text_pivot.to_csv(os.path.join(OUTPUT_DIR, "dominant_driver_codes_heatmap_table.csv"))
family_pivot.to_csv(os.path.join(OUTPUT_DIR, "dominant_driver_family_heatmap_table.csv"))
strength_pivot.to_csv(os.path.join(OUTPUT_DIR, "dominant_driver_strength_heatmap_table.csv"))

# ==========================================================
# 4. CONVERT FAMILY TO NUMERIC FOR COLORING
# ==========================================================
family_names = list(family_color_map.keys())
family_to_num = {fam: i for i, fam in enumerate(family_names)}

color_matrix = family_pivot.copy()
for fam, num in family_to_num.items():
    color_matrix = color_matrix.replace(fam, num)

color_matrix = color_matrix.astype(float)

cmap = ListedColormap([family_color_map[f] for f in family_names])

# ==========================================================
# 5. PLOT HEATMAP
# ==========================================================
fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(color_matrix.values, aspect="auto", cmap=cmap)

ax.set_xticks(np.arange(len(text_pivot.columns)))
ax.set_xticklabels(text_pivot.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(len(text_pivot.index)))
ax.set_yticklabels(text_pivot.index)

ax.set_xlabel("Depth Range (m)")
ax.set_ylabel("Sediment Type")
ax.set_title("Dominant Response Variable by Sediment and Depth")

# write driver code in each cell
for i in range(text_pivot.shape[0]):
    for j in range(text_pivot.shape[1]):
        code = text_pivot.iloc[i, j]
        if pd.notna(code):
            ax.text(j, i, str(code), ha="center", va="center", fontsize=9)

# custom legend
legend_handles = [
    plt.Line2D([0], [0], marker='s', color='w',
               markerfacecolor=family_color_map[f], markersize=10, label=f)
    for f in family_names
]

ax.legend(
    handles=legend_handles,
    title="Driver Family",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True
)

# driver code legend
driver_legend_text = "\n".join([
    "DEF = Deflection",
    "SLP = Slope",
    "SP  = Soil Pressure",
    "SF  = Shear Force",
    "BM  = Bending Moment",
    "AF  = Axial Force",
    "AS  = Axial Stress",
    "USR = Unit Shaft Resistance",
    "UBR = Unit Base Resistance",
    "TZ  = t-z response",
    "PY  = p-y response",
    "QZ  = q-z response",
])

plt.gcf().text(
    1.02, 0.38,
    driver_legend_text,
    fontsize=8,
    va="top",
    ha="left",
    family="monospace"
)

plt.subplots_adjust(right=0.76)
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "dominant_driver_by_sediment_and_depth_heatmap.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

print(f"\nSaved outputs to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import f_oneway, kruskal

# ==========================================================
# 1. CONFIG
# ==========================================================
configs = [
    {
        "name": "Sediment Type",
        "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_sediment_type.csv",
        "group_col": "Sediment Type",
        "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y/p_y_cyclic/plots"
    },
    {
        "name": "Water Depth",
        "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_water_depth.csv",
        "group_col": "Water Depth",
        "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y/p_y_cyclic/plots"
    },
    {
        "name": "Embedment Depth",
        "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_embedment_depth.csv",
        "group_col": "Embedment Depth",
        "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y/p_y_cyclic/plots"
    }
]



# ==========================================================
# COLOR MAPS
# ==========================================================

# Sediment colors (your established palette)
sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

# Water depth gradient (light blue → purple)
water_colors = {
    "10m": "#ADD8E6",
    "30m": "#87CEEB",
    "50m": "#6495ED",
    "70m": "#4169E1",
    "90m": "#9370DB"
}

# Embedment depth gradient (yellow → red)
embedment_colors = {
    "15m": "#FFFFB2",
    "25m": "#FED976",
    "30m": "#FEB24C",
    "35m": "#FD8D3C",
    "40m": "#FC4E2A",
    "45m": "#E31A1C",
    "55m": "#B10026"
}

# ==========================================================
# 2. CLEAN COLUMN NAMES
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return df

# ==========================================================
# 3. PARSE P-Y STRUCTURE
# ==========================================================
def extract_py_pairs(df):
    """
    Extract alternating p and y columns for each depth
    """
    py_data = {}

    cols = df.columns.tolist()[2:]  # skip group + index columns

    i = 0
    while i < len(cols) - 1:
        p_col = cols[i]
        y_col = cols[i + 1]

        depth_label = p_col.split(".")[0]

        py_data[depth_label] = {
            "p": df[p_col].astype(float),
            "y": df[y_col].astype(float)
        }

        i += 2

    return py_data

# ==========================================================
# 4. METRIC EXTRACTION
# ==========================================================
def compute_metrics(py_data):
    results = []

    for depth, data in py_data.items():
        p = data["p"].values
        y = data["y"].values

        if len(p) < 2:
            continue

        pmax = np.max(p)
        y_at_pmax = y[np.argmax(p)]

        # initial stiffness (first slope)
        try:
            k_initial = (p[1] - p[0]) / (y[1] - y[0])
        except:
            k_initial = np.nan

        results.append({
            "Depth": depth,
            "pmax": pmax,
            "y_at_pmax": y_at_pmax,
            "k_initial": k_initial
        })

    return pd.DataFrame(results)

# ==========================================================
# 5. STATISTICS
# ==========================================================
def run_stats(df, group_col, value_col):
    test_df = df[[group_col, value_col]].dropna().copy()

    if test_df.empty or test_df[group_col].nunique() < 2:
        return {
            "ANOVA_p": np.nan,
            "Kruskal_p": np.nan,
            "eta_sq": np.nan,
            "CV": np.nan,
            "note": "Not enough groups for comparison"
        }

    groups = [g[value_col].values for _, g in test_df.groupby(group_col)]

    all_constant = all(np.all(arr == arr[0]) for arr in groups if len(arr) > 0)

    grand_mean = test_df[value_col].mean()
    ss_between = sum(
        len(g) * (g[value_col].mean() - grand_mean) ** 2
        for _, g in test_df.groupby(group_col)
    )
    ss_total = sum((test_df[value_col] - grand_mean) ** 2)
    eta_sq = ss_between / ss_total if ss_total != 0 else np.nan

    cv = test_df[value_col].std() / test_df[value_col].mean() if test_df[value_col].mean() != 0 else np.nan

    if all_constant or test_df[value_col].nunique() == 1:
        return {
            "ANOVA_p": np.nan,
            "Kruskal_p": np.nan,
            "eta_sq": eta_sq,
            "CV": cv,
            "note": "All values constant; no meaningful significance test"
        }

    try:
        f_stat, p_val = f_oneway(*groups)
    except:
        p_val = np.nan

    try:
        h_stat, p_kw = kruskal(*groups)
    except:
        p_kw = np.nan

    return {
        "ANOVA_p": p_val,
        "Kruskal_p": p_kw,
        "eta_sq": eta_sq,
        "CV": cv,
        "note": ""
    }

# ==========================================================
# 6. PLOT
# ==========================================================
def get_color(group_name, depth_label, config_name):
    if config_name == "Sediment Type":
        return sediment_colors.get(str(group_name), "black")
    elif config_name == "Water Depth":
        return water_colors.get(str(group_name), "black")
    elif config_name == "Embedment Depth":
        return embedment_colors.get(str(group_name), "black")
    return "black"


def plot_py(py_data, group_name, config_name, output_path, show_plot=False):
    plt.figure(figsize=(6, 5))

    for depth, data in py_data.items():
        color = get_color(group_name, depth, config_name)

        plt.plot(
            data["y"],
            data["p"],
            label=f"{depth} m",
            color=color,
            linewidth=1.5
        )

    plt.xlabel("Deflection y (m)")
    plt.ylabel("Soil Resistance p (kN/m)")
    plt.title(f"Cyclic p-y Curves ({group_name})")
    plt.legend(fontsize=6)
    plt.tight_layout()

    plt.savefig(output_path, dpi=300, bbox_inches='tight')

    if show_plot:
        plt.show()
    else:
        plt.close()

# ==========================================================
# 7. MAIN LOOP
# ==========================================================
for config in configs:
    print(f"\nProcessing: {config['name']}")

    os.makedirs(config["output_dir"], exist_ok=True)

    df = pd.read_csv(config["input_csv"])
    df = clean_columns(df)

    # rename first column
    df = df.rename(columns={df.columns[0]: config["group_col"]})

    all_metrics = []

    for group, subdf in df.groupby(config["group_col"]):

        py_data = extract_py_pairs(subdf)

        # plot
        plot_path = os.path.join(
            config["output_dir"],
            f"py_cyclic_{config['name'].replace(' ', '_')}_{group}.png"
        )
        plot_py(py_data, group, config["name"], plot_path, show_plot=True)

        # metrics
        metrics_df = compute_metrics(py_data)
        metrics_df[config["group_col"]] = group
        all_metrics.append(metrics_df)

    # combine
    all_metrics_df = pd.concat(all_metrics, ignore_index=True)

    # save metrics
    metrics_path = os.path.join(config["output_dir"], "py_cyclic_metrics.csv")
    all_metrics_df.to_csv(metrics_path, index=False)

    # stats
    stats_results = []
    for metric in ["pmax", "y_at_pmax", "k_initial"]:
        stats = run_stats(all_metrics_df, config["group_col"], metric)
        if stats:
            stats["Metric"] = metric
            stats_results.append(stats)

    stats_df = pd.DataFrame(stats_results)

    stats_path = os.path.join(config["output_dir"], "py_cyclic_stats.csv")
    stats_df.to_csv(stats_path, index=False)

    print(f"Finished: {config['name']}")

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import ttest_rel, wilcoxon

# ==========================================================
# 1. CONFIG
# ==========================================================
configs = [
    {
        "name": "Sediment Type",
        "folder": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y",
        "static_file": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y/p_y_long_cleaned.csv",
        "cyclic_file": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_sediment_type.csv",
        "group_col": "Sediment Type"
    },
    {
        "name": "Water Depth",
        "folder": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y",
        "static_file": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y/p_y_long_cleaned.csv",
        "cyclic_file": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_water_depth.csv",
        "group_col": "Water Depth"
    },
    {
        "name": "Embedment Depth",
        "folder": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y",
        "static_file": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y/p_y_long_cleaned.csv",
        "cyclic_file": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_embedment_depth.csv",
        "group_col": "Embedment Depth"
    }
]

# ==========================================================
# 2. HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return df

def normalize_group_value(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # remove "m" if present
    x = x.replace("m", "").replace("M", "")

    # remove trailing decimals
    try:
        val = float(x)
        if abs(val - round(val)) < 1e-8:
            return str(int(round(val)))
        return str(val)
    except:
        return x

def normalize_curve_depth(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    try:
        val = float(x)
        if abs(val - round(val)) < 1e-8:
            return str(int(round(val)))
        return str(val)
    except:
        # handles things like 10.0000.1 by keeping left side
        return x.split(".")[0]

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

# ==========================================================
# 3. READ STATIC LONG DATA
# ==========================================================
def read_static_long(static_file, group_col):
    df = pd.read_csv(static_file)
    df = clean_columns(df)

    rename_map = {}

    for col in df.columns:
        c = col.lower().strip()

        if c == group_col.lower():
            rename_map[col] = group_col
        elif c in ["curve", "curve depth", "depth", "depth (m)", "py depth", "p-y depth"]:
            rename_map[col] = "Curve Depth"
        elif c in ["p", "p (kn/m)", "soil resistance p (kn/m)", "soil resistance", "resistance"]:
            rename_map[col] = "p"
        elif c in ["y", "deflection y (m)", "deflection", "y (m)"]:
            rename_map[col] = "y"

    df = df.rename(columns=rename_map)

    needed = [group_col, "Curve Depth", "p", "y"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"Missing columns in static file {static_file}: {missing}")
        print("Available columns:", df.columns.tolist())
        return pd.DataFrame(columns=needed)

    df[group_col] = df[group_col].apply(normalize_group_value)
    df["Curve Depth"] = df["Curve Depth"].apply(normalize_curve_depth)
    df["p"] = safe_numeric(df["p"])
    df["y"] = safe_numeric(df["y"])

    df = df.dropna(subset=[group_col, "Curve Depth", "p", "y"])

    return df[[group_col, "Curve Depth", "p", "y"]].copy()


def fix_embedment_curve_depths(df, group_col):
    """
    Correct static embedment curve depths to match updated cyclic values
    """

    df = df.copy()

    # Only apply to embedment datasets
    if group_col != "Embedment Depth":
        return df

    depth_map = {
        "77.5": "87.5",
        "115": "135"
    }

    df["Curve Depth"] = df["Curve Depth"].astype(str)

    df["Curve Depth"] = df["Curve Depth"].replace(depth_map)

    return df

# ==========================================================
# 4. CONVERT CYCLIC WIDE CSV TO LONG
# ==========================================================
def cyclic_wide_to_long(cyclic_file, group_col):
    df = pd.read_csv(cyclic_file)
    df = clean_columns(df)

    df = df.rename(columns={df.columns[0]: group_col})
    df[group_col] = df[group_col].apply(normalize_group_value)

    rows = []
    cols = df.columns.tolist()[2:]  # skip group + second odd column

    i = 0
    while i < len(cols) - 1:
        p_col = cols[i]
        y_col = cols[i + 1]

        depth_label = normalize_curve_depth(p_col)

        temp = pd.DataFrame({
            group_col: df[group_col],
            "Curve Depth": depth_label,
            "p": safe_numeric(df[p_col]),
            "y": safe_numeric(df[y_col])
        }).dropna(subset=[group_col, "Curve Depth", "p", "y"])

        rows.append(temp)
        i += 2

    if rows:
        return pd.concat(rows, ignore_index=True)

    return pd.DataFrame(columns=[group_col, "Curve Depth", "p", "y"])

# ==========================================================
# 5. COMPUTE METRICS FROM LONG DATA
# ==========================================================
def compute_metrics_from_long(df_long, group_col, condition_name):
    results = []

    for (group_value, curve_depth), sub in df_long.groupby([group_col, "Curve Depth"]):
        sub = sub.sort_values("y").copy()

        p = sub["p"].values.astype(float)
        y = sub["y"].values.astype(float)

        if len(p) < 2 or len(y) < 2:
            continue

        pmax = np.nanmax(p)
        idx = np.nanargmax(p)
        y_at_pmax = y[idx]

        k_initial = np.nan
        for j in range(1, len(y)):
            dy = y[j] - y[j - 1]
            dp = p[j] - p[j - 1]
            if dy != 0:
                k_initial = dp / dy
                break

        area = np.nan
        try:
            area = np.trapz(p, y)
        except:
            pass

        results.append({
            group_col: group_value,
            "Curve Depth": curve_depth,
            "Condition": condition_name,
            "pmax": pmax,
            "y_at_pmax": y_at_pmax,
            "k_initial": k_initial,
            "area_under_curve": area
        })

    return pd.DataFrame(results)

# ==========================================================
# 6. PAIRED STATS
# ==========================================================
def cohens_d_paired(before, after):
    diff = after - before
    diff = diff[~np.isnan(diff)]
    if len(diff) < 2:
        return np.nan
    sd = np.std(diff, ddof=1)
    if sd == 0:
        return np.nan
    return np.mean(diff) / sd

def paired_stats(df, static_col, cyclic_col):
    sub = df[[static_col, cyclic_col]].dropna()

    if len(sub) < 2:
        return {
            "n_pairs": len(sub),
            "static_mean": sub[static_col].mean() if len(sub) else np.nan,
            "cyclic_mean": sub[cyclic_col].mean() if len(sub) else np.nan,
            "mean_difference": np.nan,
            "percent_change_mean": np.nan,
            "paired_t_p": np.nan,
            "wilcoxon_p": np.nan,
            "cohens_d_paired": np.nan,
            "note": "Not enough paired data"
        }

    before = sub[static_col].values.astype(float)
    after = sub[cyclic_col].values.astype(float)

    diff = after - before
    pct_change = np.where(before != 0, (diff / before) * 100, np.nan)

    try:
        t_p = ttest_rel(before, after, nan_policy="omit").pvalue
    except:
        t_p = np.nan

    try:
        if np.allclose(diff, 0, equal_nan=True):
            w_p = np.nan
        else:
            w_p = wilcoxon(before, after).pvalue
    except:
        w_p = np.nan

    return {
        "n_pairs": len(sub),
        "static_mean": np.nanmean(before),
        "cyclic_mean": np.nanmean(after),
        "mean_difference": np.nanmean(diff),
        "percent_change_mean": np.nanmean(pct_change),
        "paired_t_p": t_p,
        "wilcoxon_p": w_p,
        "cohens_d_paired": cohens_d_paired(before, after),
        "note": ""
    }

# ==========================================================
# 7. RUN
# ==========================================================
for cfg in configs:
    print("\n" + "=" * 70)
    print(f"PROCESSING: {cfg['name']}")
    print("=" * 70)

    group_col = cfg["group_col"]
    out_dir = os.path.join(cfg["folder"], "static_vs_cyclic_comparison")
    os.makedirs(out_dir, exist_ok=True)

    static_long = read_static_long(cfg["static_file"], group_col)
    cyclic_long = cyclic_wide_to_long(cfg["cyclic_file"], group_col)

# 🔥 FIX EMBEDMENT DEPTH MISMATCH
    static_long = fix_embedment_curve_depths(static_long, group_col)

    print("Static long shape:", static_long.shape)
    print("Cyclic long shape:", cyclic_long.shape)

    if static_long.empty:
        print("Static long table is empty.")
        continue

    if cyclic_long.empty:
        print("Cyclic long table is empty.")
        continue

    static_long.to_csv(os.path.join(out_dir, "static_long_for_comparison.csv"), index=False)
    cyclic_long.to_csv(os.path.join(out_dir, "cyclic_long_for_comparison.csv"), index=False)

    static_metrics = compute_metrics_from_long(static_long, group_col, "Static")
    cyclic_metrics = compute_metrics_from_long(cyclic_long, group_col, "Cyclic")

    print("Static metrics shape:", static_metrics.shape)
    print("Cyclic metrics shape:", cyclic_metrics.shape)

    if static_metrics.empty or cyclic_metrics.empty:
        print("One of the metric tables is empty after metric extraction.")
        continue

    static_metrics.to_csv(os.path.join(out_dir, "static_metrics_extracted.csv"), index=False)
    cyclic_metrics.to_csv(os.path.join(out_dir, "cyclic_metrics_extracted.csv"), index=False)

    merged = pd.merge(
        static_metrics,
        cyclic_metrics,
        on=[group_col, "Curve Depth"],
        suffixes=("_static", "_cyclic"),
        how="inner"
    )

    print("Merged shape:", merged.shape)

    if merged.empty:
        print("Merged table is empty.")
        print("Static Curve Depth values:", sorted(static_metrics["Curve Depth"].dropna().unique().tolist())[:20])
        print("Cyclic Curve Depth values:", sorted(cyclic_metrics["Curve Depth"].dropna().unique().tolist())[:20])
        print("Static group values:", sorted(static_metrics[group_col].dropna().unique().tolist())[:20])
        print("Cyclic group values:", sorted(cyclic_metrics[group_col].dropna().unique().tolist())[:20])
        continue

    for metric in ["pmax", "y_at_pmax", "k_initial", "area_under_curve"]:
        merged[f"{metric}_diff"] = merged[f"{metric}_cyclic"] - merged[f"{metric}_static"]
        merged[f"{metric}_pct_change"] = np.where(
            merged[f"{metric}_static"] != 0,
            (merged[f"{metric}_diff"] / merged[f"{metric}_static"]) * 100,
            np.nan
        )

    merged.to_csv(os.path.join(out_dir, "static_vs_cyclic_matched_metrics.csv"), index=False)

    overall_rows = []
    for metric in ["pmax", "y_at_pmax", "k_initial", "area_under_curve"]:
        stats = paired_stats(merged, f"{metric}_static", f"{metric}_cyclic")
        stats["Metric"] = metric
        stats["Comparison Level"] = "Overall"
        overall_rows.append(stats)
    pd.DataFrame(overall_rows).to_csv(
        os.path.join(out_dir, "overall_paired_stats_static_vs_cyclic.csv"),
        index=False
    )

    by_group_rows = []
    for group_value, sub in merged.groupby(group_col):
        for metric in ["pmax", "y_at_pmax", "k_initial", "area_under_curve"]:
            stats = paired_stats(sub, f"{metric}_static", f"{metric}_cyclic")
            stats[group_col] = group_value
            stats["Metric"] = metric
            stats["Comparison Level"] = "By Group"
            by_group_rows.append(stats)
    pd.DataFrame(by_group_rows).to_csv(
        os.path.join(out_dir, "by_group_paired_stats_static_vs_cyclic.csv"),
        index=False
    )

    by_depth_rows = []
    for depth_value, sub in merged.groupby("Curve Depth"):
        for metric in ["pmax", "y_at_pmax", "k_initial", "area_under_curve"]:
            stats = paired_stats(sub, f"{metric}_static", f"{metric}_cyclic")
            stats["Curve Depth"] = depth_value
            stats["Metric"] = metric
            stats["Comparison Level"] = "By Curve Depth"
            by_depth_rows.append(stats)
    pd.DataFrame(by_depth_rows).to_csv(
        os.path.join(out_dir, "by_curve_depth_paired_stats_static_vs_cyclic.csv"),
        index=False
    )

    summary_by_group = (
        merged.groupby(group_col)[[
            "pmax_diff", "pmax_pct_change",
            "y_at_pmax_diff", "y_at_pmax_pct_change",
            "k_initial_diff", "k_initial_pct_change",
            "area_under_curve_diff", "area_under_curve_pct_change"
        ]]
        .mean()
        .reset_index()
    )
    summary_by_group.to_csv(os.path.join(out_dir, "summary_mean_change_by_group.csv"), index=False)

    summary_by_depth = (
        merged.groupby("Curve Depth")[[
            "pmax_diff", "pmax_pct_change",
            "y_at_pmax_diff", "y_at_pmax_pct_change",
            "k_initial_diff", "k_initial_pct_change",
            "area_under_curve_diff", "area_under_curve_pct_change"
        ]]
        .mean()
        .reset_index()
    )
    summary_by_depth.to_csv(os.path.join(out_dir, "summary_mean_change_by_curve_depth.csv"), index=False)

    print("Saved outputs to:", out_dir)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import f_oneway, kruskal

# ==========================================================
# 1. CONFIG
# ==========================================================
configs = [
    {
        "name": "Sediment Type",
        "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_sediment_type.csv",
        "group_col": "Sediment Type",
        "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Sediment Comparison/organized/p_y/p_y_cyclic",
    },
    {
        "name": "Water Depth",
        "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_water_depth.csv",
        "group_col": "Water Depth",
        "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Water Depth Comparison/organized/p_y/p_y_cyclic",
    },
    {
        "name": "Embedment Depth",
        "input_csv": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y/p_y_cyclic/p_y_cyclic_embedment_depth.csv",
        "group_col": "Embedment Depth",
        "output_dir": r"/Users/ophelia.christoph/Downloads/Thesis/Chapter_3/GEMS/Embedment Depth Comparison/organized/p_y/p_y_cyclic",
    }
]

# ==========================================================
# 2. COLORS
# ==========================================================
sediment_colors = {
    "f_sand": "#1f77b4",
    "vc_sand": "#ff7f0e",
    "sand_slc_sand": "#2ca02c",
    "sand_psc_sand": "#d62728",
    "slc_sand_slc": "#9467bd",
    "sand_dgs_grvl": "#8c564b",
    "sand_ngs_grvl": "#e377c2",
    "sand_silt_sand": "#7f7f7f",
    "f_silty_sand": "#bcbd22",
    "sandy_silt": "#17becf"
}

water_colors = {
    "10": "#ADD8E6",
    "30": "#87CEEB",
    "50": "#6495ED",
    "70": "#4169E1",
    "90": "#9370DB"
}

embedment_colors = {
    "15": "#FFFFB2",
    "25": "#FED976",
    "30": "#FEB24C",
    "35": "#FD8D3C",
    "40": "#FC4E2A",
    "45": "#E31A1C",
    "55": "#B10026"
}

# ==========================================================
# 3. HELPERS
# ==========================================================
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return df

def normalize_group_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("m", "").replace("M", "")
    try:
        val = float(x)
        if abs(val - round(val)) < 1e-8:
            return str(int(round(val)))
        return str(val)
    except:
        return x

def normalize_curve_depth(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    try:
        val = float(x)
        if abs(val - round(val)) < 1e-8:
            return str(int(round(val)))
        return str(val)
    except:
        return x.split(".")[0]

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def get_color(group_value, config_name):
    g = str(group_value)
    if config_name == "Sediment Type":
        return sediment_colors.get(g, "black")
    elif config_name == "Water Depth":
        return water_colors.get(g, "black")
    elif config_name == "Embedment Depth":
        return embedment_colors.get(g, "black")
    return "black"

# ==========================================================
# 4. CONVERT CYCLIC WIDE CSV TO LONG
# ==========================================================
def cyclic_wide_to_long(csv_path, group_col):
    df = pd.read_csv(csv_path)
    df = clean_columns(df)

    df = df.rename(columns={df.columns[0]: group_col})
    df[group_col] = df[group_col].apply(normalize_group_value)

    rows = []
    cols = df.columns.tolist()[2:]  # skip group + second odd column

    i = 0
    while i < len(cols) - 1:
        p_col = cols[i]
        y_col = cols[i + 1]
        curve_depth = normalize_curve_depth(p_col)

        temp = pd.DataFrame({
            group_col: df[group_col],
            "Curve Depth": curve_depth,
            "p": safe_numeric(df[p_col]),
            "y": safe_numeric(df[y_col])
        }).dropna(subset=[group_col, "Curve Depth", "p", "y"])

        rows.append(temp)
        i += 2

    if rows:
        return pd.concat(rows, ignore_index=True)

    return pd.DataFrame(columns=[group_col, "Curve Depth", "p", "y"])

# ==========================================================
# 5. METRICS
# ==========================================================
def compute_metrics_from_long(df_long, group_col):
    results = []

    for (group_value, curve_depth), sub in df_long.groupby([group_col, "Curve Depth"]):
        sub = sub.sort_values("y").copy()

        p = sub["p"].values.astype(float)
        y = sub["y"].values.astype(float)

        if len(p) < 2 or len(y) < 2:
            continue

        pmax = np.nanmax(p)
        idx = np.nanargmax(p)
        y_at_pmax = y[idx]

        k_initial = np.nan
        for j in range(1, len(y)):
            dy = y[j] - y[j - 1]
            dp = p[j] - p[j - 1]
            if dy != 0:
                k_initial = dp / dy
                break

        area = np.nan
        try:
            area = np.trapz(p, y)
        except:
            pass

        results.append({
            group_col: group_value,
            "Curve Depth": curve_depth,
            "pmax": pmax,
            "y_at_pmax": y_at_pmax,
            "k_initial": k_initial,
            "area_under_curve": area
        })

    return pd.DataFrame(results)

# ==========================================================
# 6. STATS
# ==========================================================
def run_stats(df, group_col, value_col):
    test_df = df[[group_col, value_col]].dropna().copy()

    if test_df.empty or test_df[group_col].nunique() < 2:
        return {
            "ANOVA_p": np.nan,
            "Kruskal_p": np.nan,
            "eta_sq": np.nan,
            "CV": np.nan,
            "note": "Not enough groups"
        }

    # if metric is constant across all rows
    if test_df[value_col].nunique() <= 1:
        cv = 0 if test_df[value_col].mean() != 0 else np.nan
        return {
            "ANOVA_p": np.nan,
            "Kruskal_p": np.nan,
            "eta_sq": 0.0,
            "CV": cv,
            "note": "Constant metric across groups"
        }

    groups = [g[value_col].values for _, g in test_df.groupby(group_col)]

    grand_mean = test_df[value_col].mean()
    ss_between = sum(
        len(g) * (g[value_col].mean() - grand_mean) ** 2
        for _, g in test_df.groupby(group_col)
    )
    ss_total = sum((test_df[value_col] - grand_mean) ** 2)

    if ss_total == 0:
        eta_sq = 0.0
    else:
        eta_sq = ss_between / ss_total

    if test_df[value_col].mean() == 0:
        cv = np.nan
    else:
        cv = test_df[value_col].std() / test_df[value_col].mean()

    try:
        _, anova_p = f_oneway(*groups)
    except:
        anova_p = np.nan

    try:
        _, kruskal_p = kruskal(*groups)
    except:
        kruskal_p = np.nan

    return {
        "ANOVA_p": anova_p,
        "Kruskal_p": kruskal_p,
        "eta_sq": eta_sq,
        "CV": cv,
        "note": ""
    }

# ==========================================================
# 7. PLOTS
# ==========================================================
def plot_cyclic_curves_by_factor(df_long, group_col, config_name, output_dir):
    curve_dir = os.path.join(output_dir, "cyclic_curve_plots")
    os.makedirs(curve_dir, exist_ok=True)

    for group_value, sub in df_long.groupby(group_col):
        plt.figure(figsize=(6, 5))

        for curve_depth, dsub in sub.groupby("Curve Depth"):
            dsub = dsub.sort_values("y")
            plt.plot(dsub["y"], dsub["p"], label=f"{curve_depth} m", linewidth=1.5)

        plt.xlabel("Deflection y (m)")
        plt.ylabel("Soil Resistance p (kN/m)")
        plt.title(f"Cyclic p-y Curves: {group_value}")
        plt.legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(os.path.join(curve_dir, f"cyclic_py_{config_name.replace(' ', '_')}_{group_value}.png"), dpi=300)
        plt.close()

def plot_max_p_by_group(metrics_df, group_col, config_name, output_dir):
    summary = metrics_df.groupby(group_col, as_index=False)["pmax"].mean()

    summary[group_col] = summary[group_col].astype(str)
    colors = [get_color(g, config_name) for g in summary[group_col]]

    plt.figure(figsize=(7, 5))
    plt.bar(summary[group_col], summary["pmax"], color=colors)
    plt.xlabel(group_col)
    plt.ylabel("Mean Cyclic pmax (kN/m)")
    plt.title(f"Mean Cyclic pmax by {group_col}")
    plt.tight_layout()

    if config_name == "Sediment Type":
        fname = "max_p_cyclic_by_sediment.png"
    elif config_name == "Water Depth":
        fname = "max_p_cyclic_vs_water_depth.png"
    elif config_name == "Embedment Depth":
        fname = "max_p_cyclic_vs_embedment_depth.png"
    else:
        fname = f"max_p_cyclic_by_{group_col.replace(' ', '_')}.png"

    plt.savefig(os.path.join(output_dir, fname), dpi=300)
    plt.close()

def plot_cv_bar(stats_df, output_dir, config_name):
    plot_df = stats_df.copy()
    plot_df["CV"] = pd.to_numeric(plot_df["CV"], errors="coerce").fillna(0)

    plt.figure(figsize=(6, 4))
    bars = plt.bar(plot_df["Metric"], plot_df["CV"])
    plt.ylabel("Coefficient of Variation")
    plt.title(f"Cyclic p-y CV by Metric ({config_name})")
    plt.ylim(0, max(0.05, plot_df["CV"].max() * 1.15))

    for bar, val in zip(bars, plot_df["CV"]):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "cv_cyclic_bar.png"), dpi=300)
    plt.close()

def plot_eta_squared(stats_df, output_dir, config_name):
    plot_df = stats_df.copy()

    # force numeric and replace NaN with 0 for plotting
    plot_df["eta_sq"] = pd.to_numeric(plot_df["eta_sq"], errors="coerce").fillna(0)

    plt.figure(figsize=(6, 4))
    bars = plt.bar(plot_df["Metric"], plot_df["eta_sq"])
    plt.ylabel("Eta Squared")
    plt.title(f"Cyclic p-y Effect Size by Metric ({config_name})")
    plt.ylim(0, max(0.05, plot_df["eta_sq"].max() * 1.15))

    # label bars
    for bar, val in zip(bars, plot_df["eta_sq"]):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "eta_squared_cyclic.png"), dpi=300)
    plt.close()

# ==========================================================
# 8. RUN
# ==========================================================
for cfg in configs:
    print(f"\nProcessing {cfg['name']}")

    os.makedirs(cfg["output_dir"], exist_ok=True)

    df_long = cyclic_wide_to_long(cfg["input_csv"], cfg["group_col"])
    df_long.to_csv(os.path.join(cfg["output_dir"], "cyclic_long_cleaned.csv"), index=False)

    metrics_df = compute_metrics_from_long(df_long, cfg["group_col"])
    metrics_df.to_csv(os.path.join(cfg["output_dir"], "cyclic_metrics_summary.csv"), index=False)

    stats_rows = []
    for metric in ["pmax", "y_at_pmax", "k_initial", "area_under_curve"]:
        stats = run_stats(metrics_df, cfg["group_col"], metric)
        stats["Metric"] = metric
        stats_rows.append(stats)

    stats_df = pd.DataFrame(stats_rows)
    stats_df.to_csv(os.path.join(cfg["output_dir"], "cyclic_stats_summary.csv"), index=False)

    # Figures
    plot_cyclic_curves_by_factor(df_long, cfg["group_col"], cfg["name"], cfg["output_dir"])
    plot_max_p_by_group(metrics_df, cfg["group_col"], cfg["name"], cfg["output_dir"])
    plot_cv_bar(stats_df, cfg["output_dir"], cfg["name"])
    plot_eta_squared(stats_df, cfg["output_dir"], cfg["name"])

    print(f"Saved outputs to: {cfg['output_dir']}")

In [ ]:
pip install --upgrade jupyterlab-git
